In [1]:
pip install scikit-image

Note: you may need to restart the kernel to use updated packages.


In [1]:
!pip install codecarbon --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 7.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.2 MB/s eta 0:00:00a 0:00:01


### 128 bits

In [12]:
# ============================================================
# FLEXMark (COMMON CELL 1) — Shared utilities for ALL models
# Dataset-only payload (128/256 bits) + non-blind verification + attacks
# ============================================================

import os, json, time, random, hashlib
from glob import glob
from dataclasses import dataclass
from io import BytesIO

import numpy as np
from PIL import Image, ImageFilter, ImageEnhance
import matplotlib.pyplot as plt

# ---------------------------
# CO2 (CodeCarbon)
# ---------------------------
try:
    from codecarbon import EmissionsTracker
except Exception:
    EmissionsTracker = None

# ---------------------------
# SSIM
# ---------------------------
try:
    from skimage.metrics import structural_similarity as sk_ssim
except Exception:
    sk_ssim = None

# ---------------------------
# AUC + confusion
# ---------------------------
try:
    from sklearn.metrics import roc_curve, auc, confusion_matrix
except Exception:
    roc_curve, auc, confusion_matrix = None, None, None

# ---------------------------
# Optional LPIPS (perceptual)
# ---------------------------
TORCH_OK = False
LPIPS_OK = False
try:
    import torch
    TORCH_OK = True
    try:
        import lpips
        LPIPS_OK = True
    except Exception:
        LPIPS_OK = False
except Exception:
    TORCH_OK = False
    LPIPS_OK = False


# ============================================================
# Repro / IO helpers
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_OK:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def save_json(obj, path):
    ensure_dir(os.path.dirname(path) if os.path.dirname(path) else ".")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def read_rgb(path, hw):
    img = Image.open(path).convert("RGB").resize((hw, hw), Image.BICUBIC)
    return np.asarray(img).astype(np.float32) / 255.0

def np_to_pil(x):
    x = np.clip(x * 255.0, 0, 255).astype(np.uint8)
    return Image.fromarray(x)

def pil_to_np(p):
    return np.asarray(p).astype(np.float32) / 255.0

def to_torch_img(x_np):
    # (H,W,3) float [0,1] -> (1,3,H,W)
    if not TORCH_OK:
        raise RuntimeError("Torch not available.")
    return torch.from_numpy(x_np).permute(2,0,1).unsqueeze(0).float()

def to_np_img(x_t):
    # (1,3,H,W) -> (H,W,3) float [0,1]
    x = x_t.detach().cpu().squeeze(0).permute(1,2,0).numpy()
    return np.clip(x, 0, 1)


# ============================================================
# Metrics (imperceptibility vs COVER)
# ============================================================
def mse(a, b):
    a = np.asarray(a, np.float32); b = np.asarray(b, np.float32)
    return float(np.mean((a - b) ** 2))

def psnr_from_mse(m):
    if m <= 1e-12: return 99.0
    return float(10.0 * np.log10(1.0 / m))

def psnr(a, b):
    return psnr_from_mse(mse(a, b))

def ssim_np(a, b):
    if sk_ssim is None:
        return float("nan")
    return float(sk_ssim(a, b, channel_axis=2, data_range=1.0))

def binarize01(x, thr=0.5):
    return (x >= thr).astype(np.float32)

def nc(a, b):
    a = a.astype(np.float32).flatten()
    b = b.astype(np.float32).flatten()
    denom = (np.linalg.norm(a) * np.linalg.norm(b)) + 1e-8
    return float(np.dot(a, b) / denom)

def ber(a_bin, b_bin):
    a_bin = a_bin.astype(np.float32).flatten()
    b_bin = b_bin.astype(np.float32).flatten()
    return float(np.mean(a_bin != b_bin) * 100.0)

# Optional LPIPS vs cover
_LPIPS_NET = None
def lpips_vs_cover(a, b, net="alex"):
    """
    Returns LPIPS distance (lower is better). Requires torch + lpips.
    """
    global _LPIPS_NET
    if not LPIPS_OK:
        return float("nan")
    if _LPIPS_NET is None:
        _LPIPS_NET = lpips.LPIPS(net=net)
        _LPIPS_NET.eval()
        if torch.cuda.is_available():
            _LPIPS_NET = _LPIPS_NET.cuda()

    ta = to_torch_img(a) * 2 - 1
    tb = to_torch_img(b) * 2 - 1
    if torch.cuda.is_available():
        ta = ta.cuda()
        tb = tb.cuda()
    with torch.no_grad():
        d = _LPIPS_NET(ta, tb)
    return float(d.item())


# ============================================================
# Dataset-only payload
# 128 bits -> 8x16 grid
# 256 bits -> 16x16 grid
# ============================================================
def bits_to_grid(bits, B):
    if B == 64:
        return bits.reshape(8, 8).astype(np.float32), (8, 8)
    if B == 128:
        return bits.reshape(8, 16).astype(np.float32), (8, 16)
    if B == 256:
        return bits.reshape(16, 16).astype(np.float32), (16, 16)
    raise ValueError("B must be 64, 128, or 256")

def upsample_grid_to_hw_rgb(grid01, hw):
    pil = Image.fromarray((grid01 * 255).astype(np.uint8)).resize((hw, hw), Image.NEAREST)
    arr = np.asarray(pil).astype(np.float32) / 255.0
    return np.repeat(arr[..., None], 3, axis=2)  # (H,W,3)

def upsample_grid_to_hw_1ch(grid01, hw):
    pil = Image.fromarray((grid01 * 255).astype(np.uint8)).resize((hw, hw), Image.NEAREST)
    arr = np.asarray(pil).astype(np.float32) / 255.0
    return arr[..., None]  # (H,W,1)

def mark_to_bits(mark_rgb, B):
    """
    Deterministically derive B bits from the dataset mark image.
    SHA-256 in counter mode -> supports 64/128/256 bits.
    """
    if B not in (64, 128, 256):
        raise ValueError("Supported payload bits: 64, 128, or 256")

    m = (np.clip(mark_rgb, 0, 1) * 255.0).astype(np.uint8)
    base = m.tobytes()

    need_bytes = B // 8
    out = bytearray()
    ctr = 0
    while len(out) < need_bytes:
        h = hashlib.sha256(base + ctr.to_bytes(4, "little")).digest()
        out.extend(h)
        ctr += 1

    out = bytes(out[:need_bytes])
    bits = np.unpackbits(np.frombuffer(out, dtype=np.uint8)).astype(np.int32)
    return bits[:B]

def build_payload_from_mark(mark_path, hw, B, out_channels="rgb"):
     mark_rgb = read_rgb(mark_path, hw)
    bits = mark_to_bits(mark_rgb, B)
    grid, (gh, gw) = bits_to_grid(bits, B)
    W_true = grid[..., None].astype(np.float32)

    if out_channels == "1ch":
        Wp = upsample_grid_to_hw_1ch(grid, hw)
    else:
        Wp = upsample_grid_to_hw_rgb(grid, hw)
    return Wp, W_true, (gh, gw)


# ============================================================
# Non-blind extraction helper (verifier has cover C)
# Works with residual-budget embedding: Y = clip(C + r, 0,1), r in [-eps,+eps]
# proxy = (Y-C)/eps -> [-1,1]
# then map to [0,1] and downsample to payload grid
# ============================================================
def inverse_residual_decode(Y, C, eps):
    r = np.clip(Y - C, -eps, eps)
    return r / (eps + 1e-8)  # approx [-1,1]

def decode_proxy_to_grid(w_proxy_rgb, gh, gw):
    # expects proxy in 3-ch; if you ever have 1-ch, adapt here
    if w_proxy_rgb.ndim == 3 and w_proxy_rgb.shape[2] == 3:
        g = np.mean((w_proxy_rgb + 1.0) / 2.0, axis=2)  # [-1,1] -> [0,1]
    else:
        # fallback: treat last dim as already scalar
        g = (w_proxy_rgb.squeeze() + 1.0) / 2.0

    pil = Image.fromarray((np.clip(g,0,1) * 255).astype(np.uint8)).resize((gw, gh), Image.NEAREST)
    return np.asarray(pil).astype(np.float32) / 255.0


# ============================================================
# Attack suite
# ============================================================
def attack_jpeg(x, quality=50):
    pil = np_to_pil(x)
    buf = BytesIO()
    pil.save(buf, format="JPEG", quality=int(quality))
    buf.seek(0)
    out = Image.open(buf).convert("RGB")
    return pil_to_np(out)

def attack_resize(x, scale=0.75):
    H, W, _ = x.shape
    nh, nw = int(H * scale), int(W * scale)
    pil = np_to_pil(x).resize((nw, nh), Image.BICUBIC).resize((W, H), Image.BICUBIC)
    return pil_to_np(pil)

def attack_crop_resize(x, crop_frac=0.10):
    H, W, _ = x.shape
    ch, cw = int(H * (1 - crop_frac)), int(W * (1 - crop_frac))
    y0 = np.random.randint(0, H - ch + 1)
    x0 = np.random.randint(0, W - cw + 1)
    crop = x[y0:y0+ch, x0:x0+cw, :]
    pil = np_to_pil(crop).resize((W, H), Image.BICUBIC)
    return pil_to_np(pil)

def attack_blur(x, radius=1.2):
    pil = np_to_pil(x).filter(ImageFilter.GaussianBlur(radius=float(radius)))
    return pil_to_np(pil)

def attack_sharpen(x, factor=1.6):
    pil = np_to_pil(x)
    enh = ImageEnhance.Sharpness(pil)
    return pil_to_np(enh.enhance(float(factor)))

def attack_gamma(x, gamma=1.2):
    x = np.clip(x, 0, 1)
    return np.clip(x ** float(gamma), 0, 1)

def attack_screenshot_recompress(x, q1=35, q2=80):
    y = attack_jpeg(x, q1)
    y = attack_resize(y, 0.92)
    y = attack_jpeg(y, q2)
    return y

ATTACKS = {
    "clean":      lambda im: im,
    "jpeg10":     lambda im: attack_jpeg(im, 10),
    "jpeg30":     lambda im: attack_jpeg(im, 30),
    "jpeg50":     lambda im: attack_jpeg(im, 50),
    "jpeg70":     lambda im: attack_jpeg(im, 70),
    "jpeg90":     lambda im: attack_jpeg(im, 90),
    "resize075":  lambda im: attack_resize(im, 0.75),
    "crop10":     lambda im: attack_crop_resize(im, 0.10),
    "blur":       lambda im: attack_blur(im, 1.2),
    "sharpen":    lambda im: attack_sharpen(im, 1.6),
    "gamma12":    lambda im: attack_gamma(im, 1.2),
    "screenshot": lambda im: attack_screenshot_recompress(im, 35, 80),
    "launder_only": lambda im: im,  # keep placeholder for Cell 4 laundering step
}

# ============================================================
# CodeCarbon helpers
# ============================================================
def cc_start(name, out_dir):
    if EmissionsTracker is None:
        return None
    try:
        tr = EmissionsTracker(
            project_name=name,
            output_dir=out_dir,
            measure_power_secs=1,
            log_level="error"
        )
        tr.start()
        return tr
    except Exception:
        return None

def cc_stop(tr):
    if tr is None:
        return None
    try:
        return tr.stop()
    except Exception:
        return None

# ============================================================
# Detection reliability helper (AUC)
# ============================================================
def detection_auc(scores, labels):
    """
    scores: similarity scores (higher => more likely correct)
    labels: 1 (correct) / 0 (wrong)
    """
    if roc_curve is None or auc is None:
        return float("nan"), None, None
    fpr, tpr, thr = roc_curve(labels, scores)
    return float(auc(fpr, tpr)), (fpr, tpr), thr

In [3]:
# ============================================================
# FLEXMark (COMMON CELL 2) — Config + dataset paths (DATASET-ONLY PAYLOAD)
# Reuse for LinearSVR / CNN / cGAN / HiDDeN / RL
# ============================================================

PAYLOAD_BITS = 256   # 128 or 256 (run both in separate runs if you can)

BASE_DIR = "/kaggle/input/image-watermarking-datasetcover-mark/dataset"
OUT_DIR  = f"/kaggle/working/flexmark_out_bits{PAYLOAD_BITS}"
ensure_dir(OUT_DIR)

# ---- Core protocol settings ----
SEED = 42
set_seed(SEED)

HW  = 128            # cover image size
EPS = 0.02           # residual budget (imperceptibility measured vs cover)

# ---- Experiment sizes ----
TRAIN_N = 5000
TEST_N  = 1000
EVAL_N  = 1000

# ---- Dataset layout ----
trC = sorted(glob(os.path.join(BASE_DIR, "train", "cover", "*.JPEG")))
trW = sorted(glob(os.path.join(BASE_DIR, "train", "mark",  "*.png")))
teC = sorted(glob(os.path.join(BASE_DIR, "test",  "cover", "*.JPEG")))
teW = sorted(glob(os.path.join(BASE_DIR, "test",  "mark",  "*.png")))

n_train = min(len(trC), len(trW), TRAIN_N)
n_test  = min(len(teC), len(teW), TEST_N)

trC, trW = trC[:n_train], trW[:n_train]
teC, teW = teC[:n_test],  teW[:n_test]

print("Train pairs used:", len(trC))
print("Test pairs used :", len(teC))
print("HW:", HW, "| EPS:", EPS, "| PAYLOAD_BITS:", PAYLOAD_BITS)
print("Example cover:", trC[0] if trC else "NONE")
print("Example mark :", trW[0] if trW else "NONE")

# ---- Dataset-only payload builder (per-sample) ----
def make_payload_for_model(mark_path, hw, bitsB, mode):
    out_ch = "1ch" if mode == "1ch" else "rgb"
    return build_payload_from_mark(mark_path, hw, bitsB, out_channels=out_ch)

Train pairs used: 5000
Test pairs used : 1000
HW: 128 | EPS: 0.02 | PAYLOAD_BITS: 256
Example cover: /kaggle/input/image-watermarking-datasetcover-mark/dataset/train/cover/ILSVRC2012_val_00005002.JPEG
Example mark : /kaggle/input/image-watermarking-datasetcover-mark/dataset/train/mark/airplane0003.png


In [7]:
# ============================================================
# FLEXMark (COMMON CELL 3) — Threat model + AI laundering
# Reuse for cGAN / HiDDeN / custom CNN
# NOTE: Attacks come from COMMON CELL 1 (no re-definition here).
# ============================================================

import time

TORCH_OK = False
LPIPS_OK = False
try:
    import torch
    import torch.nn as nn
    TORCH_OK = True
    try:
        import lpips
        LPIPS_OK = True
    except Exception:
        LPIPS_OK = False
except Exception:
    TORCH_OK = False
    LPIPS_OK = False


# ============================================================
# (B) Optional perceptual metric (LPIPS) — cover-referenced
# ============================================================
_lpips_fn = None
def lpips_score_np(a_np, b_np):
    global _lpips_fn
    if not (TORCH_OK and LPIPS_OK):
        return float("nan")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    if _lpips_fn is None:
        _lpips_fn = lpips.LPIPS(net="alex").to(device).eval()

    def _to_t(x):
        t = torch.from_numpy(x).permute(2,0,1).unsqueeze(0).float()
        return (t * 2.0 - 1.0).to(device)  # [-1,1]

    with torch.no_grad():
        d = _lpips_fn(_to_t(a_np), _to_t(b_np))
        return float(d.detach().cpu().item())


# ============================================================
# (C) AI-based laundering: Tiny patch-denoiser (fast, defensible)
# - trains on ALIGNED patches: (attacked watermarked patch -> cover patch)
# ============================================================
if TORCH_OK:
    class TinyDenoiser(nn.Module):
        def __init__(self, ch=3, width=32):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(ch, width, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(width, width, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(width, ch, 3, padding=1),
            )
        def forward(self, x):
            return torch.clamp(self.net(x), 0.0, 1.0)

    def _np_to_t(img_np, device):
        return torch.from_numpy(img_np).permute(2,0,1).unsqueeze(0).float().to(device)

    def _t_to_np(t):
        x = t.detach().cpu().squeeze(0).permute(1,2,0).numpy()
        return np.clip(x, 0, 1)

    def extract_aligned_patch_pair(x_np, y_np, patch=64):
        H, W, _ = x_np.shape
        patch = min(patch, H, W)
        y0 = np.random.randint(0, H - patch + 1)
        x0 = np.random.randint(0, W - patch + 1)
        xp = x_np[y0:y0+patch, x0:x0+patch, :]
        yp = y_np[y0:y0+patch, x0:x0+patch, :]
        return xp, yp

    def train_ai_launderer_patchwise(
        pairs,                      # list of tuples: (watermarked_attacked_np, cover_np)
        out_dir,
        patch=64,
        patches_per_img=2,
        epochs=1,
        lr=1e-3,
        max_pairs=50,
        device=None
    ):
        """
        Returns: (launder_fn, meta_dict)
        launder_fn(img_np) -> laundered_img_np
        """
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        use_pairs = pairs[:max_pairs]
        X_patches, Y_patches = [], []
        for (yw_attacked, c) in use_pairs:
            for _ in range(patches_per_img):
                xp, yp = extract_aligned_patch_pair(yw_attacked, c, patch=patch)
                X_patches.append(xp)
                Y_patches.append(yp)

        if len(X_patches) == 0:
            return None, {"enabled": False, "reason": "no_patches"}

        model = TinyDenoiser(ch=3, width=32).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.L1Loss()

        t0 = time.time()
        for ep in range(int(epochs)):
            model.train()
            idx = np.random.permutation(len(X_patches))
            for k in idx:
                x = _np_to_t(X_patches[k], device)
                y = _np_to_t(Y_patches[k], device)
                pred = model(x)
                loss = loss_fn(pred, y)
                opt.zero_grad()
                loss.backward()
                opt.step()

        train_s = time.time() - t0

        path = os.path.join(out_dir, "ai_launder_tinydenoiser.pt")
        torch.save(model.state_dict(), path)
        mb = os.path.getsize(path) / (1024**2)

        model.eval()
        @torch.no_grad()
        def launder_fn(img_np):
            x = _np_to_t(img_np, device)
            y = model(x)
            return _t_to_np(y)

        meta = {
            "enabled": True,
            "device": device,
            "patch": int(patch),
            "patches_per_img": int(patches_per_img),
            "epochs": int(epochs),
            "max_pairs": int(max_pairs),
            "train_seconds": float(train_s),
            "model_file": path,
            "model_file_mb": float(mb),
        }
        return launder_fn, meta

    print("AI laundering ready (TinyDenoiser, aligned patches).")
else:
    def train_ai_launderer_patchwise(*args, **kwargs):
        print("PyTorch not available; AI laundering disabled.")
        return None, {"enabled": False, "reason": "torch_missing"}
    print("AI laundering disabled (torch missing).")

Threat model loaded. Verifier: Non-blind (cover available)
AI laundering ready (TinyDenoiser, aligned patches).


In [5]:
# ============================================================
# FLEXMark — cGAN (Cell 4): Dataset + Generator + Decoder (DATASET MARK PAYLOAD)
# REQUIREMENTS:
#   - Run Common Cell 1/2 first (read_rgb, ensure_dir, etc.)
#   - Run Common Cell 3 first (TORCH_OK, etc.)  (or at least torch imports)
# ============================================================

assert TORCH_OK,

import hashlib
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

# ---------------------------
# Define HW / EPS
# ---------------------------
HW  = HW  if "HW"  in globals() else 128
EPS = EPS if "EPS" in globals() else 0.02

# ✅ reviewer-aligned small payload
PAYLOAD_BITS = 128  # 128 or 256 only
GRID_SHAPE = (8, 16) if PAYLOAD_BITS == 128 else (16, 16)

CGAN_HW = HW
CGAN_EPS = EPS

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| HW:", CGAN_HW, "| EPS:", CGAN_EPS, "| PAYLOAD_BITS:", PAYLOAD_BITS, "| GRID:", GRID_SHAPE)

# ---------------------------
# Torch helpers
# ---------------------------
def np_to_torch(img_np):
    # (H,W,3) [0,1] -> (3,H,W)
    return torch.from_numpy(img_np).permute(2,0,1).float()

def torch_to_np(img_t):
    # (3,H,W) -> (H,W,3)
    x = img_t.detach().cpu().permute(1,2,0).numpy()
    return np.clip(x, 0, 1)

def upsample_grid_to_hw_1ch(grid01, hw):
    pil = Image.fromarray((grid01 * 255).astype(np.uint8)).resize((hw, hw), Image.NEAREST)
    arr = np.asarray(pil).astype(np.float32) / 255.0
    return arr[..., None]  # (H,W,1)

def bits_to_grid(bits, B):
    if B == 128:
        return bits.reshape(8, 16).astype(np.float32), (8, 16)
    if B == 256:
        return bits.reshape(16, 16).astype(np.float32), (16, 16)
    raise ValueError("B must be 128 or 256")

def mark_to_bits(mark_rgb, B):
    """
    Deterministically derive B bits from the dataset mark image.
    SHA-256 in counter mode -> supports 128/256 bits.
    """
    if B not in (128, 256):
        raise ValueError("Supported B: 128 or 256")

    m = (np.clip(mark_rgb, 0, 1) * 255.0).astype(np.uint8)
    base = m.tobytes()

    need_bytes = B // 8
    out = bytearray()
    ctr = 0
    while len(out) < need_bytes:
        h = hashlib.sha256(base + ctr.to_bytes(4, "little")).digest()
        out.extend(h)
        ctr += 1
    out = bytes(out[:need_bytes])
    bits = np.unpackbits(np.frombuffer(out, dtype=np.uint8)).astype(np.int32)
    return bits[:B]

def build_payload_from_mark(mark_path, hw, B):
    mark_rgb = read_rgb(mark_path, hw)  # from Common Cell 1
    bits = mark_to_bits(mark_rgb, B)
    grid01, (gh, gw) = bits_to_grid(bits, B)
    Wtrue = grid01.astype(np.float32)
    Wup_1ch = upsample_grid_to_hw_1ch(grid01, hw)
    return Wup_1ch, Wtrue, (gh, gw), bits

# ---------------------------
# Dataset: uses COVER + MARK from dataset 
# ---------------------------
class CoverMarkBitsDataset(Dataset):
    def __init__(self, cover_paths, mark_paths, hw, bitsB):
        assert len(cover_paths) == len(mark_paths), 
        self.cover_paths = cover_paths
        self.mark_paths  = mark_paths
        self.hw = int(hw)
        self.bitsB = int(bitsB)

    def __len__(self):
        return len(self.cover_paths)

    def __getitem__(self, idx):
        C = read_rgb(self.cover_paths[idx], self.hw)  # (H,W,3)
        Wup_1ch, Wtrue, meta, bits = build_payload_from_mark(self.mark_paths[idx], self.hw, self.bitsB)

        C_t    = np_to_torch(C)  # (3,H,W)
        Wup_t  = torch.from_numpy(Wup_1ch).permute(2,0,1).float()   # (1,H,W)
        Wtrue_t= torch.from_numpy(Wtrue).float()                    # (gh,gw)
        bits_t = torch.from_numpy(bits.astype(np.float32))          # (B,)
        return C_t, Wup_t, Wtrue_t, bits_t

# ---------------------------
# Model blocks
# ---------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, norm=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, k, s, p)]
        if norm:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.ReLU(inplace=True))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class UNetGenerator(nn.Module):
    def __init__(self, eps=0.02):
        super().__init__()
        self.eps = float(eps)

        self.e1 = ConvBlock(4, 32, s=1)
        self.e2 = ConvBlock(32, 64, s=2)
        self.e3 = ConvBlock(64, 128, s=2)
        self.e4 = ConvBlock(128, 256, s=2)

        self.b1 = ConvBlock(256, 256, s=1)

        self.u3 = nn.ConvTranspose2d(256, 128, 4, 2, 1)
        self.d3 = ConvBlock(256, 128, s=1)

        self.u2 = nn.ConvTranspose2d(128, 64, 4, 2, 1)
        self.d2 = ConvBlock(128, 64, s=1)

        self.u1 = nn.ConvTranspose2d(64, 32, 4, 2, 1)
        self.d1 = ConvBlock(64, 32, s=1)

        self.out = nn.Conv2d(32, 3, 3, 1, 1)

    def forward(self, C, Wup):
        x = torch.cat([C, Wup], dim=1)

        e1 = self.e1(x)
        e2 = self.e2(e1)
        e3 = self.e3(e2)
        e4 = self.e4(e3)

        b  = self.b1(e4)

        u3 = self.u3(b)
        d3 = self.d3(torch.cat([u3, e3], dim=1))

        u2 = self.u2(d3)
        d2 = self.d2(torch.cat([u2, e2], dim=1))

        u1 = self.u1(d2)
        d1 = self.d1(torch.cat([u1, e1], dim=1))

        r = torch.tanh(self.out(d1)) * self.eps
        return r

class ResidualDecoder(nn.Module):
    def __init__(self, gh, gw):
        super().__init__()
        self.gh, self.gw = int(gh), int(gw)
        self.f = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, 2, 1), nn.ReLU(inplace=True),
        )
        self.head = nn.Conv2d(128, 1, 1, 1, 0)

    def forward(self, proxy_1ch):
        z = self.f(proxy_1ch)
        z = self.head(z)
        z = F.interpolate(z, size=(self.gh, self.gw), mode="bilinear", align_corners=False)
        return z.squeeze(1)  # (B,gh,gw)

# ---------------------------
# Instantiate models
# ---------------------------
gh, gw = GRID_SHAPE
G   = UNetGenerator(eps=CGAN_EPS).to(DEVICE)
Dec = ResidualDecoder(gh, gw).to(DEVICE)

print("Models ready:",
      "| G params:", sum(p.numel() for p in G.parameters()),
      "| Dec params:", sum(p.numel() for p in Dec.parameters()))

Device: cuda | HW: 128 | EPS: 0.02 | PAYLOAD_BITS: 128 | GRID: (8, 16)
Models ready: | G params: 2057219 | Dec params: 240385


In [6]:
# ============================================================
# FLEXMark — cGAN (FULL CELL 5)  ✅ UPDATED for dataset-mark payload
# Train (attack-aware) + AI laundering + Evaluate + Plots + Tables
# Outputs:
#   - mean PSNR/SSIM/LPIPS vs cover (imperceptibility)
#   - mean NC/BER/bit_acc per attack (robustness)
#   - per-attack ROC/AUC + confusion @ 1% FPR (decision reliability)
#   - ROC main (clean) + optional overlay ROC (all attacks)
#   - confusion heatmaps (std + laundered)
#   - CSV tables for paper + JSON summary
# ============================================================

assert TORCH_OK, 

import os, time, json, random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from sklearn.metrics import roc_curve, roc_auc_score

# ---------------------------
# Training hyperparams 
# ---------------------------
try:
    BATCH
except NameError:
    BATCH = 16

try:
    EPOCHS
except NameError:
    EPOCHS = 6

try:
    LR_G
except NameError:
    LR_G = 2e-4

try:
    LR_DEC
except NameError:
    LR_DEC = 2e-4


try:
    CGAN_HW
except NameError:
    CGAN_HW = HW

try:
    CGAN_EPS
except NameError:
    CGAN_EPS = EPS

print("Cell5 hyperparams:",
      "BATCH=", BATCH, "EPOCHS=", EPOCHS,
      "LR_G=", LR_G, "LR_DEC=", LR_DEC,
      "CGAN_HW=", CGAN_HW, "CGAN_EPS=", CGAN_EPS)

# ---------------------------
# Helpers
# ---------------------------
def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def confusion_at_fpr(scores_pos, scores_neg, target_fpr=0.01):
    scores_pos = np.asarray(scores_pos, dtype=np.float64)
    scores_neg = np.asarray(scores_neg, dtype=np.float64)

    thr = float(np.quantile(scores_neg, 1.0 - target_fpr))

    TP = int(np.sum(scores_pos >= thr))
    FN = int(np.sum(scores_pos <  thr))
    FP = int(np.sum(scores_neg >= thr))
    TN = int(np.sum(scores_neg <  thr))

    fpr_actual = FP / max(1, (FP + TN))
    tpr_actual = TP / max(1, (TP + FN))

    return {
        "thr": thr,
        "TP": TP, "FP": FP, "TN": TN, "FN": FN,
        "fpr_actual": float(fpr_actual),
        "tpr_actual": float(tpr_actual),
    }

def plot_confusion_heatmap(cm, title, out_path):
    mat = np.array([[cm["TN"], cm["FP"]],
                    [cm["FN"], cm["TP"]]], dtype=np.int64)

    plt.figure(figsize=(4.6, 4.1))
    plt.imshow(mat)
    plt.xticks([0,1], ["Pred 0", "Pred 1"])
    plt.yticks([0,1], ["True 0", "True 1"])
    for (r,c), v in np.ndenumerate(mat):
        plt.text(c, r, str(v), ha="center", va="center", fontsize=11)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=240)
    plt.close()

def plot_attack_metric_line(attacks, series_dict, title, out_path, ylabel=None, ylim=None):
    x = np.arange(len(attacks))
    y = [series_dict[a] for a in attacks]
    plt.figure(figsize=(12,4))
    plt.plot(x, y, marker="o")
    plt.xticks(x, attacks, rotation=45, ha="right")
    if ylabel is not None: plt.ylabel(ylabel)
    if ylim is not None: plt.ylim(*ylim)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=240)
    plt.close()

# ==============
# DataLoaders  
# ==============
train_ds = CoverMarkBitsDataset(trC[:n_train], trW[:n_train], CGAN_HW, PAYLOAD_BITS)
test_ds  = CoverMarkBitsDataset(teC[:n_test],  teW[:n_test],  CGAN_HW, PAYLOAD_BITS)

train_loader = DataLoader(
    train_ds, batch_size=BATCH, shuffle=True,
    num_workers=2, drop_last=True, pin_memory=True
)
test_loader  = DataLoader(
    test_ds, batch_size=1, shuffle=False,
    num_workers=2
)

eval_n = min(EVAL_N, len(test_ds))
print("Using n_train =", len(train_ds), "| n_test =", len(test_ds), "| eval =", eval_n)

# -----------------
# Losses / opts (stable: no discriminator)
# -----------------
bce = nn.BCEWithLogitsLoss()
l1  = nn.L1Loss()

optG   = torch.optim.Adam(G.parameters(),   lr=LR_G,   betas=(0.5, 0.999))
optDec = torch.optim.Adam(Dec.parameters(), lr=LR_DEC, betas=(0.5, 0.999))

# ---------------------------
# Fast torch attacks for TRAINING only (differentiable-ish)
# ---------------------------
def t_attack_resize(x, scale=0.75):
    B,C,H,W = x.shape
    nh, nw = max(4, int(H*scale)), max(4, int(W*scale))
    y = F.interpolate(x, size=(nh,nw), mode="bilinear", align_corners=False)
    y = F.interpolate(y, size=(H,W), mode="bilinear", align_corners=False)
    return y

def t_attack_crop_resize(x, crop_frac=0.10):
    B,C,H,W = x.shape
    ch, cw = max(4, int(H*(1-crop_frac))), max(4, int(W*(1-crop_frac)))
    y0 = torch.randint(0, H-ch+1, (1,), device=x.device).item()
    x0 = torch.randint(0, W-cw+1, (1,), device=x.device).item()
    crop = x[:, :, y0:y0+ch, x0:x0+cw]
    y = F.interpolate(crop, size=(H,W), mode="bilinear", align_corners=False)
    return y

def t_attack_blur(x):
    k = 3
    pad = k//2
    w = torch.ones((x.shape[1],1,k,k), device=x.device) / (k*k)
    return F.conv2d(x, w, padding=pad, groups=x.shape[1])

def t_attack_sharpen(x):
    blur = t_attack_blur(x)
    alpha = 0.7
    return torch.clamp(x + alpha*(x - blur), 0, 1)

def t_attack_gamma(x, gamma=1.2):
    return torch.clamp(x, 0, 1) ** gamma

def t_attack_noise(x, sigma=0.01):
    return torch.clamp(x + sigma*torch.randn_like(x), 0, 1)

TRAIN_ATTACKS = [
    ("clean",     lambda z: z),
    ("resize075", lambda z: t_attack_resize(z, 0.75)),
    ("crop10",    lambda z: t_attack_crop_resize(z, 0.10)),
    ("blur",      lambda z: t_attack_blur(z)),
    ("sharpen",   lambda z: t_attack_sharpen(z)),
    ("gamma12",   lambda z: t_attack_gamma(z, 1.2)),
    ("noise",     lambda z: t_attack_noise(z, 0.01)),
]

def sample_train_attack():
    r = random.random()
    if r < 0.30: return TRAIN_ATTACKS[0]  # clean
    if r < 0.55: return TRAIN_ATTACKS[1]  # resize
    if r < 0.70: return TRAIN_ATTACKS[2]  # crop
    if r < 0.82: return TRAIN_ATTACKS[5]  # gamma
    if r < 0.90: return TRAIN_ATTACKS[3]  # blur
    if r < 0.96: return TRAIN_ATTACKS[4]  # sharpen
    return TRAIN_ATTACKS[6]               # noise

# ---------------------------
# Weights
# ---------------------------
LAMBDA_PAYLOAD = 8.0
LAMBDA_IMP     = 0.6
LAMBDA_RES     = 0.25
LAMBDA_ADV     = 0.0  # unused (no discriminator)

print("TRAIN weights:",
      "payload=", LAMBDA_PAYLOAD,
      "imp=", LAMBDA_IMP,
      "res=", LAMBDA_RES,
      "adv=", LAMBDA_ADV)

# ============================================================
# Train (attack-aware proxy mix)
# ============================================================
ensure_dir(OUT_DIR)
train_tracker = cc_start("train_cgan_attackaware", OUT_DIR)
t0 = time.time()

G.train(); Dec.train()

for ep in range(EPOCHS):
    for step, (C, Wup, Wtrue, bits) in enumerate(train_loader):
        C = C.to(DEVICE, non_blocking=True)          # (B,3,H,W)
        Wup = Wup.to(DEVICE, non_blocking=True)      # (B,1,H,W)
        Wtrue = Wtrue.to(DEVICE, non_blocking=True)  # (B,gh,gw)

        r = G(C, Wup)
        Y = torch.clamp(C + r, 0.0, 1.0)

        loss_imp = l1(Y, C)
        loss_res = torch.mean(torch.abs(r)) / (CGAN_EPS + 1e-8)

        proxy_clean = torch.clamp((Y - C) / (CGAN_EPS + 1e-8), -1.0, 1.0)
        proxy01_clean = (proxy_clean + 1.0) / 2.0
        proxy1_clean = torch.mean(proxy01_clean, dim=1, keepdim=True)  # (B,1,H,W)

        atk_name, atk_fn = sample_train_attack()
        Y_atk = atk_fn(Y)

        proxy_atk = torch.clamp((Y_atk - C) / (CGAN_EPS + 1e-8), -1.0, 1.0)
        proxy01_atk = (proxy_atk + 1.0) / 2.0
        proxy1_atk = torch.mean(proxy01_atk, dim=1, keepdim=True)

        proxy_mix = 0.5 * proxy1_clean + 0.5 * proxy1_atk

        logits_grid = Dec(proxy_mix)
        loss_payload = bce(logits_grid, Wtrue)

        loss = (LAMBDA_IMP * loss_imp +
                LAMBDA_RES * loss_res +
                LAMBDA_PAYLOAD * loss_payload)

        optG.zero_grad(set_to_none=True)
        optDec.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(G.parameters()) + list(Dec.parameters()), 5.0)
        optG.step()
        optDec.step()

    print(f"Epoch {ep+1}/{EPOCHS} | loss={loss.item():.4f} "
          f"(imp={loss_imp.item():.4f}, res={loss_res.item():.4f}, pay={loss_payload.item():.4f}, atk={atk_name})")

train_seconds = time.time() - t0
train_co2 = cc_stop(train_tracker)

# ---------------------------
# Save models
# ---------------------------
G_path   = os.path.join(OUT_DIR, "cgan_G.pt")
Dec_path = os.path.join(OUT_DIR, "cgan_Dec.pt")
torch.save(G.state_dict(), G_path)
torch.save(Dec.state_dict(), Dec_path)

model_file_mb = (os.path.getsize(G_path) + os.path.getsize(Dec_path)) / (1024**2)
print("Saved:", G_path, Dec_path, f"| total {model_file_mb:.2f} MB")

# ============================================================
# AI laundering training (adaptive learned removal attempt)
# ============================================================
AI_LAUNDER_FIT_N = 50
AI_PATCH = 64
AI_PATCHES_PER_IMG = 2
AI_EPOCHS = 1

G.eval(); Dec.eval()
pairs = []
with torch.no_grad():
    for i, (C, Wup, Wtrue, bits) in enumerate(test_loader):
        if i >= min(AI_LAUNDER_FIT_N, eval_n):
            break
        C = C.to(DEVICE); Wup = Wup.to(DEVICE)
        r = G(C, Wup)
        Y = torch.clamp(C + r, 0.0, 1.0)

        C_np = torch_to_np(C[0])
        Y_np = torch_to_np(Y[0])

        Y_att = attack_jpeg(Y_np, 10)
        pairs.append((Y_att, C_np))

launder_fn, launder_meta = train_ai_launderer_patchwise(
    pairs=pairs,
    out_dir=OUT_DIR,
    patch=AI_PATCH,
    patches_per_img=AI_PATCHES_PER_IMG,
    epochs=AI_EPOCHS,
    lr=1e-3,
    max_pairs=min(AI_LAUNDER_FIT_N, len(pairs)),
    device=DEVICE
)
print("AI laundering meta:", launder_meta)

# ============================================================
# Evaluation: attacks only + attacks + laundering
# ============================================================
infer_tracker = cc_start("infer_cgan_attackaware", OUT_DIR)
t0 = time.time()

attacks = list(ATTACKS.keys())
gh, gw = GRID_SHAPE

cached = []
Wtrue_list = []

with torch.no_grad():
    for j, (C, Wup, Wtrue, bits) in enumerate(test_loader):
        if j >= eval_n:
            break
        C = C.to(DEVICE); Wup = Wup.to(DEVICE)
        r = G(C, Wup)
        Y = torch.clamp(C + r, 0.0, 1.0)

        C_np = torch_to_np(C[0])
        Y_np = torch_to_np(Y[0])
        Wtrue_np = Wtrue[0].numpy().astype(np.float32)

        cached.append((C_np, Y_np, Wtrue_np))
        Wtrue_list.append(Wtrue_np)

Wtrue_wrong = Wtrue_list[1:] + Wtrue_list[:1]

psnr_list, ssim_list, lpips_list = [], [], []
nc_acc = {a: [] for a in attacks}
ber_acc = {a: [] for a in attacks}
bitacc_acc = {a: [] for a in attacks}

det_pos = {a: [] for a in attacks}
det_neg = {a: [] for a in attacks}

ncL_acc = {a: [] for a in attacks}
berL_acc = {a: [] for a in attacks}
bitaccL_acc = {a: [] for a in attacks}

detL_pos = {a: [] for a in attacks}
detL_neg = {a: [] for a in attacks}

# === REPLACE ONLY THIS FUNCTION IN CELL 5 ===
def _decode_metrics(Ya_np, C_np, Wtrue_np, Wwrong_np):
    # numpy -> torch (1,3,H,W)
    Ya_t = np_to_torch(Ya_np).unsqueeze(0).to(DEVICE)
    C_t  = np_to_torch(C_np).unsqueeze(0).to(DEVICE)

    # Build residual proxy exactly aligned with training
    proxy = torch.clamp((Ya_t - C_t) / (CGAN_EPS + 1e-8), -1.0, 1.0)
    proxy01 = (proxy + 1.0) / 2.0
    proxy1  = torch.mean(proxy01, dim=1, keepdim=True)  # (1,1,H,W)

    # Decode with trained Dec
    logits = Dec(proxy1)                  # (1,gh,gw)
    probs  = torch.sigmoid(logits)[0].detach().cpu().numpy().astype(np.float32)  # (gh,gw)

    # Binarize recovered grid
    rec_bin   = binarize01(probs, 0.5)
    true_bin  = binarize01(Wtrue_np, 0.5)
    wrong_bin = binarize01(Wwrong_np, 0.5)

    # Metrics (keep same contract as before)
    nc_true = float(nc(rec_bin, true_bin))
    ber_true = float(ber(rec_bin, true_bin))
    bitacc_true = float(100.0 - ber_true)

    nc_wrong = float(nc(rec_bin, wrong_bin))
    return nc_true, ber_true, bitacc_true, nc_wrong

for i, (C_np, Y_np, Wtrue_np) in enumerate(cached):
    psnr_list.append(psnr(Y_np, C_np))
    ssim_list.append(ssim_np(Y_np, C_np))
    lpips_list.append(lpips_score_np(Y_np, C_np, device=DEVICE) if (TORCH_OK and LPIPS_OK) else float("nan"))

    for atk_name, atk_fn in ATTACKS.items():
        Ya = atk_fn(Y_np)

        nc_v, ber_v, bitacc_v, nc_wrong_v = _decode_metrics(Ya, C_np, Wtrue_np, Wtrue_wrong[i])
        nc_acc[atk_name].append(nc_v)
        ber_acc[atk_name].append(ber_v)
        bitacc_acc[atk_name].append(bitacc_v)

        det_pos[atk_name].append(nc_v)
        det_neg[atk_name].append(nc_wrong_v)

        if launder_fn is not None and atk_name != "clean":
            YaL = np.clip(launder_fn(Ya), 0, 1)
        else:
            YaL = Ya

        ncL, berL, bitaccL, nc_wrong_L = _decode_metrics(YaL, C_np, Wtrue_np, Wtrue_wrong[i])
        ncL_acc[atk_name].append(ncL)
        berL_acc[atk_name].append(berL)
        bitaccL_acc[atk_name].append(bitaccL)

        detL_pos[atk_name].append(ncL)
        detL_neg[atk_name].append(nc_wrong_L)

infer_seconds = time.time() - t0
infer_co2 = cc_stop(infer_tracker)

# ================
# Summary
# ================
summary = {
    "model": "cgan_cond_residual_eps_nonblind_attackaware",
    "payload_bits": int(PAYLOAD_BITS),
    "eps": float(CGAN_EPS),
    "hw": int(CGAN_HW),
    "n_train": int(len(train_ds)),
    "n_test_eval": int(len(cached)),

    "mean_psnr_cover": float(np.mean(psnr_list)),
    "mean_ssim_cover": float(np.mean(ssim_list)),
    "mean_lpips_cover": float(np.nanmean(lpips_list)),

    "mean_nc":      {a: float(np.mean(nc_acc[a])) for a in attacks},
    "mean_ber":     {a: float(np.mean(ber_acc[a])) for a in attacks},
    "mean_bit_acc": {a: float(np.mean(bitacc_acc[a])) for a in attacks},

    "mean_nc_laundered":      {a: float(np.mean(ncL_acc[a])) for a in attacks},
    "mean_ber_laundered":     {a: float(np.mean(berL_acc[a])) for a in attacks},
    "mean_bit_acc_laundered": {a: float(np.mean(bitaccL_acc[a])) for a in attacks},

    "train_seconds": float(train_seconds),
    "infer_seconds": float(infer_seconds),
    "train_co2_kg": float(train_co2) if train_co2 is not None else None,
    "infer_co2_kg": float(infer_co2) if infer_co2 is not None else None,
    "model_file_mb": float(model_file_mb),

    "ai_laundering_meta": launder_meta,

    "train_recipe": {
        "lambda_payload": float(LAMBDA_PAYLOAD),
        "lambda_imp": float(LAMBDA_IMP),
        "lambda_res": float(LAMBDA_RES),
        "lambda_adv": float(LAMBDA_ADV),
        "payload_proxy_mix": "0.5*clean + 0.5*attacked"
    },

    "detector_auc_by_attack": {},
    "detector_confusion_1pctfpr_by_attack": {},
    "detector_auc_by_attack_laundered": {},
    "detector_confusion_1pctfpr_by_attack_laundered": {},
}

# ============================================================
# Decision metrics + heatmaps
# ============================================================
for atk_name in attacks:
    pos = np.asarray(det_pos[atk_name], dtype=np.float64)
    neg = np.asarray(det_neg[atk_name], dtype=np.float64)

    y_true = np.array([1]*len(pos) + [0]*len(neg))
    y_score = np.array(list(pos) + list(neg))

    A = float(roc_auc_score(y_true, y_score))
    cm = confusion_at_fpr(pos, neg, target_fpr=0.01)

    summary["detector_auc_by_attack"][atk_name] = A
    summary["detector_confusion_1pctfpr_by_attack"][atk_name] = cm

    plot_confusion_heatmap(cm,
        f"cGAN Confusion @1%FPR — {atk_name}",
        os.path.join(OUT_DIR, f"cgan_confusion_standard_{atk_name}.png")
    )

    posL = np.asarray(detL_pos[atk_name], dtype=np.float64)
    negL = np.asarray(detL_neg[atk_name], dtype=np.float64)
    y_trueL = np.array([1]*len(posL) + [0]*len(negL))
    y_scoreL = np.array(list(posL) + list(negL))

    AL = float(roc_auc_score(y_trueL, y_scoreL))
    cmL = confusion_at_fpr(posL, negL, target_fpr=0.01)

    summary["detector_auc_by_attack_laundered"][atk_name] = AL
    summary["detector_confusion_1pctfpr_by_attack_laundered"][atk_name] = cmL

    plot_confusion_heatmap(cmL,
        f"cGAN Confusion @1%FPR (Laundered) — {atk_name}",
        os.path.join(OUT_DIR, f"cgan_confusion_laundered_{atk_name}.png")
    )

# ============================================================
# ROC plot (Clean) — main
# ============================================================
pos_clean = np.asarray(det_pos["clean"], dtype=np.float64)
neg_clean = np.asarray(det_neg["clean"], dtype=np.float64)
y_true_clean = np.array([1]*len(pos_clean) + [0]*len(neg_clean))
y_score_clean = np.array(list(pos_clean) + list(neg_clean))

fpr, tpr, _ = roc_curve(y_true_clean, y_score_clean)
auc_clean = float(roc_auc_score(y_true_clean, y_score_clean))

plt.figure()
plt.plot(fpr, tpr, label=f"Clean AUC={auc_clean:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("cGAN Detection ROC (clean): correct vs wrong payload")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "cgan_detection_roc_clean.png"), dpi=300)
plt.close()

# OPTIONAL: ROC overlay (all attacks)
plt.figure(figsize=(7,6))
for atk_name in attacks:
    pos = np.asarray(det_pos[atk_name], dtype=np.float64)
    neg = np.asarray(det_neg[atk_name], dtype=np.float64)
    y_true = np.array([1]*len(pos) + [0]*len(neg))
    y_score = np.array(list(pos) + list(neg))
    fpr, tpr, _ = roc_curve(y_true, y_score)
    A = summary["detector_auc_by_attack"][atk_name]
    plt.plot(fpr, tpr, label=f"{atk_name} (AUC={A:.2f})")
plt.plot([0,1],[0,1], linestyle="--")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title("cGAN Detector ROC — All Attacks (Standard)")
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "cgan_detector_roc_all_attacks.png"), dpi=300)
plt.close()

# ============================================================
# Plots: NC / BER / BitAcc (standard + laundered)
# ============================================================
plot_attack_metric_line(attacks, summary["mean_nc"],
    "cGAN mean NC under attacks (standard)",
    os.path.join(OUT_DIR, "cgan_NC_attacks.png"),
    ylabel="NC", ylim=(-1, 1)
)
plot_attack_metric_line(attacks, summary["mean_ber"],
    "cGAN mean BER (%) under attacks (standard)",
    os.path.join(OUT_DIR, "cgan_BER_attacks.png"),
    ylabel="BER (%)", ylim=(0, 100)
)
plot_attack_metric_line(attacks, summary["mean_bit_acc"],
    "cGAN mean Bit Accuracy (%) under attacks (standard)",
    os.path.join(OUT_DIR, "cgan_bitacc_attacks.png"),
    ylabel="Bit Accuracy (%)", ylim=(0, 105)
)

plot_attack_metric_line(attacks, summary["mean_nc_laundered"],
    "cGAN mean NC under attacks (after laundering)",
    os.path.join(OUT_DIR, "cgan_NC_attacks_laundered.png"),
    ylabel="NC", ylim=(-1, 1)
)
plot_attack_metric_line(attacks, summary["mean_ber_laundered"],
    "cGAN mean BER (%) under attacks (after laundering)",
    os.path.join(OUT_DIR, "cgan_BER_attacks_laundered.png"),
    ylabel="BER (%)", ylim=(0, 100)
)
plot_attack_metric_line(attacks, summary["mean_bit_acc_laundered"],
    "cGAN mean Bit Accuracy (%) under attacks (after laundering)",
    os.path.join(OUT_DIR, "cgan_bitacc_attacks_laundered.png"),
    ylabel="Bit Accuracy (%)", ylim=(0, 105)
)

# Delta BitAcc
bit_std = np.array([summary["mean_bit_acc"][a] for a in attacks], dtype=float)
bit_lau = np.array([summary["mean_bit_acc_laundered"][a] for a in attacks], dtype=float)
delta = bit_std - bit_lau

plt.figure(figsize=(12,4))
plt.bar(attacks, delta)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Bit-Accuracy Drop (pp)")
plt.title("AI Laundering Impact: Δ(BitAcc) = Standard − Laundered (cGAN)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "cgan_delta_bitacc_standard_minus_laundered.png"), dpi=240)
plt.close()

# PSNR histogram
plt.figure(figsize=(6,4))
plt.hist(psnr_list, bins=30)
plt.xlabel("PSNR vs cover (dB)")
plt.ylabel("count")
plt.title("cGAN cover-referenced imperceptibility (PSNR)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "cgan_imperceptibility_psnr_hist.png"), dpi=240)
plt.close()

# ============================================================
# Export CSV tables (paper-friendly)
# ============================================================
try:
    import pandas as pd

    rob_rows = []
    for a in attacks:
        rob_rows.append({
            "attack": a,
            "NC_standard": summary["mean_nc"][a],
            "BER_standard": summary["mean_ber"][a],
            "BitAcc_standard": summary["mean_bit_acc"][a],
            "NC_laundered": summary["mean_nc_laundered"][a],
            "BER_laundered": summary["mean_ber_laundered"][a],
            "BitAcc_laundered": summary["mean_bit_acc_laundered"][a],
        })
    pd.DataFrame(rob_rows).to_csv(os.path.join(OUT_DIR, "cgan_robustness_table.csv"), index=False)

    det_rows = []
    for a in attacks:
        cm  = summary["detector_confusion_1pctfpr_by_attack"][a]
        cmL = summary["detector_confusion_1pctfpr_by_attack_laundered"][a]
        det_rows.append({
            "attack": a,
            "AUC_standard": summary["detector_auc_by_attack"][a],
            "thr_standard": cm["thr"],
            "TPR@1%FPR_standard": cm["tpr_actual"],
            "FPR@1%FPR_standard": cm["fpr_actual"],
            "AUC_laundered": summary["detector_auc_by_attack_laundered"][a],
            "thr_laundered": cmL["thr"],
            "TPR@1%FPR_laundered": cmL["tpr_actual"],
            "FPR@1%FPR_laundered": cmL["fpr_actual"],
        })
    pd.DataFrame(det_rows).to_csv(os.path.join(OUT_DIR, "cgan_detector_table.csv"), index=False)

except Exception as e:
    print("CSV export skipped (pandas missing?):", repr(e))

# ============================================================
# Save final summary JSON
# ============================================================
save_json(summary, os.path.join(OUT_DIR, "cgan_summary.json"))
print(json.dumps(summary, indent=2))
print("✅ Outputs in:", OUT_DIR)

Cell5 hyperparams: BATCH= 16 EPOCHS= 6 LR_G= 0.0002 LR_DEC= 0.0002 CGAN_HW= 128 CGAN_EPS= 0.02
Using n_train = 5000 | n_test = 1000 | eval = 1000


[codecarbon WARNING @ 17:03:32] Multiple instances of codecarbon are allowed to run at the same time.


TRAIN weights: payload= 8.0 imp= 0.6 res= 0.25 adv= 0.0
Epoch 1/6 | loss=0.0437 (imp=0.0032, res=0.1669, pay=0.0000, atk=clean)
Epoch 2/6 | loss=0.0241 (imp=0.0018, res=0.0921, pay=0.0000, atk=clean)
Epoch 3/6 | loss=0.0245 (imp=0.0019, res=0.0937, pay=0.0000, atk=noise)
Epoch 4/6 | loss=0.0405 (imp=0.0017, res=0.0852, pay=0.0023, atk=blur)
Epoch 5/6 | loss=0.0206 (imp=0.0015, res=0.0783, pay=0.0000, atk=gamma12)
Epoch 6/6 | loss=0.0255 (imp=0.0015, res=0.0750, pay=0.0007, atk=sharpen)
Saved: /kaggle/working/flexmark_out_bits128/cgan_G.pt /kaggle/working/flexmark_out_bits128/cgan_Dec.pt | total 8.79 MB
AI laundering meta: {'enabled': True, 'device': 'cuda', 'patch': 64, 'patches_per_img': 2, 'epochs': 1, 'max_pairs': 50, 'train_seconds': 0.18672537803649902, 'model_file': '/kaggle/working/flexmark_out_bits128/ai_launder_tinydenoiser.pt', 'model_file_mb': 0.04524707794189453}


/tmp/ipykernel_55/345400547.py:455: RuntimeWarning: Mean of empty slice
  "mean_lpips_cover": float(np.nanmean(lpips_list)),


{
  "model": "cgan_cond_residual_eps_nonblind_attackaware",
  "payload_bits": 128,
  "eps": 0.02,
  "hw": 128,
  "n_train": 5000,
  "n_test_eval": 1000,
  "mean_psnr_cover": 47.974823397031656,
  "mean_ssim_cover": 0.9960154550671577,
  "mean_lpips_cover": NaN,
  "mean_nc": {
    "clean": 1.0000000229477883,
    "jpeg10": 0.5096007011532784,
    "jpeg30": 0.5674790821820498,
    "jpeg50": 0.6215821987688541,
    "jpeg70": 0.7011496389508247,
    "jpeg90": 0.9560715764164924,
    "resize075": 0.7288286573290825,
    "crop10": 0.48184205511212347,
    "blur": 0.5988381927609444,
    "sharpen": 0.9942581349611282,
    "gamma12": 0.5475692704916001,
    "screenshot": 0.5713130072951317,
    "launder_only": 1.0000000229477883
  },
  "mean_ber": {
    "clean": 0.0,
    "jpeg10": 47.7765625,
    "jpeg30": 41.85703125,
    "jpeg50": 37.02265625,
    "jpeg70": 29.4625,
    "jpeg90": 4.409375,
    "resize075": 26.578125,
    "crop10": 49.50859375,
    "blur": 39.1,
    "sharpen": 0.57421875,
   

### 256 bits for cGAN

In [8]:
# ============================================================
# FLEXMark — cGAN (Cell 4): Dataset + Generator + Decoder (DATASET MARK PAYLOAD)
# REQUIREMENTS:
#   - Run Common Cell 1/2 first (read_rgb, ensure_dir, etc.)
#   - Run Common Cell 3 first (TORCH_OK, etc.)  (or at least torch imports)
# ============================================================

assert TORCH_OK, "PyTorch required (TORCH_OK=False). Make sure torch is available."

import hashlib
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset


HW  = HW  if "HW"  in globals() else 128
EPS = EPS if "EPS" in globals() else 0.02


PAYLOAD_BITS = 256  # 128 or 256 only
GRID_SHAPE = (8, 16) if PAYLOAD_BITS == 128 else (16, 16)

CGAN_HW = HW
CGAN_EPS = EPS

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| HW:", CGAN_HW, "| EPS:", CGAN_EPS, "| PAYLOAD_BITS:", PAYLOAD_BITS, "| GRID:", GRID_SHAPE)

# ---------------------------
# Torch helpers
# ---------------------------
def np_to_torch(img_np):
    # (H,W,3) [0,1] -> (3,H,W)
    return torch.from_numpy(img_np).permute(2,0,1).float()

def torch_to_np(img_t):
    # (3,H,W) -> (H,W,3)
    x = img_t.detach().cpu().permute(1,2,0).numpy()
    return np.clip(x, 0, 1)

def upsample_grid_to_hw_1ch(grid01, hw):
    pil = Image.fromarray((grid01 * 255).astype(np.uint8)).resize((hw, hw), Image.NEAREST)
    arr = np.asarray(pil).astype(np.float32) / 255.0
    return arr[..., None]  # (H,W,1)

def bits_to_grid(bits, B):
    if B == 128:
        return bits.reshape(8, 16).astype(np.float32), (8, 16)
    if B == 256:
        return bits.reshape(16, 16).astype(np.float32), (16, 16)
    raise ValueError("B must be 128 or 256")

def mark_to_bits(mark_rgb, B):
    """
    Deterministically derive B bits from the dataset mark image.
    SHA-256 in counter mode -> supports 128/256 bits.
    """
    if B not in (128, 256):
        raise ValueError("Supported B: 128 or 256")

    m = (np.clip(mark_rgb, 0, 1) * 255.0).astype(np.uint8)
    base = m.tobytes()

    need_bytes = B // 8
    out = bytearray()
    ctr = 0
    while len(out) < need_bytes:
        h = hashlib.sha256(base + ctr.to_bytes(4, "little")).digest()
        out.extend(h)
        ctr += 1
    out = bytes(out[:need_bytes])
    bits = np.unpackbits(np.frombuffer(out, dtype=np.uint8)).astype(np.int32)
    return bits[:B]

def build_payload_from_mark(mark_path, hw, B):
    """
    mark_path: dataset mark PNG
    Returns:
      Wup_1ch: (H,W,1) float [0,1]
      Wtrue:   (gh,gw) float [0,1]
      meta:    (gh,gw)
      bits:    (B,) int32
    """
    mark_rgb = read_rgb(mark_path, hw) 
    bits = mark_to_bits(mark_rgb, B)
    grid01, (gh, gw) = bits_to_grid(bits, B)
    Wtrue = grid01.astype(np.float32)
    Wup_1ch = upsample_grid_to_hw_1ch(grid01, hw)
    return Wup_1ch, Wtrue, (gh, gw), bits

# ---------------------------
# Dataset: uses COVER + MARK from dataset 
# ---------------------------
class CoverMarkBitsDataset(Dataset):
    def __init__(self, cover_paths, mark_paths, hw, bitsB):
        assert len(cover_paths) == len(mark_paths), "cover/mark lists must align"
        self.cover_paths = cover_paths
        self.mark_paths  = mark_paths
        self.hw = int(hw)
        self.bitsB = int(bitsB)

    def __len__(self):
        return len(self.cover_paths)

    def __getitem__(self, idx):
        C = read_rgb(self.cover_paths[idx], self.hw)  # (H,W,3)
        Wup_1ch, Wtrue, meta, bits = build_payload_from_mark(self.mark_paths[idx], self.hw, self.bitsB)

        C_t    = np_to_torch(C)  # (3,H,W)
        Wup_t  = torch.from_numpy(Wup_1ch).permute(2,0,1).float()   # (1,H,W)
        Wtrue_t= torch.from_numpy(Wtrue).float()                    # (gh,gw)
        bits_t = torch.from_numpy(bits.astype(np.float32))          # (B,)
        return C_t, Wup_t, Wtrue_t, bits_t

# ---------------------------
# Model blocks
# ---------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, norm=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, k, s, p)]
        if norm:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.ReLU(inplace=True))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class UNetGenerator(nn.Module):
    """
    Input: [C (3ch), W_up (1ch)] => 4ch
    Output: residual r (3ch) bounded by eps via tanh scaling
    """
    def __init__(self, eps=0.02):
        super().__init__()
        self.eps = float(eps)

        self.e1 = ConvBlock(4, 32, s=1)
        self.e2 = ConvBlock(32, 64, s=2)
        self.e3 = ConvBlock(64, 128, s=2)
        self.e4 = ConvBlock(128, 256, s=2)

        self.b1 = ConvBlock(256, 256, s=1)

        self.u3 = nn.ConvTranspose2d(256, 128, 4, 2, 1)
        self.d3 = ConvBlock(256, 128, s=1)

        self.u2 = nn.ConvTranspose2d(128, 64, 4, 2, 1)
        self.d2 = ConvBlock(128, 64, s=1)

        self.u1 = nn.ConvTranspose2d(64, 32, 4, 2, 1)
        self.d1 = ConvBlock(64, 32, s=1)

        self.out = nn.Conv2d(32, 3, 3, 1, 1)

    def forward(self, C, Wup):
        x = torch.cat([C, Wup], dim=1)

        e1 = self.e1(x)
        e2 = self.e2(e1)
        e3 = self.e3(e2)
        e4 = self.e4(e3)

        b  = self.b1(e4)

        u3 = self.u3(b)
        d3 = self.d3(torch.cat([u3, e3], dim=1))

        u2 = self.u2(d3)
        d2 = self.d2(torch.cat([u2, e2], dim=1))

        u1 = self.u1(d2)
        d1 = self.d1(torch.cat([u1, e1], dim=1))

        r = torch.tanh(self.out(d1)) * self.eps
        return r

class ResidualDecoder(nn.Module):
    """
    Decoder recovers payload grid from residual proxy (grayscale proxy in [0,1]).
    Output: logits grid (B,gh,gw) for BCEWithLogitsLoss against Wtrue (0/1).
    """
    def __init__(self, gh, gw):
        super().__init__()
        self.gh, self.gw = int(gh), int(gw)
        self.f = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, 2, 1), nn.ReLU(inplace=True),
        )
        self.head = nn.Conv2d(128, 1, 1, 1, 0)

    def forward(self, proxy_1ch):
        z = self.f(proxy_1ch)
        z = self.head(z)
        z = F.interpolate(z, size=(self.gh, self.gw), mode="bilinear", align_corners=False)
        return z.squeeze(1)  # (B,gh,gw)

# ---------------------------
# Instantiate models
# ---------------------------
gh, gw = GRID_SHAPE
G   = UNetGenerator(eps=CGAN_EPS).to(DEVICE)
Dec = ResidualDecoder(gh, gw).to(DEVICE)

print("Models ready:",
      "| G params:", sum(p.numel() for p in G.parameters()),
      "| Dec params:", sum(p.numel() for p in Dec.parameters()))

Device: cuda | HW: 128 | EPS: 0.02 | PAYLOAD_BITS: 256 | GRID: (16, 16)
Models ready: | G params: 2057219 | Dec params: 240385


In [9]:
# ============================================================
# FLEXMark — cGAN (FULL CELL 5) 
# Train (attack-aware) + AI laundering + Evaluate + Plots + Tables
# Outputs:
#   - mean PSNR/SSIM/LPIPS vs cover (imperceptibility)
#   - mean NC/BER/bit_acc per attack (robustness)
#   - per-attack ROC/AUC + confusion @ 1% FPR (decision reliability)
#   - ROC main (clean) + optional overlay ROC (all attacks)
#   - confusion heatmaps (std + laundered)
#   - CSV tables for paper + JSON summary
# ============================================================

assert TORCH_OK, "PyTorch required (TORCH_OK=False)."

import os, time, json, random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from sklearn.metrics import roc_curve, roc_auc_score

# ---------------------------
# Training hyperparams
# ---------------------------
try:
    BATCH
except NameError:
    BATCH = 16

try:
    EPOCHS
except NameError:
    EPOCHS = 6

try:
    LR_G
except NameError:
    LR_G = 2e-4

try:
    LR_DEC
except NameError:
    LR_DEC = 2e-4


try:
    CGAN_HW
except NameError:
    CGAN_HW = HW

try:
    CGAN_EPS
except NameError:
    CGAN_EPS = EPS

print("Cell5 hyperparams:",
      "BATCH=", BATCH, "EPOCHS=", EPOCHS,
      "LR_G=", LR_G, "LR_DEC=", LR_DEC,
      "CGAN_HW=", CGAN_HW, "CGAN_EPS=", CGAN_EPS)

# ---------------------------
# Helpers
# ---------------------------
def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def confusion_at_fpr(scores_pos, scores_neg, target_fpr=0.01):
    """
    Quantile-based thresholding on NEG scores to achieve ~ target FPR.
    Returns dict with thr, TP/FP/TN/FN and actual fpr/tpr.
    """
    scores_pos = np.asarray(scores_pos, dtype=np.float64)
    scores_neg = np.asarray(scores_neg, dtype=np.float64)

    thr = float(np.quantile(scores_neg, 1.0 - target_fpr))

    TP = int(np.sum(scores_pos >= thr))
    FN = int(np.sum(scores_pos <  thr))
    FP = int(np.sum(scores_neg >= thr))
    TN = int(np.sum(scores_neg <  thr))

    fpr_actual = FP / max(1, (FP + TN))
    tpr_actual = TP / max(1, (TP + FN))

    return {
        "thr": thr,
        "TP": TP, "FP": FP, "TN": TN, "FN": FN,
        "fpr_actual": float(fpr_actual),
        "tpr_actual": float(tpr_actual),
    }

def plot_confusion_heatmap(cm, title, out_path):
    """
    cm is dict from confusion_at_fpr. We plot:
    [[TN, FP],
     [FN, TP]]
    """
    mat = np.array([[cm["TN"], cm["FP"]],
                    [cm["FN"], cm["TP"]]], dtype=np.int64)

    plt.figure(figsize=(4.6, 4.1))
    plt.imshow(mat)
    plt.xticks([0,1], ["Pred 0", "Pred 1"])
    plt.yticks([0,1], ["True 0", "True 1"])
    for (r,c), v in np.ndenumerate(mat):
        plt.text(c, r, str(v), ha="center", va="center", fontsize=11)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=240)
    plt.close()

def plot_attack_metric_line(attacks, series_dict, title, out_path, ylabel=None, ylim=None):
    x = np.arange(len(attacks))
    y = [series_dict[a] for a in attacks]
    plt.figure(figsize=(12,4))
    plt.plot(x, y, marker="o")
    plt.xticks(x, attacks, rotation=45, ha="right")
    if ylabel is not None: plt.ylabel(ylabel)
    if ylim is not None: plt.ylim(*ylim)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=240)
    plt.close()

# ============================================================
# DataLoaders
# ============================================================
train_ds = CoverMarkBitsDataset(trC[:n_train], trW[:n_train], CGAN_HW, PAYLOAD_BITS)
test_ds  = CoverMarkBitsDataset(teC[:n_test],  teW[:n_test],  CGAN_HW, PAYLOAD_BITS)

train_loader = DataLoader(
    train_ds, batch_size=BATCH, shuffle=True,
    num_workers=2, drop_last=True, pin_memory=True
)
test_loader  = DataLoader(
    test_ds, batch_size=1, shuffle=False,
    num_workers=2
)

eval_n = min(EVAL_N, len(test_ds))
print("Using n_train =", len(train_ds), "| n_test =", len(test_ds), "| eval =", eval_n)

# ---------------------------
# Losses / opts (stable: no discriminator)
# ---------------------------
bce = nn.BCEWithLogitsLoss()
l1  = nn.L1Loss()

optG   = torch.optim.Adam(G.parameters(),   lr=LR_G,   betas=(0.5, 0.999))
optDec = torch.optim.Adam(Dec.parameters(), lr=LR_DEC, betas=(0.5, 0.999))

# ---------------------------
# Fast torch attacks for TRAINING only (differentiable-ish)
# ---------------------------
def t_attack_resize(x, scale=0.75):
    B,C,H,W = x.shape
    nh, nw = max(4, int(H*scale)), max(4, int(W*scale))
    y = F.interpolate(x, size=(nh,nw), mode="bilinear", align_corners=False)
    y = F.interpolate(y, size=(H,W), mode="bilinear", align_corners=False)
    return y

def t_attack_crop_resize(x, crop_frac=0.10):
    B,C,H,W = x.shape
    ch, cw = max(4, int(H*(1-crop_frac))), max(4, int(W*(1-crop_frac)))
    y0 = torch.randint(0, H-ch+1, (1,), device=x.device).item()
    x0 = torch.randint(0, W-cw+1, (1,), device=x.device).item()
    crop = x[:, :, y0:y0+ch, x0:x0+cw]
    y = F.interpolate(crop, size=(H,W), mode="bilinear", align_corners=False)
    return y

def t_attack_blur(x):
    k = 3
    pad = k//2
    w = torch.ones((x.shape[1],1,k,k), device=x.device) / (k*k)
    return F.conv2d(x, w, padding=pad, groups=x.shape[1])

def t_attack_sharpen(x):
    blur = t_attack_blur(x)
    alpha = 0.7
    return torch.clamp(x + alpha*(x - blur), 0, 1)

def t_attack_gamma(x, gamma=1.2):
    return torch.clamp(x, 0, 1) ** gamma

def t_attack_noise(x, sigma=0.01):
    return torch.clamp(x + sigma*torch.randn_like(x), 0, 1)

TRAIN_ATTACKS = [
    ("clean",     lambda z: z),
    ("resize075", lambda z: t_attack_resize(z, 0.75)),
    ("crop10",    lambda z: t_attack_crop_resize(z, 0.10)),
    ("blur",      lambda z: t_attack_blur(z)),
    ("sharpen",   lambda z: t_attack_sharpen(z)),
    ("gamma12",   lambda z: t_attack_gamma(z, 1.2)),
    ("noise",     lambda z: t_attack_noise(z, 0.01)),
]

def sample_train_attack():
    r = random.random()
    if r < 0.30: return TRAIN_ATTACKS[0]  # clean
    if r < 0.55: return TRAIN_ATTACKS[1]  # resize
    if r < 0.70: return TRAIN_ATTACKS[2]  # crop
    if r < 0.82: return TRAIN_ATTACKS[5]  # gamma
    if r < 0.90: return TRAIN_ATTACKS[3]  # blur
    if r < 0.96: return TRAIN_ATTACKS[4]  # sharpen
    return TRAIN_ATTACKS[6]               # noise

# ---------------------------
# Weights
# ---------------------------
LAMBDA_PAYLOAD = 8.0
LAMBDA_IMP     = 0.6
LAMBDA_RES     = 0.25
LAMBDA_ADV     = 0.0  # unused (no discriminator)

print("TRAIN weights:",
      "payload=", LAMBDA_PAYLOAD,
      "imp=", LAMBDA_IMP,
      "res=", LAMBDA_RES,
      "adv=", LAMBDA_ADV)

# ============================================================
# Train (attack-aware proxy mix)
# ============================================================
ensure_dir(OUT_DIR)
train_tracker = cc_start("train_cgan_attackaware", OUT_DIR)
t0 = time.time()

G.train(); Dec.train()

for ep in range(EPOCHS):
    for step, (C, Wup, Wtrue, bits) in enumerate(train_loader):
        C = C.to(DEVICE, non_blocking=True)          # (B,3,H,W)
        Wup = Wup.to(DEVICE, non_blocking=True)      # (B,1,H,W)
        Wtrue = Wtrue.to(DEVICE, non_blocking=True)  # (B,gh,gw)

        r = G(C, Wup)
        Y = torch.clamp(C + r, 0.0, 1.0)

        loss_imp = l1(Y, C)
        loss_res = torch.mean(torch.abs(r)) / (CGAN_EPS + 1e-8)

        proxy_clean = torch.clamp((Y - C) / (CGAN_EPS + 1e-8), -1.0, 1.0)
        proxy01_clean = (proxy_clean + 1.0) / 2.0
        proxy1_clean = torch.mean(proxy01_clean, dim=1, keepdim=True)  # (B,1,H,W)

        atk_name, atk_fn = sample_train_attack()
        Y_atk = atk_fn(Y)

        proxy_atk = torch.clamp((Y_atk - C) / (CGAN_EPS + 1e-8), -1.0, 1.0)
        proxy01_atk = (proxy_atk + 1.0) / 2.0
        proxy1_atk = torch.mean(proxy01_atk, dim=1, keepdim=True)

        proxy_mix = 0.5 * proxy1_clean + 0.5 * proxy1_atk

        logits_grid = Dec(proxy_mix)
        loss_payload = bce(logits_grid, Wtrue)

        loss = (LAMBDA_IMP * loss_imp +
                LAMBDA_RES * loss_res +
                LAMBDA_PAYLOAD * loss_payload)

        optG.zero_grad(set_to_none=True)
        optDec.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(G.parameters()) + list(Dec.parameters()), 5.0)
        optG.step()
        optDec.step()

    print(f"Epoch {ep+1}/{EPOCHS} | loss={loss.item():.4f} "
          f"(imp={loss_imp.item():.4f}, res={loss_res.item():.4f}, pay={loss_payload.item():.4f}, atk={atk_name})")

train_seconds = time.time() - t0
train_co2 = cc_stop(train_tracker)

# ---------------------------
# Save models
# ---------------------------
G_path   = os.path.join(OUT_DIR, "cgan_G.pt")
Dec_path = os.path.join(OUT_DIR, "cgan_Dec.pt")
torch.save(G.state_dict(), G_path)
torch.save(Dec.state_dict(), Dec_path)

model_file_mb = (os.path.getsize(G_path) + os.path.getsize(Dec_path)) / (1024**2)
print("Saved:", G_path, Dec_path, f"| total {model_file_mb:.2f} MB")

# ============================================================
# AI laundering training (adaptive learned removal attempt)
# ============================================================
AI_LAUNDER_FIT_N = 50
AI_PATCH = 64
AI_PATCHES_PER_IMG = 2
AI_EPOCHS = 1

G.eval(); Dec.eval()
pairs = []
with torch.no_grad():
    for i, (C, Wup, Wtrue, bits) in enumerate(test_loader):
        if i >= min(AI_LAUNDER_FIT_N, eval_n):
            break
        C = C.to(DEVICE); Wup = Wup.to(DEVICE)
        r = G(C, Wup)
        Y = torch.clamp(C + r, 0.0, 1.0)

        C_np = torch_to_np(C[0])
        Y_np = torch_to_np(Y[0])

        Y_att = attack_jpeg(Y_np, 10)
        pairs.append((Y_att, C_np))

launder_fn, launder_meta = train_ai_launderer_patchwise(
    pairs=pairs,
    out_dir=OUT_DIR,
    patch=AI_PATCH,
    patches_per_img=AI_PATCHES_PER_IMG,
    epochs=AI_EPOCHS,
    lr=1e-3,
    max_pairs=min(AI_LAUNDER_FIT_N, len(pairs)),
    device=DEVICE
)
print("AI laundering meta:", launder_meta)

# ============================================================
# Evaluation: attacks only + attacks + laundering
# ============================================================
infer_tracker = cc_start("infer_cgan_attackaware", OUT_DIR)
t0 = time.time()

attacks = list(ATTACKS.keys())
gh, gw = GRID_SHAPE

cached = []
Wtrue_list = []

with torch.no_grad():
    for j, (C, Wup, Wtrue, bits) in enumerate(test_loader):
        if j >= eval_n:
            break
        C = C.to(DEVICE); Wup = Wup.to(DEVICE)
        r = G(C, Wup)
        Y = torch.clamp(C + r, 0.0, 1.0)

        C_np = torch_to_np(C[0])
        Y_np = torch_to_np(Y[0])
        Wtrue_np = Wtrue[0].numpy().astype(np.float32)

        cached.append((C_np, Y_np, Wtrue_np))
        Wtrue_list.append(Wtrue_np)

Wtrue_wrong = Wtrue_list[1:] + Wtrue_list[:1]

psnr_list, ssim_list, lpips_list = [], [], []
nc_acc = {a: [] for a in attacks}
ber_acc = {a: [] for a in attacks}
bitacc_acc = {a: [] for a in attacks}

det_pos = {a: [] for a in attacks}
det_neg = {a: [] for a in attacks}

ncL_acc = {a: [] for a in attacks}
berL_acc = {a: [] for a in attacks}
bitaccL_acc = {a: [] for a in attacks}

detL_pos = {a: [] for a in attacks}
detL_neg = {a: [] for a in attacks}

# === REPLACE ONLY THIS FUNCTION IN CELL 5 ===
def _decode_metrics(Ya_np, C_np, Wtrue_np, Wwrong_np):
    # numpy -> torch (1,3,H,W)
    Ya_t = np_to_torch(Ya_np).unsqueeze(0).to(DEVICE)
    C_t  = np_to_torch(C_np).unsqueeze(0).to(DEVICE)

    # Build residual proxy exactly aligned with training
    proxy = torch.clamp((Ya_t - C_t) / (CGAN_EPS + 1e-8), -1.0, 1.0)
    proxy01 = (proxy + 1.0) / 2.0
    proxy1  = torch.mean(proxy01, dim=1, keepdim=True)  # (1,1,H,W)

    # Decode with trained Dec
    logits = Dec(proxy1)                  # (1,gh,gw)
    probs  = torch.sigmoid(logits)[0].detach().cpu().numpy().astype(np.float32)  # (gh,gw)

    # Binarize recovered grid
    rec_bin   = binarize01(probs, 0.5)
    true_bin  = binarize01(Wtrue_np, 0.5)
    wrong_bin = binarize01(Wwrong_np, 0.5)

    # Metrics (keep same contract as before)
    nc_true = float(nc(rec_bin, true_bin))
    ber_true = float(ber(rec_bin, true_bin))
    bitacc_true = float(100.0 - ber_true)

    nc_wrong = float(nc(rec_bin, wrong_bin))
    return nc_true, ber_true, bitacc_true, nc_wrong

for i, (C_np, Y_np, Wtrue_np) in enumerate(cached):
    psnr_list.append(psnr(Y_np, C_np))
    ssim_list.append(ssim_np(Y_np, C_np))
    # Common Cell 3 provides lpips_score_np; if missing -> NaN
    lpips_list.append(lpips_score_np(Y_np, C_np, device=DEVICE) if (TORCH_OK and LPIPS_OK) else float("nan"))

    for atk_name, atk_fn in ATTACKS.items():
        Ya = atk_fn(Y_np)

        nc_v, ber_v, bitacc_v, nc_wrong_v = _decode_metrics(Ya, C_np, Wtrue_np, Wtrue_wrong[i])
        nc_acc[atk_name].append(nc_v)
        ber_acc[atk_name].append(ber_v)
        bitacc_acc[atk_name].append(bitacc_v)

        det_pos[atk_name].append(nc_v)
        det_neg[atk_name].append(nc_wrong_v)

        if launder_fn is not None and atk_name != "clean":
            YaL = np.clip(launder_fn(Ya), 0, 1)
        else:
            YaL = Ya

        ncL, berL, bitaccL, nc_wrong_L = _decode_metrics(YaL, C_np, Wtrue_np, Wtrue_wrong[i])
        ncL_acc[atk_name].append(ncL)
        berL_acc[atk_name].append(berL)
        bitaccL_acc[atk_name].append(bitaccL)

        detL_pos[atk_name].append(ncL)
        detL_neg[atk_name].append(nc_wrong_L)

infer_seconds = time.time() - t0
infer_co2 = cc_stop(infer_tracker)

# ===================
# Summary
# ===================
summary = {
    "model": "cgan_cond_residual_eps_nonblind_attackaware",
    "payload_bits": int(PAYLOAD_BITS),
    "eps": float(CGAN_EPS),
    "hw": int(CGAN_HW),
    "n_train": int(len(train_ds)),
    "n_test_eval": int(len(cached)),

    "mean_psnr_cover": float(np.mean(psnr_list)),
    "mean_ssim_cover": float(np.mean(ssim_list)),
    "mean_lpips_cover": float(np.nanmean(lpips_list)),

    "mean_nc":      {a: float(np.mean(nc_acc[a])) for a in attacks},
    "mean_ber":     {a: float(np.mean(ber_acc[a])) for a in attacks},
    "mean_bit_acc": {a: float(np.mean(bitacc_acc[a])) for a in attacks},

    "mean_nc_laundered":      {a: float(np.mean(ncL_acc[a])) for a in attacks},
    "mean_ber_laundered":     {a: float(np.mean(berL_acc[a])) for a in attacks},
    "mean_bit_acc_laundered": {a: float(np.mean(bitaccL_acc[a])) for a in attacks},

    "train_seconds": float(train_seconds),
    "infer_seconds": float(infer_seconds),
    "train_co2_kg": float(train_co2) if train_co2 is not None else None,
    "infer_co2_kg": float(infer_co2) if infer_co2 is not None else None,
    "model_file_mb": float(model_file_mb),

    "ai_laundering_meta": launder_meta,

    "train_recipe": {
        "lambda_payload": float(LAMBDA_PAYLOAD),
        "lambda_imp": float(LAMBDA_IMP),
        "lambda_res": float(LAMBDA_RES),
        "lambda_adv": float(LAMBDA_ADV),
        "payload_proxy_mix": "0.5*clean + 0.5*attacked"
    },

    "detector_auc_by_attack": {},
    "detector_confusion_1pctfpr_by_attack": {},
    "detector_auc_by_attack_laundered": {},
    "detector_confusion_1pctfpr_by_attack_laundered": {},
}

# ==================================
# Decision metrics + heatmaps
# ==================================
for atk_name in attacks:
    pos = np.asarray(det_pos[atk_name], dtype=np.float64)
    neg = np.asarray(det_neg[atk_name], dtype=np.float64)

    y_true = np.array([1]*len(pos) + [0]*len(neg))
    y_score = np.array(list(pos) + list(neg))

    A = float(roc_auc_score(y_true, y_score))
    cm = confusion_at_fpr(pos, neg, target_fpr=0.01)

    summary["detector_auc_by_attack"][atk_name] = A
    summary["detector_confusion_1pctfpr_by_attack"][atk_name] = cm

    plot_confusion_heatmap(cm,
        f"cGAN Confusion @1%FPR — {atk_name}",
        os.path.join(OUT_DIR, f"cgan_confusion_standard_{atk_name}.png")
    )

    posL = np.asarray(detL_pos[atk_name], dtype=np.float64)
    negL = np.asarray(detL_neg[atk_name], dtype=np.float64)
    y_trueL = np.array([1]*len(posL) + [0]*len(negL))
    y_scoreL = np.array(list(posL) + list(negL))

    AL = float(roc_auc_score(y_trueL, y_scoreL))
    cmL = confusion_at_fpr(posL, negL, target_fpr=0.01)

    summary["detector_auc_by_attack_laundered"][atk_name] = AL
    summary["detector_confusion_1pctfpr_by_attack_laundered"][atk_name] = cmL

    plot_confusion_heatmap(cmL,
        f"cGAN Confusion @1%FPR (Laundered) — {atk_name}",
        os.path.join(OUT_DIR, f"cgan_confusion_laundered_{atk_name}.png")
    )

# =============================
# ROC plot (Clean) — main
# =============================
pos_clean = np.asarray(det_pos["clean"], dtype=np.float64)
neg_clean = np.asarray(det_neg["clean"], dtype=np.float64)
y_true_clean = np.array([1]*len(pos_clean) + [0]*len(neg_clean))
y_score_clean = np.array(list(pos_clean) + list(neg_clean))

fpr, tpr, _ = roc_curve(y_true_clean, y_score_clean)
auc_clean = float(roc_auc_score(y_true_clean, y_score_clean))

plt.figure()
plt.plot(fpr, tpr, label=f"Clean AUC={auc_clean:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("cGAN Detection ROC (clean): correct vs wrong payload")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "cgan_detection_roc_clean.png"), dpi=300)
plt.close()

# OPTIONAL: ROC overlay (all attacks)
plt.figure(figsize=(7,6))
for atk_name in attacks:
    pos = np.asarray(det_pos[atk_name], dtype=np.float64)
    neg = np.asarray(det_neg[atk_name], dtype=np.float64)
    y_true = np.array([1]*len(pos) + [0]*len(neg))
    y_score = np.array(list(pos) + list(neg))
    fpr, tpr, _ = roc_curve(y_true, y_score)
    A = summary["detector_auc_by_attack"][atk_name]
    plt.plot(fpr, tpr, label=f"{atk_name} (AUC={A:.2f})")
plt.plot([0,1],[0,1], linestyle="--")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title("cGAN Detector ROC — All Attacks (Standard)")
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "cgan_detector_roc_all_attacks.png"), dpi=300)
plt.close()

# ============================================================
# Plots: NC / BER / BitAcc (standard + laundered)
# ============================================================
plot_attack_metric_line(attacks, summary["mean_nc"],
    "cGAN mean NC under attacks (standard)",
    os.path.join(OUT_DIR, "cgan_NC_attacks.png"),
    ylabel="NC", ylim=(-1, 1)
)
plot_attack_metric_line(attacks, summary["mean_ber"],
    "cGAN mean BER (%) under attacks (standard)",
    os.path.join(OUT_DIR, "cgan_BER_attacks.png"),
    ylabel="BER (%)", ylim=(0, 100)
)
plot_attack_metric_line(attacks, summary["mean_bit_acc"],
    "cGAN mean Bit Accuracy (%) under attacks (standard)",
    os.path.join(OUT_DIR, "cgan_bitacc_attacks.png"),
    ylabel="Bit Accuracy (%)", ylim=(0, 105)
)

plot_attack_metric_line(attacks, summary["mean_nc_laundered"],
    "cGAN mean NC under attacks (after laundering)",
    os.path.join(OUT_DIR, "cgan_NC_attacks_laundered.png"),
    ylabel="NC", ylim=(-1, 1)
)
plot_attack_metric_line(attacks, summary["mean_ber_laundered"],
    "cGAN mean BER (%) under attacks (after laundering)",
    os.path.join(OUT_DIR, "cgan_BER_attacks_laundered.png"),
    ylabel="BER (%)", ylim=(0, 100)
)
plot_attack_metric_line(attacks, summary["mean_bit_acc_laundered"],
    "cGAN mean Bit Accuracy (%) under attacks (after laundering)",
    os.path.join(OUT_DIR, "cgan_bitacc_attacks_laundered.png"),
    ylabel="Bit Accuracy (%)", ylim=(0, 105)
)

# Delta BitAcc
bit_std = np.array([summary["mean_bit_acc"][a] for a in attacks], dtype=float)
bit_lau = np.array([summary["mean_bit_acc_laundered"][a] for a in attacks], dtype=float)
delta = bit_std - bit_lau

plt.figure(figsize=(12,4))
plt.bar(attacks, delta)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Bit-Accuracy Drop (pp)")
plt.title("AI Laundering Impact: Δ(BitAcc) = Standard − Laundered (cGAN)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "cgan_delta_bitacc_standard_minus_laundered.png"), dpi=240)
plt.close()

# PSNR histogram
plt.figure(figsize=(6,4))
plt.hist(psnr_list, bins=30)
plt.xlabel("PSNR vs cover (dB)")
plt.ylabel("count")
plt.title("cGAN cover-referenced imperceptibility (PSNR)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "cgan_imperceptibility_psnr_hist.png"), dpi=240)
plt.close()

# ============================================================
# Export CSV tables (paper-friendly)
# ============================================================
try:
    import pandas as pd

    rob_rows = []
    for a in attacks:
        rob_rows.append({
            "attack": a,
            "NC_standard": summary["mean_nc"][a],
            "BER_standard": summary["mean_ber"][a],
            "BitAcc_standard": summary["mean_bit_acc"][a],
            "NC_laundered": summary["mean_nc_laundered"][a],
            "BER_laundered": summary["mean_ber_laundered"][a],
            "BitAcc_laundered": summary["mean_bit_acc_laundered"][a],
        })
    pd.DataFrame(rob_rows).to_csv(os.path.join(OUT_DIR, "cgan_robustness_table.csv"), index=False)

    det_rows = []
    for a in attacks:
        cm  = summary["detector_confusion_1pctfpr_by_attack"][a]
        cmL = summary["detector_confusion_1pctfpr_by_attack_laundered"][a]
        det_rows.append({
            "attack": a,
            "AUC_standard": summary["detector_auc_by_attack"][a],
            "thr_standard": cm["thr"],
            "TPR@1%FPR_standard": cm["tpr_actual"],
            "FPR@1%FPR_standard": cm["fpr_actual"],
            "AUC_laundered": summary["detector_auc_by_attack_laundered"][a],
            "thr_laundered": cmL["thr"],
            "TPR@1%FPR_laundered": cmL["tpr_actual"],
            "FPR@1%FPR_laundered": cmL["fpr_actual"],
        })
    pd.DataFrame(det_rows).to_csv(os.path.join(OUT_DIR, "cgan_detector_table.csv"), index=False)

except Exception as e:
    print("CSV export skipped (pandas missing?):", repr(e))

# ============================================================
# Save final summary JSON
# ============================================================
save_json(summary, os.path.join(OUT_DIR, "cgan_summary.json"))
print(json.dumps(summary, indent=2))
print("✅ Outputs in:", OUT_DIR)

Cell5 hyperparams: BATCH= 16 EPOCHS= 6 LR_G= 0.0002 LR_DEC= 0.0002 CGAN_HW= 128 CGAN_EPS= 0.02
Using n_train = 5000 | n_test = 1000 | eval = 1000


[codecarbon WARNING @ 17:14:18] Multiple instances of codecarbon are allowed to run at the same time.


TRAIN weights: payload= 8.0 imp= 0.6 res= 0.25 adv= 0.0
Epoch 1/6 | loss=0.0534 (imp=0.0039, res=0.2038, pay=0.0000, atk=clean)
Epoch 2/6 | loss=0.0325 (imp=0.0024, res=0.1241, pay=0.0000, atk=clean)
Epoch 3/6 | loss=0.0308 (imp=0.0023, res=0.1174, pay=0.0000, atk=noise)
Epoch 4/6 | loss=0.0421 (imp=0.0020, res=0.1034, pay=0.0019, atk=blur)
Epoch 5/6 | loss=0.0240 (imp=0.0018, res=0.0913, pay=0.0000, atk=gamma12)
Epoch 6/6 | loss=0.0357 (imp=0.0017, res=0.0853, pay=0.0017, atk=sharpen)
Saved: /kaggle/working/flexmark_out_bits256/cgan_G.pt /kaggle/working/flexmark_out_bits256/cgan_Dec.pt | total 8.79 MB
AI laundering meta: {'enabled': True, 'device': 'cuda', 'patch': 64, 'patches_per_img': 2, 'epochs': 1, 'max_pairs': 50, 'train_seconds': 0.19165277481079102, 'model_file': '/kaggle/working/flexmark_out_bits256/ai_launder_tinydenoiser.pt', 'model_file_mb': 0.04524707794189453}


/tmp/ipykernel_55/345400547.py:455: RuntimeWarning: Mean of empty slice
  "mean_lpips_cover": float(np.nanmean(lpips_list)),


{
  "model": "cgan_cond_residual_eps_nonblind_attackaware",
  "payload_bits": 256,
  "eps": 0.02,
  "hw": 128,
  "n_train": 5000,
  "n_test_eval": 1000,
  "mean_psnr_cover": 46.93630845161286,
  "mean_ssim_cover": 0.9948052486181259,
  "mean_lpips_cover": NaN,
  "mean_nc": {
    "clean": 1.0000000295639038,
    "jpeg10": 0.4371764914244413,
    "jpeg30": 0.4999865144789219,
    "jpeg50": 0.5473964271396399,
    "jpeg70": 0.6348026228249073,
    "jpeg90": 0.9050220763683319,
    "resize075": 0.6570694815516472,
    "crop10": 0.39794857838749886,
    "blur": 0.521480431497097,
    "sharpen": 0.9865531013011932,
    "gamma12": 0.550108536966145,
    "screenshot": 0.4971001496016979,
    "launder_only": 1.0000000295639038
  },
  "mean_ber": {
    "clean": 0.0,
    "jpeg10": 47.76328125,
    "jpeg30": 43.003125,
    "jpeg50": 39.25703125,
    "jpeg70": 32.390234375,
    "jpeg90": 9.021484375,
    "resize075": 31.238671875,
    "crop10": 49.707421875,
    "blur": 42.20703125,
    "sharpen": 

### 128 bits HiDDEN

In [8]:
# ============================================================
# CELL 4 PATCH — HiDDeN-N-DB (Non-blind) GRID decoder
# Output is (B, gh, gw) logits instead of (B, payload_bits)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, s, p),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class HiddenEncoder(nn.Module):
    """
    Input: concat([cover (3ch), payload_map (payload_ch)]) -> residual r (3ch)
    Residual bounded by eps via tanh.
    """
    def __init__(self, payload_ch=1, eps=0.02, width=64):
        super().__init__()
        self.eps = float(eps)
        in_ch = 3 + payload_ch
        self.net = nn.Sequential(
            ConvBNReLU(in_ch, width),
            ConvBNReLU(width, width),
            ConvBNReLU(width, width),
            nn.Conv2d(width, 3, 1, 1, 0),
        )

    def forward(self, C, P):
        x = torch.cat([C, P], dim=1)
        r = torch.tanh(self.net(x)) * self.eps
        Y = torch.clamp(C + r, 0.0, 1.0)
        return Y, r

class HiddenDecoderNonBlindGrid(nn.Module):
    """
    Non-blind GRID decoder:
    input: concat([suspect (3ch), cover (3ch)]) => 6ch
    output: logits_grid (B, gh, gw) where gh*gw = payload_bits
    """
    def __init__(self, gh=8, gw=16, width=64):
        super().__init__()
        self.gh, self.gw = int(gh), int(gw)
        self.f = nn.Sequential(
            ConvBNReLU(6, width, s=1),
            ConvBNReLU(width, width, s=2),
            ConvBNReLU(width, width, s=2),
            ConvBNReLU(width, width, s=2),
            nn.Conv2d(width, 1, 1, 1, 0),
        )

    def forward(self, Y_noised, C):
        x = torch.cat([Y_noised, C], dim=1)
        z = self.f(x)  # (B,1,h,w)
        # For bit-grids, nearest is often a better inductive bias than bilinear.
        z = F.interpolate(z, size=(self.gh, self.gw), mode="nearest")
        return z.squeeze(1)  # (B,gh,gw)

class PresenceDetector(nn.Module):
    """
    Watermarked-vs-clean detector (adversary).
    Outputs logits: (B, 1)
    """
    def __init__(self, width=64):
        super().__init__()
        self.net = nn.Sequential(
            ConvBNReLU(3, width, s=2),
            ConvBNReLU(width, width, s=2),
            ConvBNReLU(width, width, s=2),
            nn.AdaptiveAvgPool2d((1,1)),
        )
        self.head = nn.Linear(width, 1)

    def forward(self, img):
        f = self.net(img).flatten(1)  # (B,width)
        return self.head(f)

print("HiDDeN-N-DB grid-decoder model classes ready.")

HiDDeN-N-DB grid-decoder model classes ready.


In [9]:
# ============================================================
# CELL 5 — HiDDeN-N-DB: Curriculum Train + Eval + Presence metrics + Separate laundering
# FINAL (matches LinearSVR / cGAN reporting):
#  - Robustness: NC_bits / BER / bit_acc per attack (standard + laundered)
#  - Presence: ROC/AUC + confusion@1%FPR (standard + laundered)
#  - Plots: ROC(clean), ROC(per attack), confusion heatmaps, attack curves, delta plots
#
# IMPORTANT:
#  (1) DATASET-ONLY PAYLOAD: bits derived from dataset mark PNG via make_payload_for_model(...)
#  (2) PAIRED GEOMETRIC ATTACKS in eval (crop/resize/screenshot) -> SAME params for (Y,C)
#  (3) TRAIN: when crop/resize channel chosen, apply SAME transform to (Y,C) by sampling coords once
# ============================================================

import os, time, random, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from io import BytesIO

import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------
# Utility: save json
# -----------------------
def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

# -----------------------
# Utility: confusion @ target FPR (with actual rates)
# -----------------------
def confusion_at_fpr(scores_pos, scores_neg, target_fpr=0.01):
    scores_pos = np.asarray(scores_pos, dtype=np.float64)
    scores_neg = np.asarray(scores_neg, dtype=np.float64)

    thr = float(np.quantile(scores_neg, 1.0 - target_fpr))

    TP = int(np.sum(scores_pos >= thr))
    FN = int(np.sum(scores_pos <  thr))
    FP = int(np.sum(scores_neg >= thr))
    TN = int(np.sum(scores_neg <  thr))

    fpr_actual = FP / max(1, (FP + TN))
    tpr_actual = TP / max(1, (TP + FN))

    return {
        "thr": thr,
        "TP": TP, "FP": FP, "TN": TN, "FN": FN,
        "fpr_actual": float(fpr_actual),
        "tpr_actual": float(tpr_actual),
    }

def plot_confusion_heatmap(cm_dict, title, out_path):
    mat = np.array([[cm_dict["TN"], cm_dict["FP"]],
                    [cm_dict["FN"], cm_dict["TP"]]], dtype=np.int64)
    plt.figure(figsize=(4.6, 4.1))
    plt.imshow(mat)
    plt.xticks([0,1], ["Pred 0", "Pred 1"])
    plt.yticks([0,1], ["True 0", "True 1"])
    for (r,c), v in np.ndenumerate(mat):
        plt.text(c, r, str(v), ha="center", va="center", fontsize=11)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=240)
    plt.close()

def plot_attack_metric_line(attacks, series_dict, title, out_path, ylabel=None, ylim=None):
    x = np.arange(len(attacks))
    y = [series_dict[a] for a in attacks]
    plt.figure(figsize=(12,4))
    plt.plot(x, y, marker="o")
    plt.xticks(x, attacks, rotation=45, ha="right")
    if ylabel is not None:
        plt.ylabel(ylabel)
    if ylim is not None:
        plt.ylim(*ylim)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=240)
    plt.close()

# -----------------------
# Real JPEG forward helper (per-batch, per-image)
# -----------------------
def _torch_bchw_to_np_bhwc01(x):
    return x.detach().clamp(0,1).cpu().permute(0,2,3,1).numpy()

def _np_bhwc01_to_torch_bchw(x_np, device):
    return torch.from_numpy(x_np).float().permute(0,3,1,2).to(device).clamp(0,1)

def jpeg_batch_torch(Y, quality=50, device=None):
    if device is None:
        device = Y.device
    Y_np = _torch_bchw_to_np_bhwc01(Y)
    out = []
    for i in range(Y_np.shape[0]):
        pil = Image.fromarray((Y_np[i] * 255).astype(np.uint8))
        buf = BytesIO()
        pil.save(buf, format="JPEG", quality=int(quality))
        buf.seek(0)
        rec = Image.open(buf).convert("RGB")
        out.append(np.asarray(rec).astype(np.float32) / 255.0)
    out_np = np.stack(out, axis=0)
    return _np_bhwc01_to_torch_bchw(out_np, device)

# -----------------------
# Config
# -----------------------
HW = int(HW)
EPS = float(EPS)
PAYLOAD_BITS = int(PAYLOAD_BITS)
assert PAYLOAD_BITS in (128, 256)
gh, gw = (8,16) if PAYLOAD_BITS == 128 else (16,16)

BATCH = 16
LR_E = 2e-4
LR_ADV = 2e-4

# Loss weights
LAMBDA_MSG = 10.0
LAMBDA_IMP = 0.10
LAMBDA_RES = 0.05
LAMBDA_ADV = 0.0  # keep 0; detector still trained so AUC meaningful

# Curriculum schedule
EPOCHS_WARMUP = 2
EPOCHS_ROBUST = 6

# JPEG realism
JPEG_TRAIN_P = 0.50
JPEG_QUALS   = [10, 30, 50]

# Crop curriculum
CROP_P_START = 0.05
CROP_P_END   = 0.25

print("HiDDeN-N-DB curriculum:",
      "| warmup:", EPOCHS_WARMUP,
      "| robust:", EPOCHS_ROBUST,
      "| L_msg:", LAMBDA_MSG,
      "| L_imp:", LAMBDA_IMP,
      "| L_res:", LAMBDA_RES,
      "| L_adv:", LAMBDA_ADV,
      "| JPEG_P:", JPEG_TRAIN_P,
      "| JPEG_QUALS:", JPEG_QUALS,
      "| crop_p:", (CROP_P_START, "->", CROP_P_END),
      "| grid:", (gh,gw),
      "| DEVICE:", DEVICE)

# ============================================================
# DATASET (DATASET-ONLY payload from mark PNG)
# Requires Common Cell 2:
#   make_payload_for_model(mark_path, hw, bitsB, mode)
# where it returns: (W_payload_hw, W_true_grid, meta)
# ============================================================
class CoverMarkHiddenDataset(Dataset):
    def __init__(self, cover_paths, mark_paths, hw, payload_bits):
        assert len(cover_paths) == len(mark_paths), "cover/mark list mismatch"
        self.cover_paths = cover_paths
        self.mark_paths  = mark_paths
        self.hw = int(hw)
        self.payload_bits = int(payload_bits)

    def __len__(self):
        return len(self.cover_paths)

    def __getitem__(self, idx):
        C = read_rgb(self.cover_paths[idx], self.hw)

        # Dataset-only: derive bits from mark PNG deterministically
        # mode="1ch" -> payload map P is (H,W,1)
        P_np, Wtrue_np, meta = make_payload_for_model(
            self.mark_paths[idx], self.hw, self.payload_bits, mode="1ch"
        )

        # Wtrue_np should be (gh,gw,1) in [0,1] -> bits (0/1)
        if Wtrue_np.ndim == 3:
            bits_np = (Wtrue_np[..., 0] >= 0.5).astype(np.int32).reshape(-1)
        else:
            # if build_payload_from_mark returns (gh,gw) already
            bits_np = (Wtrue_np >= 0.5).astype(np.int32).reshape(-1)

        C_t = torch.from_numpy(C).permute(2,0,1).float()      # (3,H,W)
        P_t = torch.from_numpy(P_np).permute(2,0,1).float()   # (1,H,W)
        bits_t = torch.from_numpy(bits_np.astype(np.float32)) # (B,)
        return C_t, P_t, bits_t

train_ds = CoverMarkHiddenDataset(trC[:n_train], trW[:n_train], HW, PAYLOAD_BITS)
test_ds  = CoverMarkHiddenDataset(teC[:n_test],  teW[:n_test],  HW, PAYLOAD_BITS)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False, num_workers=2)

print("Using n_train =", len(train_ds), "| n_test =", len(test_ds), "| eval =", min(EVAL_N, n_test))

# ======================
# MODELS (from Cell 4)
# =====================
Enc = HiddenEncoder(payload_ch=1, eps=EPS, width=64).to(DEVICE)
Dec = HiddenDecoderNonBlindGrid(gh=gh, gw=gw, width=64).to(DEVICE)
Adv = PresenceDetector(width=64).to(DEVICE)

optEncDec = torch.optim.Adam(list(Enc.parameters()) + list(Dec.parameters()), lr=LR_E, betas=(0.5,0.999))
optAdv    = torch.optim.Adam(Adv.parameters(), lr=LR_ADV, betas=(0.5,0.999))

bce_logits = nn.BCEWithLogitsLoss()
l1 = nn.L1Loss()

# ============================================================
# TRAIN-TIME differentiable-ish attacks
# IMPORTANT: for crop/resize, apply SAME sampled transform to both (Y,C)
# ============================================================
def t_resize_pair(Y, C, scale=0.75):
    B, _, H, W = Y.shape
    nh, nw = max(4, int(H*scale)), max(4, int(W*scale))
    Y1 = F.interpolate(Y, size=(nh,nw), mode="bilinear", align_corners=False)
    C1 = F.interpolate(C, size=(nh,nw), mode="bilinear", align_corners=False)
    Y2 = F.interpolate(Y1, size=(H,W), mode="bilinear", align_corners=False)
    C2 = F.interpolate(C1, size=(H,W), mode="bilinear", align_corners=False)
    return Y2, C2

def t_crop_resize_pair(Y, C, crop_frac=0.10):
    B, _, H, W = Y.shape
    ch, cw = max(4, int(H*(1-crop_frac))), max(4, int(W*(1-crop_frac)))
    y0 = torch.randint(0, H - ch + 1, (1,), device=Y.device).item()
    x0 = torch.randint(0, W - cw + 1, (1,), device=Y.device).item()
    Yc = Y[:, :, y0:y0+ch, x0:x0+cw]
    Cc = C[:, :, y0:y0+ch, x0:x0+cw]
    Y2 = F.interpolate(Yc, size=(H,W), mode="bilinear", align_corners=False)
    C2 = F.interpolate(Cc, size=(H,W), mode="bilinear", align_corners=False)
    return Y2, C2

def t_blur(x):
    k=3; pad=1
    w = torch.ones((x.shape[1],1,k,k), device=x.device) / (k*k)
    return F.conv2d(x, w, padding=pad, groups=x.shape[1])

def t_sharpen(x):
    blur = t_blur(x)
    alpha = 0.7
    return torch.clamp(x + alpha*(x-blur), 0, 1)

def t_gamma(x, gamma=1.2):
    return torch.clamp(x,0,1) ** gamma

def t_noise(x, sigma=0.01):
    return torch.clamp(x + sigma*torch.randn_like(x), 0, 1)

TRAIN_ATKS_NO_CROP = [
    ("clean",     lambda y,c: (y,c)),
    ("blur",      lambda y,c: (t_blur(y), c)),
    ("sharpen",   lambda y,c: (t_sharpen(y), c)),
    ("gamma12",   lambda y,c: (t_gamma(y, 1.2), c)),
    ("noise",     lambda y,c: (t_noise(y, 0.01), c)),
]

def sample_train_channel(stage, robust_epoch_idx=0):
    if stage == "warmup":
        return ("clean", "clean", None)

    t = 0.0 if EPOCHS_ROBUST <= 1 else (robust_epoch_idx / (EPOCHS_ROBUST - 1))
    crop_p = CROP_P_START + (CROP_P_END - CROP_P_START) * t

    if random.random() < JPEG_TRAIN_P:
        return ("jpeg", "jpeg", random.choice(JPEG_QUALS))

    # crop path
    if random.random() < crop_p:
        return ("crop10", "crop10", None)

    # include paired resize as an option
    if random.random() < 0.25:
        return ("resize075", "resize075", None)

    name, fn = random.choice(TRAIN_ATKS_NO_CROP)
    return (name, name, None)


def train_one_epoch(stage, robust_epoch_idx=0):
    Enc.train(); Dec.train(); Adv.train()
    last = None

    for (C, P, bits) in train_loader:
        C = C.to(DEVICE, non_blocking=True)
        P = P.to(DEVICE, non_blocking=True)
        bits = bits.to(DEVICE, non_blocking=True)

        Y, r = Enc(C, P)

        ch_name, ch_kind, ch_q = sample_train_channel(stage, robust_epoch_idx)

        # Apply channel to Y and C consistently
        if ch_kind == "jpeg":
            Y_noised = jpeg_batch_torch(Y, quality=ch_q, device=DEVICE)
            C_noised = jpeg_batch_torch(C, quality=ch_q, device=DEVICE)
        elif ch_kind == "crop10":
            Y_noised, C_noised = t_crop_resize_pair(Y, C, crop_frac=0.10)
        elif ch_kind == "resize075":
            Y_noised, C_noised = t_resize_pair(Y, C, scale=0.75)
        else:
            # non-geometric: operate on Y only; keep C aligned
            fn = dict(TRAIN_ATKS_NO_CROP).get(ch_kind, None)
            if fn is None:
                Y_noised, C_noised = Y, C
            else:
                Y_noised, C_noised = fn(Y, C)

        bits_grid = bits.view(-1, gh, gw)
        logits_grid = Dec(Y_noised, C_noised)
        loss_msg = bce_logits(logits_grid, bits_grid)

        loss_imp = l1(Y, C)
        loss_res = torch.mean(torch.abs(r)) / (EPS + 1e-8)

        # Presence detector training ALWAYS
        adv_logits_w = Adv(Y_noised.detach())
        adv_logits_c = Adv(C_noised.detach())
        adv_loss = bce_logits(adv_logits_w, torch.ones_like(adv_logits_w)) + \
                   bce_logits(adv_logits_c, torch.zeros_like(adv_logits_c))

        optAdv.zero_grad(set_to_none=True)
        adv_loss.backward()
        torch.nn.utils.clip_grad_norm_(Adv.parameters(), 5.0)
        optAdv.step()

        # Encoder fooling (optional)
        loss_fool = torch.tensor(0.0, device=DEVICE)
        if LAMBDA_ADV > 0:
            adv_logits_enc = Adv(Y_noised)
            loss_fool = bce_logits(adv_logits_enc, torch.zeros_like(adv_logits_enc))

        loss = (LAMBDA_MSG * loss_msg +
                LAMBDA_IMP * loss_imp +
                LAMBDA_RES * loss_res +
                LAMBDA_ADV * loss_fool)

        optEncDec.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(Enc.parameters()) + list(Dec.parameters()), 5.0)
        optEncDec.step()

        last = (loss.item(), loss_msg.item(), loss_imp.item(), loss_res.item(),
                loss_fool.item(), adv_loss.item(), ch_name)

    return last

# ============
# TRAIN
# ============
ensure_dir(OUT_DIR)
train_tracker = cc_start("train_hidden_ndb_curriculum", OUT_DIR)
t0 = time.time()

for ep in range(EPOCHS_WARMUP):
    last = train_one_epoch("warmup", 0)
    print(f"[Warmup] {ep+1}/{EPOCHS_WARMUP} | "
          f"loss={last[0]:.4f} (msg={last[1]:.4f}, imp={last[2]:.4f}, res={last[3]:.4f}, "
          f"fool={last[4]:.4f}, adv={last[5]:.4f}, ch={last[6]})")

for ep in range(EPOCHS_ROBUST):
    last = train_one_epoch("robust", ep)
    print(f"[Robust] {ep+1}/{EPOCHS_ROBUST} | "
          f"loss={last[0]:.4f} (msg={last[1]:.4f}, imp={last[2]:.4f}, res={last[3]:.4f}, "
          f"fool={last[4]:.4f}, adv={last[5]:.4f}, ch={last[6]})")

train_seconds = time.time() - t0
train_co2 = cc_stop(train_tracker)

# Save
Enc_path = os.path.join(OUT_DIR, "hidden_ndb_Enc.pt")
Dec_path = os.path.join(OUT_DIR, "hidden_ndb_DecGrid.pt")
Adv_path = os.path.join(OUT_DIR, "hidden_ndb_Adv.pt")

torch.save(Enc.state_dict(), Enc_path)
torch.save(Dec.state_dict(), Dec_path)
torch.save(Adv.state_dict(), Adv_path)

model_file_mb = (os.path.getsize(Enc_path)+os.path.getsize(Dec_path)+os.path.getsize(Adv_path)) / (1024**2)
print("Saved:", Enc_path, Dec_path, Adv_path, f"| total {model_file_mb:.2f} MB")

# ============================================================
# EVAL (STANDARD): Robustness + Presence (ROC/AUC + Confusion)
# ============================================================
infer_tracker = cc_start("infer_hidden_ndb", OUT_DIR)
t0 = time.time()

Enc.eval(); Dec.eval(); Adv.eval()

cached = []
with torch.no_grad():
    for j, (C, P, bits) in enumerate(test_loader):
        if j >= min(EVAL_N, n_test):
            break
        C = C.to(DEVICE); P = P.to(DEVICE)
        Y, _ = Enc(C, P)
        cached.append((
            C[0].detach().cpu().permute(1,2,0).numpy(),
            Y[0].detach().cpu().permute(1,2,0).numpy(),
            bits[0].numpy().astype(np.int32)
        ))

attacks = list(ATTACKS.keys())

# ---- Paired geometric attacks for eval (same params for Y and C) ----
def paired_resize_np(Y, C, scale=0.75):
    H, W, _ = Y.shape
    nh, nw = max(1, int(H*scale)), max(1, int(W*scale))
    Yp = np_to_pil(Y).resize((nw, nh), Image.BICUBIC).resize((W, H), Image.BICUBIC)
    Cp = np_to_pil(C).resize((nw, nh), Image.BICUBIC).resize((W, H), Image.BICUBIC)
    return pil_to_np(Yp), pil_to_np(Cp)

def paired_crop_np(Y, C, crop_frac=0.10):
    H, W, _ = Y.shape
    ch, cw = max(1, int(H*(1-crop_frac))), max(1, int(W*(1-crop_frac)))
    y0 = np.random.randint(0, H - ch + 1)
    x0 = np.random.randint(0, W - cw + 1)
    Yc = Y[y0:y0+ch, x0:x0+cw, :]
    Cc = C[y0:y0+ch, x0:x0+cw, :]
    Yp = np_to_pil(Yc).resize((W, H), Image.BICUBIC)
    Cp = np_to_pil(Cc).resize((W, H), Image.BICUBIC)
    return pil_to_np(Yp), pil_to_np(Cp)

def paired_screenshot_np(Y, C, q1=35, q2=80):
    Y1 = attack_jpeg(Y, q1); C1 = attack_jpeg(C, q1)
    Y2, C2 = paired_resize_np(Y1, C1, 0.92)
    Y3 = attack_jpeg(Y2, q2); C3 = attack_jpeg(C2, q2)
    return Y3, C3

# Robustness rows
rows = []
# Presence score banks (STANDARD)
pres_pos = {a: [] for a in attacks}
pres_neg = {a: [] for a in attacks}

for (C_np, Y_np, bits_np) in cached:
    psnr_c  = psnr(Y_np, C_np)
    ssim_c  = ssim_np(Y_np, C_np)
    lpips_c = lpips_score_np(Y_np, C_np, device=DEVICE) if (TORCH_OK and LPIPS_OK) else float("nan")

    rob = {}
    for atk_name, atk_fn in ATTACKS.items():

        # Decide Ya, Ca correctly (paired for geometric)
        if atk_name == "crop10":
            Ya, Ca = paired_crop_np(Y_np, C_np, 0.10)
        elif atk_name == "resize075":
            Ya, Ca = paired_resize_np(Y_np, C_np, 0.75)
        elif atk_name == "screenshot":
            Ya, Ca = paired_screenshot_np(Y_np, C_np, 35, 80)
        elif atk_name in ("jpeg10","jpeg30","jpeg50","jpeg70","jpeg90"):
            Ya = atk_fn(Y_np)
            Ca = atk_fn(C_np)
        else:
            Ya = atk_fn(Y_np)
            Ca = C_np

        Ya_t = torch.from_numpy(Ya).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
        Ca_t = torch.from_numpy(Ca).permute(2,0,1).unsqueeze(0).float().to(DEVICE)

        with torch.no_grad():
            logits_grid = Dec(Ya_t, Ca_t)
            probs_grid  = torch.sigmoid(logits_grid)[0].detach().cpu().numpy()

        pred_bits = (probs_grid.reshape(-1) >= 0.5).astype(np.int32)
        ber_bits  = float(np.mean(pred_bits != bits_np) * 100.0)
        bit_acc   = float(100.0 - ber_bits)

        tb = (bits_np * 2 - 1).astype(np.float32)
        pb = (pred_bits * 2 - 1).astype(np.float32)
        nc_bits = float(np.mean(tb * pb))

        rob[atk_name] = {"NC_bits": nc_bits, "BER": ber_bits, "bit_acc": bit_acc}

        # Presence scores (STANDARD)
        with torch.no_grad():
            s_pos = float(torch.sigmoid(Adv(Ya_t)).item())
            s_neg = float(torch.sigmoid(Adv(Ca_t)).item())
        pres_pos[atk_name].append(s_pos)
        pres_neg[atk_name].append(s_neg)

    rows.append({"psnr_cover": psnr_c, "ssim_cover": ssim_c, "lpips_cover": lpips_c, "robustness": rob})

# Presence metrics + ROC plots + confusion heatmaps (STANDARD)
presence_auc = {}
presence_conf = {}
for atk_name in attacks:
    pos_scores = pres_pos[atk_name]
    neg_scores = pres_neg[atk_name]

    y_true = np.array([1]*len(pos_scores) + [0]*len(neg_scores))
    y_score = np.array(pos_scores + neg_scores)

    A = float(roc_auc_score(y_true, y_score))
    cm = confusion_at_fpr(pos_scores, neg_scores, target_fpr=0.01)

    presence_auc[atk_name] = A
    presence_conf[atk_name] = cm

    fpr, tpr, _ = roc_curve(y_true, y_score)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={A:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title(f"HiDDeN Presence ROC (STANDARD) — {atk_name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"hidden_presence_roc_standard_{atk_name}.png"), dpi=220)
    plt.close()

    plot_confusion_heatmap(
        cm,
        f"Presence Confusion @1%FPR (STANDARD) — {atk_name}",
        os.path.join(OUT_DIR, f"hidden_presence_confusion_standard_{atk_name}.png")
    )

# Clean ROC “main” plot
if "clean" in attacks:
    pos = pres_pos["clean"]
    neg = pres_neg["clean"]
    y_true = np.array([1]*len(pos) + [0]*len(neg))
    y_score = np.array(pos + neg)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    A = float(roc_auc_score(y_true, y_score))

    plt.figure()
    plt.plot(fpr, tpr, label=f"Clean AUC={A:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title("HiDDeN Presence ROC (clean): watermarked vs cover")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "hidden_presence_roc_clean.png"), dpi=260)
    plt.close()

infer_seconds = time.time() - t0
infer_co2 = cc_stop(infer_tracker)

# ============================================================
# AI laundering (SEPARATE) + Presence after laundering + plots
# NOTE: laundering acts on suspect Ya only (attacker removes watermark),
#       cover Ca remains what verifier uses (attacked-cover for geometric).
# ============================================================
pairs = []
for (C_np, Y_np, _) in cached[:50]:
    y1 = attack_jpeg(Y_np, 10)
    y2 = attack_resize(Y_np, 0.75)
    y3 = attack_screenshot_recompress(Y_np, 35, 80)
    pairs.append((y1, C_np))
    pairs.append((y2, C_np))
    pairs.append((y3, C_np))

launder_fn, launder_meta = train_ai_launderer_patchwise(
    pairs=pairs, out_dir=OUT_DIR, patch=64, patches_per_img=2, epochs=1, lr=1e-3, max_pairs=150, device=DEVICE
)

rows_launder = []
presence_auc_L, presence_conf_L = {}, {}
presL_pos = {a: [] for a in attacks}
presL_neg = {a: [] for a in attacks}

if launder_fn is not None:
    for (C_np, Y_np, bits_np) in cached:
        robL = {}
        for atk_name, atk_fn in ATTACKS.items():

            # Build attacked Ya, Ca (paired for geometric)
            if atk_name == "crop10":
                Ya, Ca = paired_crop_np(Y_np, C_np, 0.10)
            elif atk_name == "resize075":
                Ya, Ca = paired_resize_np(Y_np, C_np, 0.75)
            elif atk_name == "screenshot":
                Ya, Ca = paired_screenshot_np(Y_np, C_np, 35, 80)
            elif atk_name in ("jpeg10","jpeg30","jpeg50","jpeg70","jpeg90"):
                Ya = atk_fn(Y_np)
                Ca = atk_fn(C_np)
            else:
                Ya = atk_fn(Y_np)
                Ca = C_np

            # Apply laundering (skip clean by default)
            if atk_name != "clean":
                Ya = np.clip(launder_fn(Ya), 0, 1)

            Ya_t = torch.from_numpy(Ya).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
            Ca_t = torch.from_numpy(Ca).permute(2,0,1).unsqueeze(0).float().to(DEVICE)

            with torch.no_grad():
                logits_grid = Dec(Ya_t, Ca_t)
                probs_grid  = torch.sigmoid(logits_grid)[0].detach().cpu().numpy()

            pred_bits = (probs_grid.reshape(-1) >= 0.5).astype(np.int32)
            ber_bits  = float(np.mean(pred_bits != bits_np) * 100.0)
            bit_acc   = float(100.0 - ber_bits)

            tb = (bits_np * 2 - 1).astype(np.float32)
            pb = (pred_bits * 2 - 1).astype(np.float32)
            nc_bits = float(np.mean(tb * pb))

            robL[atk_name] = {"NC_bits": nc_bits, "BER": ber_bits, "bit_acc": bit_acc}

            # Presence after laundering
            with torch.no_grad():
                s_pos = float(torch.sigmoid(Adv(Ya_t)).item())
                s_neg = float(torch.sigmoid(Adv(Ca_t)).item())
            presL_pos[atk_name].append(s_pos)
            presL_neg[atk_name].append(s_neg)

        rows_launder.append({"robustness": robL})

    # Presence metrics + ROC plots + confusion heatmaps (LAUNDERED)
    for atk_name in attacks:
        pos_scores = presL_pos[atk_name]
        neg_scores = presL_neg[atk_name]
        y_true = np.array([1]*len(pos_scores) + [0]*len(neg_scores))
        y_score = np.array(pos_scores + neg_scores)

        A = float(roc_auc_score(y_true, y_score))
        cm = confusion_at_fpr(pos_scores, neg_scores, target_fpr=0.01)

        presence_auc_L[atk_name] = A
        presence_conf_L[atk_name] = cm

        fpr, tpr, _ = roc_curve(y_true, y_score)
        plt.figure()
        plt.plot(fpr, tpr, label=f"AUC={A:.3f}")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("FPR"); plt.ylabel("TPR")
        plt.title(f"HiDDeN Presence ROC (LAUNDERED) — {atk_name}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f"hidden_presence_roc_laundered_{atk_name}.png"), dpi=220)
        plt.close()

        plot_confusion_heatmap(
            cm,
            f"Presence Confusion @1%FPR (LAUNDERED) — {atk_name}",
            os.path.join(OUT_DIR, f"hidden_presence_confusion_laundered_{atk_name}.png")
        )

# ============================================================
# SUMMARY (STANDARD + LAUNDERED)
# ============================================================
summary = {
    "model": "hidden_ndb_nonblind_attackaware_griddecoder_curriculum_v2_datasetpayload_pairedgeom",
    "payload_bits": PAYLOAD_BITS,
    "hw": HW,
    "eps": EPS,
    "grid": [gh, gw],
    "n_train": len(train_ds),
    "n_test_eval": len(rows),

    "mean_psnr_cover": float(np.mean([r["psnr_cover"] for r in rows])),
    "mean_ssim_cover": float(np.mean([r["ssim_cover"] for r in rows])),
    "mean_lpips_cover": float(np.nanmean([r["lpips_cover"] for r in rows])),

    "mean_nc_bits": {a: float(np.mean([r["robustness"][a]["NC_bits"] for r in rows])) for a in attacks},
    "mean_ber":     {a: float(np.mean([r["robustness"][a]["BER"] for r in rows])) for a in attacks},
    "mean_bit_acc": {a: float(np.mean([r["robustness"][a]["bit_acc"] for r in rows])) for a in attacks},

    "presence_auc_by_attack": presence_auc,
    "presence_confusion_1pctfpr_by_attack": presence_conf,

    "train_seconds": float(train_seconds),
    "infer_seconds": float(infer_seconds),
    "train_co2_kg": float(train_co2) if train_co2 is not None else None,
    "infer_co2_kg": float(infer_co2) if infer_co2 is not None else None,
    "model_file_mb": float(model_file_mb),

    "ai_laundering_meta": launder_meta,

    "train_recipe": {
        "warmup_epochs": EPOCHS_WARMUP,
        "robust_epochs": EPOCHS_ROBUST,
        "lambda_msg": LAMBDA_MSG,
        "lambda_imp": LAMBDA_IMP,
        "lambda_res": LAMBDA_RES,
        "lambda_adv": LAMBDA_ADV,
        "jpeg_train_p": JPEG_TRAIN_P,
        "jpeg_quals": JPEG_QUALS,
        "crop_p_start": CROP_P_START,
        "crop_p_end": CROP_P_END,
        "decoder": "nonblind_grid(ghxgw)_fullyconv",
        "note": "Main robustness/presence = attacks only; laundering reported separately; paired geom transforms."
    }
}

if len(rows_launder) > 0:
    summary["mean_nc_bits_laundered"] = {a: float(np.mean([r["robustness"][a]["NC_bits"] for r in rows_launder])) for a in attacks}
    summary["mean_ber_laundered"]     = {a: float(np.mean([r["robustness"][a]["BER"] for r in rows_launder])) for a in attacks}
    summary["mean_bit_acc_laundered"] = {a: float(np.mean([r["robustness"][a]["bit_acc"] for r in rows_launder])) for a in attacks}

    summary["presence_auc_by_attack_laundered"] = presence_auc_L
    summary["presence_confusion_1pctfpr_by_attack_laundered"] = presence_conf_L
else:
    summary["mean_nc_bits_laundered"] = None
    summary["mean_ber_laundered"] = None
    summary["mean_bit_acc_laundered"] = None
    summary["presence_auc_by_attack_laundered"] = None
    summary["presence_confusion_1pctfpr_by_attack_laundered"] = None

# ============================================================
# Attack plots (STANDARD + LAUNDERED + delta)
# ============================================================
plot_attack_metric_line(attacks, summary["mean_nc_bits"],
                        "HiDDeN mean NC_bits under attacks (standard)",
                        os.path.join(OUT_DIR, "hidden_NC_attacks.png"),
                        ylabel="NC_bits", ylim=(-1, 1))

plot_attack_metric_line(attacks, summary["mean_ber"],
                        "HiDDeN mean BER (%) under attacks (standard)",
                        os.path.join(OUT_DIR, "hidden_BER_attacks.png"),
                        ylabel="BER (%)", ylim=(0, 100))

plot_attack_metric_line(attacks, summary["mean_bit_acc"],
                        "HiDDeN mean Bit Accuracy (%) under attacks (standard)",
                        os.path.join(OUT_DIR, "hidden_bitacc_attacks.png"),
                        ylabel="Bit Accuracy (%)", ylim=(0, 105))

if summary["mean_bit_acc_laundered"] is not None:
    plot_attack_metric_line(attacks, summary["mean_nc_bits_laundered"],
                            "HiDDeN mean NC_bits under attacks (after laundering)",
                            os.path.join(OUT_DIR, "hidden_NC_attacks_laundered.png"),
                            ylabel="NC_bits", ylim=(-1, 1))

    plot_attack_metric_line(attacks, summary["mean_ber_laundered"],
                            "HiDDeN mean BER (%) under attacks (after laundering)",
                            os.path.join(OUT_DIR, "hidden_BER_attacks_laundered.png"),
                            ylabel="BER (%)", ylim=(0, 100))

    plot_attack_metric_line(attacks, summary["mean_bit_acc_laundered"],
                            "HiDDeN mean Bit Accuracy (%) under attacks (after laundering)",
                            os.path.join(OUT_DIR, "hidden_bitacc_attacks_laundered.png"),
                            ylabel="Bit Accuracy (%)", ylim=(0, 105))

    bit_std = np.array([summary["mean_bit_acc"][a] for a in attacks], dtype=float)
    bit_lau = np.array([summary["mean_bit_acc_laundered"][a] for a in attacks], dtype=float)
    delta = bit_std - bit_lau

    plt.figure(figsize=(12,4))
    plt.bar(attacks, delta)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Bit-Accuracy Drop (pp)")
    plt.title("AI Laundering Impact: Δ(BitAcc) = Standard − Laundered (HiDDeN)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "hidden_delta_bitacc_standard_minus_laundered.png"), dpi=220)
    plt.close()

# ============================================================
# Save summary
# ============================================================
save_json(summary, os.path.join(OUT_DIR, "hidden_ndb_summary.json"))
print(json.dumps(summary, indent=2))
print("✅ Outputs in:", OUT_DIR)

HiDDeN-N-DB curriculum: | warmup: 2 | robust: 6 | L_msg: 10.0 | L_imp: 0.1 | L_res: 0.05 | L_adv: 0.0 | JPEG_P: 0.5 | JPEG_QUALS: [10, 30, 50] | crop_p: (0.05, '->', 0.25) | grid: (8, 16) | DEVICE: cuda
Using n_train = 5000 | n_test = 1000 | eval = 1000


[codecarbon WARNING @ 01:03:27] Multiple instances of codecarbon are allowed to run at the same time.


[Warmup] 1/2 | loss=0.1018 (msg=0.0070, imp=0.0120, res=0.6071, fool=0.0000, adv=1.3526, ch=clean)
[Warmup] 2/2 | loss=0.0380 (msg=0.0019, imp=0.0071, res=0.3628, fool=0.0000, adv=1.1484, ch=clean)
[Robust] 1/6 | loss=2.6283 (msg=0.2596, imp=0.0124, res=0.6256, fool=0.0000, adv=1.3778, ch=jpeg)
[Robust] 2/6 | loss=0.0688 (msg=0.0038, imp=0.0118, res=0.6015, fool=0.0000, adv=1.0936, ch=jpeg)
[Robust] 3/6 | loss=0.0686 (msg=0.0040, imp=0.0108, res=0.5484, fool=0.0000, adv=0.9739, ch=jpeg)
[Robust] 4/6 | loss=0.1121 (msg=0.0084, imp=0.0103, res=0.5328, fool=0.0000, adv=0.5521, ch=blur)
[Robust] 5/6 | loss=5.6047 (msg=0.5576, imp=0.0107, res=0.5454, fool=0.0000, adv=1.0349, ch=crop10)
[Robust] 6/6 | loss=0.1332 (msg=0.0105, imp=0.0106, res=0.5405, fool=0.0000, adv=1.1153, ch=jpeg)
Saved: /kaggle/working/flexmark_out_bits128/hidden_ndb_Enc.pt /kaggle/working/flexmark_out_bits128/hidden_ndb_DecGrid.pt /kaggle/working/flexmark_out_bits128/hidden_ndb_Adv.pt | total 1.05 MB


/tmp/ipykernel_55/824233214.py:661: RuntimeWarning: Mean of empty slice
  "mean_lpips_cover": float(np.nanmean([r["lpips_cover"] for r in rows])),


{
  "model": "hidden_ndb_nonblind_attackaware_griddecoder_curriculum_v2_datasetpayload_pairedgeom",
  "payload_bits": 128,
  "hw": 128,
  "eps": 0.02,
  "grid": [
    8,
    16
  ],
  "n_train": 5000,
  "n_test_eval": 1000,
  "mean_psnr_cover": 37.39373970794593,
  "mean_ssim_cover": 0.9793275873064995,
  "mean_lpips_cover": NaN,
  "mean_nc_bits": {
    "clean": 1.0,
    "jpeg10": 0.7725,
    "jpeg30": 0.996984375,
    "jpeg50": 0.999703125,
    "jpeg70": 0.999953125,
    "jpeg90": 1.0,
    "resize075": 1.0,
    "crop10": 0.41890625,
    "blur": 0.926328125,
    "sharpen": 0.999890625,
    "gamma12": 0.106203125,
    "screenshot": 0.99640625,
    "launder_only": 1.0
  },
  "mean_ber": {
    "clean": 0.0,
    "jpeg10": 11.375,
    "jpeg30": 0.15078125,
    "jpeg50": 0.01484375,
    "jpeg70": 0.00234375,
    "jpeg90": 0.0,
    "resize075": 0.0,
    "crop10": 29.0546875,
    "blur": 3.68359375,
    "sharpen": 0.00546875,
    "gamma12": 44.68984375,
    "screenshot": 0.1796875,
    "launde

### 256 bits HiDDEN

In [6]:
# ============================================================
# CELL 4 PATCH — HiDDeN-N-DB (Non-blind) GRID decoder
# Output is (B, gh, gw) logits instead of (B, payload_bits)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, s, p),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class HiddenEncoder(nn.Module):
    """
    Input: concat([cover (3ch), payload_map (payload_ch)]) -> residual r (3ch)
    Residual bounded by eps via tanh.
    """
    def __init__(self, payload_ch=1, eps=0.02, width=64):
        super().__init__()
        self.eps = float(eps)
        in_ch = 3 + payload_ch
        self.net = nn.Sequential(
            ConvBNReLU(in_ch, width),
            ConvBNReLU(width, width),
            ConvBNReLU(width, width),
            nn.Conv2d(width, 3, 1, 1, 0),
        )

    def forward(self, C, P):
        x = torch.cat([C, P], dim=1)
        r = torch.tanh(self.net(x)) * self.eps
        Y = torch.clamp(C + r, 0.0, 1.0)
        return Y, r

class HiddenDecoderNonBlindGrid(nn.Module):
    """
    Non-blind GRID decoder:
    input: concat([suspect (3ch), cover (3ch)]) => 6ch
    output: logits_grid (B, gh, gw) where gh*gw = payload_bits
    """
    def __init__(self, gh=8, gw=16, width=64):
        super().__init__()
        self.gh, self.gw = int(gh), int(gw)
        self.f = nn.Sequential(
            ConvBNReLU(6, width, s=1),
            ConvBNReLU(width, width, s=2),
            ConvBNReLU(width, width, s=2),
            ConvBNReLU(width, width, s=2),
            nn.Conv2d(width, 1, 1, 1, 0),
        )

    def forward(self, Y_noised, C):
        x = torch.cat([Y_noised, C], dim=1)
        z = self.f(x)  # (B,1,h,w)
        # For bit-grids, nearest is often a better inductive bias than bilinear.
        z = F.interpolate(z, size=(self.gh, self.gw), mode="nearest")
        return z.squeeze(1)  # (B,gh,gw)

class PresenceDetector(nn.Module):
    """
    Watermarked-vs-clean detector (adversary).
    Outputs logits: (B, 1)
    """
    def __init__(self, width=64):
        super().__init__()
        self.net = nn.Sequential(
            ConvBNReLU(3, width, s=2),
            ConvBNReLU(width, width, s=2),
            ConvBNReLU(width, width, s=2),
            nn.AdaptiveAvgPool2d((1,1)),
        )
        self.head = nn.Linear(width, 1)

    def forward(self, img):
        f = self.net(img).flatten(1)  # (B,width)
        return self.head(f)

print("HiDDeN-N-DB grid-decoder model classes ready.")

HiDDeN-N-DB grid-decoder model classes ready.


In [7]:
# ============================================================
# CELL 5 — HiDDeN-N-DB: Curriculum Train + Eval + Presence metrics + Separate laundering
# FINAL (matches LinearSVR / cGAN reporting):
#  - Robustness: NC_bits / BER / bit_acc per attack (standard + laundered)
#  - Presence: ROC/AUC + confusion@1%FPR (standard + laundered)
#  - Plots: ROC(clean), ROC(per attack), confusion heatmaps, attack curves, delta plots
#
# IMPORTANT:
#  (1) DATASET-ONLY PAYLOAD: bits derived from dataset mark PNG via make_payload_for_model(...)
#  (2) PAIRED GEOMETRIC ATTACKS in eval (crop/resize/screenshot) -> SAME params for (Y,C)
#  (3) TRAIN: when crop/resize channel chosen, apply SAME transform to (Y,C) by sampling coords once
# ============================================================

import os, time, random, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from io import BytesIO

import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------
# Utility: save json
# -----------------------
def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

# -----------------------
# Utility: confusion @ target FPR (with actual rates)
# -----------------------
def confusion_at_fpr(scores_pos, scores_neg, target_fpr=0.01):
    scores_pos = np.asarray(scores_pos, dtype=np.float64)
    scores_neg = np.asarray(scores_neg, dtype=np.float64)

    thr = float(np.quantile(scores_neg, 1.0 - target_fpr))

    TP = int(np.sum(scores_pos >= thr))
    FN = int(np.sum(scores_pos <  thr))
    FP = int(np.sum(scores_neg >= thr))
    TN = int(np.sum(scores_neg <  thr))

    fpr_actual = FP / max(1, (FP + TN))
    tpr_actual = TP / max(1, (TP + FN))

    return {
        "thr": thr,
        "TP": TP, "FP": FP, "TN": TN, "FN": FN,
        "fpr_actual": float(fpr_actual),
        "tpr_actual": float(tpr_actual),
    }

def plot_confusion_heatmap(cm_dict, title, out_path):
    mat = np.array([[cm_dict["TN"], cm_dict["FP"]],
                    [cm_dict["FN"], cm_dict["TP"]]], dtype=np.int64)
    plt.figure(figsize=(4.6, 4.1))
    plt.imshow(mat)
    plt.xticks([0,1], ["Pred 0", "Pred 1"])
    plt.yticks([0,1], ["True 0", "True 1"])
    for (r,c), v in np.ndenumerate(mat):
        plt.text(c, r, str(v), ha="center", va="center", fontsize=11)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=240)
    plt.close()

def plot_attack_metric_line(attacks, series_dict, title, out_path, ylabel=None, ylim=None):
    x = np.arange(len(attacks))
    y = [series_dict[a] for a in attacks]
    plt.figure(figsize=(12,4))
    plt.plot(x, y, marker="o")
    plt.xticks(x, attacks, rotation=45, ha="right")
    if ylabel is not None:
        plt.ylabel(ylabel)
    if ylim is not None:
        plt.ylim(*ylim)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=240)
    plt.close()

# -----------------------
# Real JPEG forward helper (per-batch, per-image)
# -----------------------
def _torch_bchw_to_np_bhwc01(x):
    return x.detach().clamp(0,1).cpu().permute(0,2,3,1).numpy()

def _np_bhwc01_to_torch_bchw(x_np, device):
    return torch.from_numpy(x_np).float().permute(0,3,1,2).to(device).clamp(0,1)

def jpeg_batch_torch(Y, quality=50, device=None):
    if device is None:
        device = Y.device
    Y_np = _torch_bchw_to_np_bhwc01(Y)
    out = []
    for i in range(Y_np.shape[0]):
        pil = Image.fromarray((Y_np[i] * 255).astype(np.uint8))
        buf = BytesIO()
        pil.save(buf, format="JPEG", quality=int(quality))
        buf.seek(0)
        rec = Image.open(buf).convert("RGB")
        out.append(np.asarray(rec).astype(np.float32) / 255.0)
    out_np = np.stack(out, axis=0)
    return _np_bhwc01_to_torch_bchw(out_np, device)

# -----------------------
# Config
# -----------------------
HW = int(HW)
EPS = float(EPS)
PAYLOAD_BITS = int(PAYLOAD_BITS)
assert PAYLOAD_BITS in (128, 256)
gh, gw = (8,16) if PAYLOAD_BITS == 128 else (16,16)

BATCH = 16
LR_E = 2e-4
LR_ADV = 2e-4

# Loss weights
LAMBDA_MSG = 10.0
LAMBDA_IMP = 0.10
LAMBDA_RES = 0.05
LAMBDA_ADV = 0.0  # keep 0; detector still trained so AUC meaningful

# Curriculum schedule
EPOCHS_WARMUP = 2
EPOCHS_ROBUST = 6

# JPEG realism
JPEG_TRAIN_P = 0.50
JPEG_QUALS   = [10, 30, 50]

# Crop curriculum
CROP_P_START = 0.05
CROP_P_END   = 0.25

print("HiDDeN-N-DB curriculum:",
      "| warmup:", EPOCHS_WARMUP,
      "| robust:", EPOCHS_ROBUST,
      "| L_msg:", LAMBDA_MSG,
      "| L_imp:", LAMBDA_IMP,
      "| L_res:", LAMBDA_RES,
      "| L_adv:", LAMBDA_ADV,
      "| JPEG_P:", JPEG_TRAIN_P,
      "| JPEG_QUALS:", JPEG_QUALS,
      "| crop_p:", (CROP_P_START, "->", CROP_P_END),
      "| grid:", (gh,gw),
      "| DEVICE:", DEVICE)

# ============================================================
# DATASET (DATASET-ONLY payload from mark PNG)
# Requires Common Cell 2:
#   make_payload_for_model(mark_path, hw, bitsB, mode)
# where it returns: (W_payload_hw, W_true_grid, meta)
# ============================================================
class CoverMarkHiddenDataset(Dataset):
    def __init__(self, cover_paths, mark_paths, hw, payload_bits):
        assert len(cover_paths) == len(mark_paths), "cover/mark list mismatch"
        self.cover_paths = cover_paths
        self.mark_paths  = mark_paths
        self.hw = int(hw)
        self.payload_bits = int(payload_bits)

    def __len__(self):
        return len(self.cover_paths)

    def __getitem__(self, idx):
        C = read_rgb(self.cover_paths[idx], self.hw)

        # Dataset-only: derive bits from mark PNG deterministically
        # mode="1ch" -> payload map P is (H,W,1)
        P_np, Wtrue_np, meta = make_payload_for_model(
            self.mark_paths[idx], self.hw, self.payload_bits, mode="1ch"
        )

        # Wtrue_np should be (gh,gw,1) in [0,1] -> bits (0/1)
        if Wtrue_np.ndim == 3:
            bits_np = (Wtrue_np[..., 0] >= 0.5).astype(np.int32).reshape(-1)
        else:
            # if build_payload_from_mark returns (gh,gw) already
            bits_np = (Wtrue_np >= 0.5).astype(np.int32).reshape(-1)

        C_t = torch.from_numpy(C).permute(2,0,1).float()      # (3,H,W)
        P_t = torch.from_numpy(P_np).permute(2,0,1).float()   # (1,H,W)
        bits_t = torch.from_numpy(bits_np.astype(np.float32)) # (B,)
        return C_t, P_t, bits_t

train_ds = CoverMarkHiddenDataset(trC[:n_train], trW[:n_train], HW, PAYLOAD_BITS)
test_ds  = CoverMarkHiddenDataset(teC[:n_test],  teW[:n_test],  HW, PAYLOAD_BITS)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False, num_workers=2)

print("Using n_train =", len(train_ds), "| n_test =", len(test_ds), "| eval =", min(EVAL_N, n_test))

# ============================================================
# MODELS (from Cell 4)
# ============================================================
Enc = HiddenEncoder(payload_ch=1, eps=EPS, width=64).to(DEVICE)
Dec = HiddenDecoderNonBlindGrid(gh=gh, gw=gw, width=64).to(DEVICE)
Adv = PresenceDetector(width=64).to(DEVICE)

optEncDec = torch.optim.Adam(list(Enc.parameters()) + list(Dec.parameters()), lr=LR_E, betas=(0.5,0.999))
optAdv    = torch.optim.Adam(Adv.parameters(), lr=LR_ADV, betas=(0.5,0.999))

bce_logits = nn.BCEWithLogitsLoss()
l1 = nn.L1Loss()

# ============================================================
# TRAIN-TIME differentiable-ish attacks
# ============================================================
def t_resize_pair(Y, C, scale=0.75):
    B, _, H, W = Y.shape
    nh, nw = max(4, int(H*scale)), max(4, int(W*scale))
    Y1 = F.interpolate(Y, size=(nh,nw), mode="bilinear", align_corners=False)
    C1 = F.interpolate(C, size=(nh,nw), mode="bilinear", align_corners=False)
    Y2 = F.interpolate(Y1, size=(H,W), mode="bilinear", align_corners=False)
    C2 = F.interpolate(C1, size=(H,W), mode="bilinear", align_corners=False)
    return Y2, C2

def t_crop_resize_pair(Y, C, crop_frac=0.10):
    B, _, H, W = Y.shape
    ch, cw = max(4, int(H*(1-crop_frac))), max(4, int(W*(1-crop_frac)))
    y0 = torch.randint(0, H - ch + 1, (1,), device=Y.device).item()
    x0 = torch.randint(0, W - cw + 1, (1,), device=Y.device).item()
    Yc = Y[:, :, y0:y0+ch, x0:x0+cw]
    Cc = C[:, :, y0:y0+ch, x0:x0+cw]
    Y2 = F.interpolate(Yc, size=(H,W), mode="bilinear", align_corners=False)
    C2 = F.interpolate(Cc, size=(H,W), mode="bilinear", align_corners=False)
    return Y2, C2

def t_blur(x):
    k=3; pad=1
    w = torch.ones((x.shape[1],1,k,k), device=x.device) / (k*k)
    return F.conv2d(x, w, padding=pad, groups=x.shape[1])

def t_sharpen(x):
    blur = t_blur(x)
    alpha = 0.7
    return torch.clamp(x + alpha*(x-blur), 0, 1)

def t_gamma(x, gamma=1.2):
    return torch.clamp(x,0,1) ** gamma

def t_noise(x, sigma=0.01):
    return torch.clamp(x + sigma*torch.randn_like(x), 0, 1)

TRAIN_ATKS_NO_CROP = [
    ("clean",     lambda y,c: (y,c)),
    ("blur",      lambda y,c: (t_blur(y), c)),
    ("sharpen",   lambda y,c: (t_sharpen(y), c)),
    ("gamma12",   lambda y,c: (t_gamma(y, 1.2), c)),
    ("noise",     lambda y,c: (t_noise(y, 0.01), c)),
]

def sample_train_channel(stage, robust_epoch_idx=0):
    if stage == "warmup":
        return ("clean", "clean", None)

    t = 0.0 if EPOCHS_ROBUST <= 1 else (robust_epoch_idx / (EPOCHS_ROBUST - 1))
    crop_p = CROP_P_START + (CROP_P_END - CROP_P_START) * t

    if random.random() < JPEG_TRAIN_P:
        return ("jpeg", "jpeg", random.choice(JPEG_QUALS))

    # crop path
    if random.random() < crop_p:
        return ("crop10", "crop10", None)

    # include paired resize as an option
    if random.random() < 0.25:
        return ("resize075", "resize075", None)

    name, fn = random.choice(TRAIN_ATKS_NO_CROP)
    return (name, name, None)


def train_one_epoch(stage, robust_epoch_idx=0):
    Enc.train(); Dec.train(); Adv.train()
    last = None

    for (C, P, bits) in train_loader:
        C = C.to(DEVICE, non_blocking=True)
        P = P.to(DEVICE, non_blocking=True)
        bits = bits.to(DEVICE, non_blocking=True)

        Y, r = Enc(C, P)

        ch_name, ch_kind, ch_q = sample_train_channel(stage, robust_epoch_idx)

        # Apply channel to Y and C consistently
        if ch_kind == "jpeg":
            Y_noised = jpeg_batch_torch(Y, quality=ch_q, device=DEVICE)
            C_noised = jpeg_batch_torch(C, quality=ch_q, device=DEVICE)
        elif ch_kind == "crop10":
            Y_noised, C_noised = t_crop_resize_pair(Y, C, crop_frac=0.10)
        elif ch_kind == "resize075":
            Y_noised, C_noised = t_resize_pair(Y, C, scale=0.75)
        else:
            # non-geometric: operate on Y only; keep C aligned
            fn = dict(TRAIN_ATKS_NO_CROP).get(ch_kind, None)
            if fn is None:
                Y_noised, C_noised = Y, C
            else:
                Y_noised, C_noised = fn(Y, C)

        bits_grid = bits.view(-1, gh, gw)
        logits_grid = Dec(Y_noised, C_noised)
        loss_msg = bce_logits(logits_grid, bits_grid)

        loss_imp = l1(Y, C)
        loss_res = torch.mean(torch.abs(r)) / (EPS + 1e-8)

        # Presence detector training ALWAYS
        adv_logits_w = Adv(Y_noised.detach())
        adv_logits_c = Adv(C_noised.detach())
        adv_loss = bce_logits(adv_logits_w, torch.ones_like(adv_logits_w)) + \
                   bce_logits(adv_logits_c, torch.zeros_like(adv_logits_c))

        optAdv.zero_grad(set_to_none=True)
        adv_loss.backward()
        torch.nn.utils.clip_grad_norm_(Adv.parameters(), 5.0)
        optAdv.step()

        # Encoder fooling (optional)
        loss_fool = torch.tensor(0.0, device=DEVICE)
        if LAMBDA_ADV > 0:
            adv_logits_enc = Adv(Y_noised)
            loss_fool = bce_logits(adv_logits_enc, torch.zeros_like(adv_logits_enc))

        loss = (LAMBDA_MSG * loss_msg +
                LAMBDA_IMP * loss_imp +
                LAMBDA_RES * loss_res +
                LAMBDA_ADV * loss_fool)

        optEncDec.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(Enc.parameters()) + list(Dec.parameters()), 5.0)
        optEncDec.step()

        last = (loss.item(), loss_msg.item(), loss_imp.item(), loss_res.item(),
                loss_fool.item(), adv_loss.item(), ch_name)

    return last

# ============================================================
# TRAIN
# ============================================================
ensure_dir(OUT_DIR)
train_tracker = cc_start("train_hidden_ndb_curriculum", OUT_DIR)
t0 = time.time()

for ep in range(EPOCHS_WARMUP):
    last = train_one_epoch("warmup", 0)
    print(f"[Warmup] {ep+1}/{EPOCHS_WARMUP} | "
          f"loss={last[0]:.4f} (msg={last[1]:.4f}, imp={last[2]:.4f}, res={last[3]:.4f}, "
          f"fool={last[4]:.4f}, adv={last[5]:.4f}, ch={last[6]})")

for ep in range(EPOCHS_ROBUST):
    last = train_one_epoch("robust", ep)
    print(f"[Robust] {ep+1}/{EPOCHS_ROBUST} | "
          f"loss={last[0]:.4f} (msg={last[1]:.4f}, imp={last[2]:.4f}, res={last[3]:.4f}, "
          f"fool={last[4]:.4f}, adv={last[5]:.4f}, ch={last[6]})")

train_seconds = time.time() - t0
train_co2 = cc_stop(train_tracker)

# Save
Enc_path = os.path.join(OUT_DIR, "hidden_ndb_Enc.pt")
Dec_path = os.path.join(OUT_DIR, "hidden_ndb_DecGrid.pt")
Adv_path = os.path.join(OUT_DIR, "hidden_ndb_Adv.pt")

torch.save(Enc.state_dict(), Enc_path)
torch.save(Dec.state_dict(), Dec_path)
torch.save(Adv.state_dict(), Adv_path)

model_file_mb = (os.path.getsize(Enc_path)+os.path.getsize(Dec_path)+os.path.getsize(Adv_path)) / (1024**2)
print("Saved:", Enc_path, Dec_path, Adv_path, f"| total {model_file_mb:.2f} MB")

# ============================================================
# EVAL (STANDARD): Robustness + Presence (ROC/AUC + Confusion)
# ============================================================
infer_tracker = cc_start("infer_hidden_ndb", OUT_DIR)
t0 = time.time()

Enc.eval(); Dec.eval(); Adv.eval()

cached = []
with torch.no_grad():
    for j, (C, P, bits) in enumerate(test_loader):
        if j >= min(EVAL_N, n_test):
            break
        C = C.to(DEVICE); P = P.to(DEVICE)
        Y, _ = Enc(C, P)
        cached.append((
            C[0].detach().cpu().permute(1,2,0).numpy(),
            Y[0].detach().cpu().permute(1,2,0).numpy(),
            bits[0].numpy().astype(np.int32)
        ))

attacks = list(ATTACKS.keys())

# ---- Paired geometric attacks for eval (same params for Y and C) ----
def paired_resize_np(Y, C, scale=0.75):
    H, W, _ = Y.shape
    nh, nw = max(1, int(H*scale)), max(1, int(W*scale))
    Yp = np_to_pil(Y).resize((nw, nh), Image.BICUBIC).resize((W, H), Image.BICUBIC)
    Cp = np_to_pil(C).resize((nw, nh), Image.BICUBIC).resize((W, H), Image.BICUBIC)
    return pil_to_np(Yp), pil_to_np(Cp)

def paired_crop_np(Y, C, crop_frac=0.10):
    H, W, _ = Y.shape
    ch, cw = max(1, int(H*(1-crop_frac))), max(1, int(W*(1-crop_frac)))
    y0 = np.random.randint(0, H - ch + 1)
    x0 = np.random.randint(0, W - cw + 1)
    Yc = Y[y0:y0+ch, x0:x0+cw, :]
    Cc = C[y0:y0+ch, x0:x0+cw, :]
    Yp = np_to_pil(Yc).resize((W, H), Image.BICUBIC)
    Cp = np_to_pil(Cc).resize((W, H), Image.BICUBIC)
    return pil_to_np(Yp), pil_to_np(Cp)

def paired_screenshot_np(Y, C, q1=35, q2=80):
    Y1 = attack_jpeg(Y, q1); C1 = attack_jpeg(C, q1)
    Y2, C2 = paired_resize_np(Y1, C1, 0.92)
    Y3 = attack_jpeg(Y2, q2); C3 = attack_jpeg(C2, q2)
    return Y3, C3

# Robustness rows
rows = []
# Presence score banks (STANDARD)
pres_pos = {a: [] for a in attacks}
pres_neg = {a: [] for a in attacks}

for (C_np, Y_np, bits_np) in cached:
    psnr_c  = psnr(Y_np, C_np)
    ssim_c  = ssim_np(Y_np, C_np)
    lpips_c = lpips_score_np(Y_np, C_np, device=DEVICE) if (TORCH_OK and LPIPS_OK) else float("nan")

    rob = {}
    for atk_name, atk_fn in ATTACKS.items():

        # Decide Ya, Ca correctly (paired for geometric)
        if atk_name == "crop10":
            Ya, Ca = paired_crop_np(Y_np, C_np, 0.10)
        elif atk_name == "resize075":
            Ya, Ca = paired_resize_np(Y_np, C_np, 0.75)
        elif atk_name == "screenshot":
            Ya, Ca = paired_screenshot_np(Y_np, C_np, 35, 80)
        elif atk_name in ("jpeg10","jpeg30","jpeg50","jpeg70","jpeg90"):
            Ya = atk_fn(Y_np)
            Ca = atk_fn(C_np)
        else:
            Ya = atk_fn(Y_np)
            Ca = C_np

        Ya_t = torch.from_numpy(Ya).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
        Ca_t = torch.from_numpy(Ca).permute(2,0,1).unsqueeze(0).float().to(DEVICE)

        with torch.no_grad():
            logits_grid = Dec(Ya_t, Ca_t)
            probs_grid  = torch.sigmoid(logits_grid)[0].detach().cpu().numpy()

        pred_bits = (probs_grid.reshape(-1) >= 0.5).astype(np.int32)
        ber_bits  = float(np.mean(pred_bits != bits_np) * 100.0)
        bit_acc   = float(100.0 - ber_bits)

        tb = (bits_np * 2 - 1).astype(np.float32)
        pb = (pred_bits * 2 - 1).astype(np.float32)
        nc_bits = float(np.mean(tb * pb))

        rob[atk_name] = {"NC_bits": nc_bits, "BER": ber_bits, "bit_acc": bit_acc}

        # Presence scores (STANDARD)
        with torch.no_grad():
            s_pos = float(torch.sigmoid(Adv(Ya_t)).item())
            s_neg = float(torch.sigmoid(Adv(Ca_t)).item())
        pres_pos[atk_name].append(s_pos)
        pres_neg[atk_name].append(s_neg)

    rows.append({"psnr_cover": psnr_c, "ssim_cover": ssim_c, "lpips_cover": lpips_c, "robustness": rob})

# Presence metrics + ROC plots + confusion heatmaps (STANDARD)
presence_auc = {}
presence_conf = {}
for atk_name in attacks:
    pos_scores = pres_pos[atk_name]
    neg_scores = pres_neg[atk_name]

    y_true = np.array([1]*len(pos_scores) + [0]*len(neg_scores))
    y_score = np.array(pos_scores + neg_scores)

    A = float(roc_auc_score(y_true, y_score))
    cm = confusion_at_fpr(pos_scores, neg_scores, target_fpr=0.01)

    presence_auc[atk_name] = A
    presence_conf[atk_name] = cm

    fpr, tpr, _ = roc_curve(y_true, y_score)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={A:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title(f"HiDDeN Presence ROC (STANDARD) — {atk_name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"hidden_presence_roc_standard_{atk_name}.png"), dpi=220)
    plt.close()

    plot_confusion_heatmap(
        cm,
        f"Presence Confusion @1%FPR (STANDARD) — {atk_name}",
        os.path.join(OUT_DIR, f"hidden_presence_confusion_standard_{atk_name}.png")
    )

# Clean ROC “main” plot
if "clean" in attacks:
    pos = pres_pos["clean"]
    neg = pres_neg["clean"]
    y_true = np.array([1]*len(pos) + [0]*len(neg))
    y_score = np.array(pos + neg)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    A = float(roc_auc_score(y_true, y_score))

    plt.figure()
    plt.plot(fpr, tpr, label=f"Clean AUC={A:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title("HiDDeN Presence ROC (clean): watermarked vs cover")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "hidden_presence_roc_clean.png"), dpi=260)
    plt.close()

infer_seconds = time.time() - t0
infer_co2 = cc_stop(infer_tracker)

# ============================================================
# AI laundering (SEPARATE) + Presence after laundering + plots
# NOTE: laundering acts on suspect Ya only (attacker removes watermark),
#       cover Ca remains what verifier uses (attacked-cover for geometric).
# ============================================================
pairs = []
for (C_np, Y_np, _) in cached[:50]:
    y1 = attack_jpeg(Y_np, 10)
    y2 = attack_resize(Y_np, 0.75)
    y3 = attack_screenshot_recompress(Y_np, 35, 80)
    pairs.append((y1, C_np))
    pairs.append((y2, C_np))
    pairs.append((y3, C_np))

launder_fn, launder_meta = train_ai_launderer_patchwise(
    pairs=pairs, out_dir=OUT_DIR, patch=64, patches_per_img=2, epochs=1, lr=1e-3, max_pairs=150, device=DEVICE
)

rows_launder = []
presence_auc_L, presence_conf_L = {}, {}
presL_pos = {a: [] for a in attacks}
presL_neg = {a: [] for a in attacks}

if launder_fn is not None:
    for (C_np, Y_np, bits_np) in cached:
        robL = {}
        for atk_name, atk_fn in ATTACKS.items():

            # Build attacked Ya, Ca (paired for geometric)
            if atk_name == "crop10":
                Ya, Ca = paired_crop_np(Y_np, C_np, 0.10)
            elif atk_name == "resize075":
                Ya, Ca = paired_resize_np(Y_np, C_np, 0.75)
            elif atk_name == "screenshot":
                Ya, Ca = paired_screenshot_np(Y_np, C_np, 35, 80)
            elif atk_name in ("jpeg10","jpeg30","jpeg50","jpeg70","jpeg90"):
                Ya = atk_fn(Y_np)
                Ca = atk_fn(C_np)
            else:
                Ya = atk_fn(Y_np)
                Ca = C_np

            # Apply laundering (skip clean by default)
            if atk_name != "clean":
                Ya = np.clip(launder_fn(Ya), 0, 1)

            Ya_t = torch.from_numpy(Ya).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
            Ca_t = torch.from_numpy(Ca).permute(2,0,1).unsqueeze(0).float().to(DEVICE)

            with torch.no_grad():
                logits_grid = Dec(Ya_t, Ca_t)
                probs_grid  = torch.sigmoid(logits_grid)[0].detach().cpu().numpy()

            pred_bits = (probs_grid.reshape(-1) >= 0.5).astype(np.int32)
            ber_bits  = float(np.mean(pred_bits != bits_np) * 100.0)
            bit_acc   = float(100.0 - ber_bits)

            tb = (bits_np * 2 - 1).astype(np.float32)
            pb = (pred_bits * 2 - 1).astype(np.float32)
            nc_bits = float(np.mean(tb * pb))

            robL[atk_name] = {"NC_bits": nc_bits, "BER": ber_bits, "bit_acc": bit_acc}

            # Presence after laundering
            with torch.no_grad():
                s_pos = float(torch.sigmoid(Adv(Ya_t)).item())
                s_neg = float(torch.sigmoid(Adv(Ca_t)).item())
            presL_pos[atk_name].append(s_pos)
            presL_neg[atk_name].append(s_neg)

        rows_launder.append({"robustness": robL})

    # Presence metrics + ROC plots + confusion heatmaps (LAUNDERED)
    for atk_name in attacks:
        pos_scores = presL_pos[atk_name]
        neg_scores = presL_neg[atk_name]
        y_true = np.array([1]*len(pos_scores) + [0]*len(neg_scores))
        y_score = np.array(pos_scores + neg_scores)

        A = float(roc_auc_score(y_true, y_score))
        cm = confusion_at_fpr(pos_scores, neg_scores, target_fpr=0.01)

        presence_auc_L[atk_name] = A
        presence_conf_L[atk_name] = cm

        fpr, tpr, _ = roc_curve(y_true, y_score)
        plt.figure()
        plt.plot(fpr, tpr, label=f"AUC={A:.3f}")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("FPR"); plt.ylabel("TPR")
        plt.title(f"HiDDeN Presence ROC (LAUNDERED) — {atk_name}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f"hidden_presence_roc_laundered_{atk_name}.png"), dpi=220)
        plt.close()

        plot_confusion_heatmap(
            cm,
            f"Presence Confusion @1%FPR (LAUNDERED) — {atk_name}",
            os.path.join(OUT_DIR, f"hidden_presence_confusion_laundered_{atk_name}.png")
        )

# ============================================================
# SUMMARY (STANDARD + LAUNDERED)
# ============================================================
summary = {
    "model": "hidden_ndb_nonblind_attackaware_griddecoder_curriculum_v2_datasetpayload_pairedgeom",
    "payload_bits": PAYLOAD_BITS,
    "hw": HW,
    "eps": EPS,
    "grid": [gh, gw],
    "n_train": len(train_ds),
    "n_test_eval": len(rows),

    "mean_psnr_cover": float(np.mean([r["psnr_cover"] for r in rows])),
    "mean_ssim_cover": float(np.mean([r["ssim_cover"] for r in rows])),
    "mean_lpips_cover": float(np.nanmean([r["lpips_cover"] for r in rows])),

    "mean_nc_bits": {a: float(np.mean([r["robustness"][a]["NC_bits"] for r in rows])) for a in attacks},
    "mean_ber":     {a: float(np.mean([r["robustness"][a]["BER"] for r in rows])) for a in attacks},
    "mean_bit_acc": {a: float(np.mean([r["robustness"][a]["bit_acc"] for r in rows])) for a in attacks},

    "presence_auc_by_attack": presence_auc,
    "presence_confusion_1pctfpr_by_attack": presence_conf,

    "train_seconds": float(train_seconds),
    "infer_seconds": float(infer_seconds),
    "train_co2_kg": float(train_co2) if train_co2 is not None else None,
    "infer_co2_kg": float(infer_co2) if infer_co2 is not None else None,
    "model_file_mb": float(model_file_mb),

    "ai_laundering_meta": launder_meta,

    "train_recipe": {
        "warmup_epochs": EPOCHS_WARMUP,
        "robust_epochs": EPOCHS_ROBUST,
        "lambda_msg": LAMBDA_MSG,
        "lambda_imp": LAMBDA_IMP,
        "lambda_res": LAMBDA_RES,
        "lambda_adv": LAMBDA_ADV,
        "jpeg_train_p": JPEG_TRAIN_P,
        "jpeg_quals": JPEG_QUALS,
        "crop_p_start": CROP_P_START,
        "crop_p_end": CROP_P_END,
        "decoder": "nonblind_grid(ghxgw)_fullyconv",
        "note": "Main robustness/presence = attacks only; laundering reported separately; paired geom transforms."
    }
}

if len(rows_launder) > 0:
    summary["mean_nc_bits_laundered"] = {a: float(np.mean([r["robustness"][a]["NC_bits"] for r in rows_launder])) for a in attacks}
    summary["mean_ber_laundered"]     = {a: float(np.mean([r["robustness"][a]["BER"] for r in rows_launder])) for a in attacks}
    summary["mean_bit_acc_laundered"] = {a: float(np.mean([r["robustness"][a]["bit_acc"] for r in rows_launder])) for a in attacks}

    summary["presence_auc_by_attack_laundered"] = presence_auc_L
    summary["presence_confusion_1pctfpr_by_attack_laundered"] = presence_conf_L
else:
    summary["mean_nc_bits_laundered"] = None
    summary["mean_ber_laundered"] = None
    summary["mean_bit_acc_laundered"] = None
    summary["presence_auc_by_attack_laundered"] = None
    summary["presence_confusion_1pctfpr_by_attack_laundered"] = None

# ============================================================
# Attack plots (STANDARD + LAUNDERED + delta)
# ============================================================
plot_attack_metric_line(attacks, summary["mean_nc_bits"],
                        "HiDDeN mean NC_bits under attacks (standard)",
                        os.path.join(OUT_DIR, "hidden_NC_attacks.png"),
                        ylabel="NC_bits", ylim=(-1, 1))

plot_attack_metric_line(attacks, summary["mean_ber"],
                        "HiDDeN mean BER (%) under attacks (standard)",
                        os.path.join(OUT_DIR, "hidden_BER_attacks.png"),
                        ylabel="BER (%)", ylim=(0, 100))

plot_attack_metric_line(attacks, summary["mean_bit_acc"],
                        "HiDDeN mean Bit Accuracy (%) under attacks (standard)",
                        os.path.join(OUT_DIR, "hidden_bitacc_attacks.png"),
                        ylabel="Bit Accuracy (%)", ylim=(0, 105))

if summary["mean_bit_acc_laundered"] is not None:
    plot_attack_metric_line(attacks, summary["mean_nc_bits_laundered"],
                            "HiDDeN mean NC_bits under attacks (after laundering)",
                            os.path.join(OUT_DIR, "hidden_NC_attacks_laundered.png"),
                            ylabel="NC_bits", ylim=(-1, 1))

    plot_attack_metric_line(attacks, summary["mean_ber_laundered"],
                            "HiDDeN mean BER (%) under attacks (after laundering)",
                            os.path.join(OUT_DIR, "hidden_BER_attacks_laundered.png"),
                            ylabel="BER (%)", ylim=(0, 100))

    plot_attack_metric_line(attacks, summary["mean_bit_acc_laundered"],
                            "HiDDeN mean Bit Accuracy (%) under attacks (after laundering)",
                            os.path.join(OUT_DIR, "hidden_bitacc_attacks_laundered.png"),
                            ylabel="Bit Accuracy (%)", ylim=(0, 105))

    bit_std = np.array([summary["mean_bit_acc"][a] for a in attacks], dtype=float)
    bit_lau = np.array([summary["mean_bit_acc_laundered"][a] for a in attacks], dtype=float)
    delta = bit_std - bit_lau

    plt.figure(figsize=(12,4))
    plt.bar(attacks, delta)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Bit-Accuracy Drop (pp)")
    plt.title("AI Laundering Impact: Δ(BitAcc) = Standard − Laundered (HiDDeN)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "hidden_delta_bitacc_standard_minus_laundered.png"), dpi=220)
    plt.close()

# ============================================================
# Save summary
# ============================================================
save_json(summary, os.path.join(OUT_DIR, "hidden_ndb_summary.json"))
print(json.dumps(summary, indent=2))
print("✅ Outputs in:", OUT_DIR)

HiDDeN-N-DB curriculum: | warmup: 2 | robust: 6 | L_msg: 10.0 | L_imp: 0.1 | L_res: 0.05 | L_adv: 0.0 | JPEG_P: 0.5 | JPEG_QUALS: [10, 30, 50] | crop_p: (0.05, '->', 0.25) | grid: (16, 16) | DEVICE: cuda
Using n_train = 5000 | n_test = 1000 | eval = 1000


[codecarbon WARNING @ 01:17:20] Multiple instances of codecarbon are allowed to run at the same time.


[Warmup] 1/2 | loss=0.1945 (msg=0.0163, imp=0.0120, res=0.6042, fool=0.0000, adv=1.2520, ch=clean)
[Warmup] 2/2 | loss=0.0611 (msg=0.0042, imp=0.0072, res=0.3605, fool=0.0000, adv=1.1410, ch=clean)
[Robust] 1/6 | loss=4.3913 (msg=0.4356, imp=0.0135, res=0.6836, fool=0.0000, adv=1.3472, ch=jpeg)
[Robust] 2/6 | loss=0.1262 (msg=0.0091, imp=0.0133, res=0.6783, fool=0.0000, adv=1.1728, ch=jpeg)
[Robust] 3/6 | loss=0.1034 (msg=0.0071, imp=0.0122, res=0.6138, fool=0.0000, adv=1.0932, ch=jpeg)
[Robust] 4/6 | loss=0.2410 (msg=0.0207, imp=0.0129, res=0.6598, fool=0.0000, adv=0.4836, ch=blur)
[Robust] 5/6 | loss=6.1334 (msg=0.6101, imp=0.0122, res=0.6198, fool=0.0000, adv=1.2809, ch=crop10)
[Robust] 6/6 | loss=0.2625 (msg=0.0229, imp=0.0126, res=0.6375, fool=0.0000, adv=1.1591, ch=jpeg)
Saved: /kaggle/working/flexmark_out_bits256/hidden_ndb_Enc.pt /kaggle/working/flexmark_out_bits256/hidden_ndb_DecGrid.pt /kaggle/working/flexmark_out_bits256/hidden_ndb_Adv.pt | total 1.05 MB


/tmp/ipykernel_55/824233214.py:661: RuntimeWarning: Mean of empty slice
  "mean_lpips_cover": float(np.nanmean([r["lpips_cover"] for r in rows])),


{
  "model": "hidden_ndb_nonblind_attackaware_griddecoder_curriculum_v2_datasetpayload_pairedgeom",
  "payload_bits": 256,
  "hw": 128,
  "eps": 0.02,
  "grid": [
    16,
    16
  ],
  "n_train": 5000,
  "n_test_eval": 1000,
  "mean_psnr_cover": 37.094904775992475,
  "mean_ssim_cover": 0.9696250320672989,
  "mean_lpips_cover": NaN,
  "mean_nc_bits": {
    "clean": 0.9999765625,
    "jpeg10": 0.622203125,
    "jpeg30": 0.9547734375,
    "jpeg50": 0.99446875,
    "jpeg70": 0.9992421875,
    "jpeg90": 0.999984375,
    "resize075": 1.0,
    "crop10": 0.2392421875,
    "blur": 0.832859375,
    "sharpen": 0.99946875,
    "gamma12": 0.1011484375,
    "screenshot": 0.9668984375,
    "launder_only": 0.9999765625
  },
  "mean_ber": {
    "clean": 0.001171875,
    "jpeg10": 18.88984375,
    "jpeg30": 2.261328125,
    "jpeg50": 0.2765625,
    "jpeg70": 0.037890625,
    "jpeg90": 0.00078125,
    "resize075": 0.0,
    "crop10": 38.037890625,
    "blur": 8.35703125,
    "sharpen": 0.0265625,
    "gam

### 128 bits RCRA-RL

In [6]:
# ============================================================
# CELL 4 — RCRA-RL (Controller) Model Classes
# Reviewer-aligned:
# - Grid decoder (8×16 for 128 bits; 16×16 for 256 bits)
# - Non-blind decode: inputs [Y_noised, C_noised]
# - RL is a CONTROLLER: chooses (mask grid + strength level)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------------------
# Small conv block
# ---------------------------
class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, s, p),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

# ============================================================
# RCRA Encoder (supervised)
# Input: concat([C (3ch), P (1ch)]) -> residual r_raw (3ch)
# ============================================================
class FLEXEncoder(nn.Module):
    def __init__(self, payload_ch=1, width=64):
        super().__init__()
        in_ch = 3 + payload_ch
        self.net = nn.Sequential(
            ConvBNReLU(in_ch, width),
            ConvBNReLU(width, width),
            ConvBNReLU(width, width),
            nn.Conv2d(width, 3, 1, 1, 0),
        )

    def forward(self, C, P):
        x = torch.cat([C, P], dim=1)
        r_raw = self.net(x)                    # unconstrained residual
        return r_raw

# ============================================================
# RCRA Grid Decoder (supervised)
# Output logits_grid: (B, gh, gw)
# ============================================================
class FLEXDecoderNonBlindGrid(nn.Module):
    def __init__(self, gh=8, gw=16, width=64):
        super().__init__()
        self.gh, self.gw = int(gh), int(gw)
        self.f = nn.Sequential(
            ConvBNReLU(6, width, s=1),
            ConvBNReLU(width, width, s=2),
            ConvBNReLU(width, width, s=2),
            ConvBNReLU(width, width, s=2),
            nn.Conv2d(width, 1, 1, 1, 0),
        )

    def forward(self, Y_noised, C_noised):
        x = torch.cat([Y_noised, C_noised], dim=1)
        z = self.f(x)  # (B,1,h,w)
        z = F.interpolate(z, size=(self.gh, self.gw), mode="bilinear", align_corners=False)
        return z.squeeze(1)  # (B,gh,gw)

# ============================================================
# RL Controller
# - action 1: mask grid over payload cells (Bernoulli per cell)
# - action 2: strength level (Categorical)
#
# IMPORTANT: RL is a controller, not the embedder.
# ============================================================
class FLEXRLController(nn.Module):
    def __init__(self, gh=8, gw=16, width=32, strength_levels=(0.25, 0.50, 0.75, 1.00)):
        super().__init__()
        self.gh, self.gw = int(gh), int(gw)
        self.strength_levels = list(map(float, strength_levels))
        nS = len(self.strength_levels)

        # small feature extractor on cover (fast)
        self.feat = nn.Sequential(
            ConvBNReLU(3, width, s=2),
            ConvBNReLU(width, width, s=2),
            ConvBNReLU(width, width, s=2),
        )

        # mask head -> logits over (gh, gw)
        self.mask_head = nn.Conv2d(width, 1, 1, 1, 0)

        # strength head -> logits over strength levels
        self.str_head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Linear(width, nS)
        )

    def forward(self, C):
        """
        Returns:
          mask_logits: (B, gh, gw)
          str_logits:  (B, nS)
        """
        f = self.feat(C)  # (B,width,h,w)
        m = self.mask_head(f)  # (B,1,h,w)
        m = F.interpolate(m, size=(self.gh, self.gw), mode="bilinear", align_corners=False).squeeze(1)  # (B,gh,gw)
        s = self.str_head(f)  # (B,nS)
        return m, s

    def sample_actions(self, C, deterministic=False):
        """
        Returns:
          mask: (B,gh,gw) float in {0,1} (or {0,1} det)
          strength: (B,) float from strength_levels
          logprob: (B,) sum log-prob of mask+strength (for REINFORCE)
          info: dict with indices
        """
        mask_logits, str_logits = self.forward(C)

        # strength categorical
        if deterministic:
            s_idx = torch.argmax(str_logits, dim=1)
            s_logprob = torch.zeros((C.size(0),), device=C.device)
        else:
            dist_s = torch.distributions.Categorical(logits=str_logits)
            s_idx = dist_s.sample()
            s_logprob = dist_s.log_prob(s_idx)

        strength = torch.tensor(self.strength_levels, device=C.device)[s_idx]  # (B,)

        # mask Bernoulli per cell
        if deterministic:
            mask = (torch.sigmoid(mask_logits) >= 0.5).float()
            m_logprob = torch.zeros((C.size(0),), device=C.device)
        else:
            dist_m = torch.distributions.Bernoulli(logits=mask_logits)
            mask = dist_m.sample()
            # sum logprob across grid for each sample
            m_logprob = dist_m.log_prob(mask).view(C.size(0), -1).sum(dim=1)

        logprob = m_logprob + s_logprob
        return mask, strength, logprob, {"s_idx": s_idx}

# ============================================================
# Helper: apply controller to residual
# r = tanh(r_raw)*eps
# r_ctrl = r * upsample(mask) * strength
# ============================================================
def apply_controller_to_residual(C, r_raw, mask_grid, strength, eps, hw):
    """
    C: (B,3,H,W)
    r_raw: (B,3,H,W)
    mask_grid: (B,gh,gw) 0/1
    strength: (B,) scalar
    Returns: Y, r_ctrl
    """
    B, _, H, W = C.shape
    assert H == hw and W == hw

    r = torch.tanh(r_raw) * float(eps)  # bounded residual

    # upsample mask grid to image resolution
    m = mask_grid.unsqueeze(1)  # (B,1,gh,gw)
    m_up = F.interpolate(m, size=(hw, hw), mode="nearest")  # (B,1,H,W)

    s = strength.view(B,1,1,1)  # broadcast
    r_ctrl = torch.clamp(r * m_up * s, -float(eps), float(eps))
    Y = torch.clamp(C + r_ctrl, 0.0, 1.0)
    return Y, r_ctrl

print("CELL 4 ready: FLEXEncoder + FLEXDecoderNonBlindGrid + FLEXRLController.")


CELL 4 ready: FLEXEncoder + FLEXDecoderNonBlindGrid + FLEXRLController.


In [7]:
# ============================================================
# CELL 5 — FLEXMark-RL (Controller): Curriculum Train + Eval (FINAL + visuals)
# CLEAN corrected version:
# Dataset-only payload from mark PNG (matches cGAN/HiDDeN protocol)
# RL reward uses SMOOTH (soft) correctness (REINFORCE-friendly)
# Paired geometric transforms for non-blind decoding (resize/crop/screenshot)
# Same reporting: PSNR/SSIM + NC/BER/bit_acc + decision ROC/AUC + confusion@1%FPR
# Separate AI laundering evaluation
# ============================================================

import os, time, random, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from io import BytesIO

import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------
# Utility: save json
# -----------------------
def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

# -----------------------
# Utility: confusion @ target FPR (with actual rates)
# -----------------------
def confusion_at_fpr(scores_pos, scores_neg, target_fpr=0.01):
    scores_pos = np.asarray(scores_pos, dtype=np.float64)
    scores_neg = np.asarray(scores_neg, dtype=np.float64)

    thr = float(np.quantile(scores_neg, 1.0 - target_fpr))

    TP = int(np.sum(scores_pos >= thr))
    FN = int(np.sum(scores_pos <  thr))
    FP = int(np.sum(scores_neg >= thr))
    TN = int(np.sum(scores_neg <  thr))

    fpr_actual = FP / max(1, (FP + TN))
    tpr_actual = TP / max(1, (TP + FN))

    return {
        "thr": thr,
        "TP": TP, "FP": FP, "TN": TN, "FN": FN,
        "fpr_actual": float(fpr_actual),
        "tpr_actual": float(tpr_actual),
    }

def plot_confusion_heatmap(cm_dict, title, out_path):
    mat = np.array([[cm_dict["TN"], cm_dict["FP"]],
                    [cm_dict["FN"], cm_dict["TP"]]], dtype=np.int64)
    plt.figure(figsize=(4.6, 4.1))
    plt.imshow(mat)
    plt.xticks([0,1], ["Pred 0", "Pred 1"])
    plt.yticks([0,1], ["True 0", "True 1"])
    for (r,c), v in np.ndenumerate(mat):
        plt.text(c, r, str(v), ha="center", va="center", fontsize=11)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=240)
    plt.close()

def plot_attack_metric_line(attacks, series_dict, title, out_path, ylabel=None, ylim=None):
    x = np.arange(len(attacks))
    y = [series_dict[a] for a in attacks]
    plt.figure(figsize=(12,4))
    plt.plot(x, y, marker="o")
    plt.xticks(x, attacks, rotation=45, ha="right")
    if ylabel is not None:
        plt.ylabel(ylabel)
    if ylim is not None:
        plt.ylim(*ylim)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=240)
    plt.close()

# -----------------------
# Real JPEG forward helper (per-batch, per-image)
# -----------------------
def _torch_bchw_to_np_bhwc01(x):
    return x.detach().clamp(0,1).cpu().permute(0,2,3,1).numpy()

def _np_bhwc01_to_torch_bchw(x_np, device):
    return torch.from_numpy(x_np).float().permute(0,3,1,2).to(device).clamp(0,1)

def jpeg_batch_torch(Y, quality=50, device=None):
    if device is None:
        device = Y.device
    Y_np = _torch_bchw_to_np_bhwc01(Y)
    out = []
    for i in range(Y_np.shape[0]):
        pil = Image.fromarray((Y_np[i] * 255).astype(np.uint8))
        buf = BytesIO()
        pil.save(buf, format="JPEG", quality=int(quality))
        buf.seek(0)
        rec = Image.open(buf).convert("RGB")
        out.append(np.asarray(rec).astype(np.float32) / 255.0)
    out_np = np.stack(out, axis=0)
    return _np_bhwc01_to_torch_bchw(out_np, device)

# -----------------------
# Config (uses Common Cell 2)
# -----------------------
HW = int(HW)
EPS = float(EPS)
PAYLOAD_BITS = int(PAYLOAD_BITS)
assert PAYLOAD_BITS in (128,256)
gh, gw = (8,16) if PAYLOAD_BITS == 128 else (16,16)

BATCH = 16
LR_ED = 2e-4
LR_POL = 2e-4

LAMBDA_MSG = 10.0
LAMBDA_IMP = 0.10
LAMBDA_RES = 0.05

LAMBDA_RL = 0.25
PSNR_FLOOR = 35.0

EPOCHS_WARMUP = 2
EPOCHS_ROBUST = 6

JPEG_TRAIN_P = 0.50
JPEG_QUALS = [10, 30, 50]

CROP_P_START = 0.05
CROP_P_END   = 0.25

print("FLEXMark-RL curriculum:",
      "| warmup:", EPOCHS_WARMUP,
      "| robust:", EPOCHS_ROBUST,
      "| grid:", (gh,gw),
      "| EPS:", EPS,
      "| JPEG_P:", JPEG_TRAIN_P,
      "| JPEG_QUALS:", JPEG_QUALS,
      "| PSNR_FLOOR:", PSNR_FLOOR,
      "| DEVICE:", DEVICE)

# ============================================================
# Dataset: DATASET-ONLY payload from mark PNG (paired cover+mark)
# Common Cell 2 provides: make_payload_for_model(mark_path, hw, bitsB, mode)
# Returns: (P_np (H,W,1), Wtrue_np (gh,gw,1), meta)
# ============================================================
class CoverMarkRLDataset(Dataset):
    def __init__(self, cover_paths, mark_paths, hw, payload_bits):
        assert len(cover_paths) == len(mark_paths), "cover/mark list mismatch"
        self.cover_paths = cover_paths
        self.mark_paths  = mark_paths
        self.hw = int(hw)
        self.payload_bits = int(payload_bits)

    def __len__(self):
        return len(self.cover_paths)

    def __getitem__(self, idx):
        C = read_rgb(self.cover_paths[idx], self.hw)

        # dataset-only payload derived deterministically from mark PNG
        P_np, Wtrue_np, _ = make_payload_for_model(
            self.mark_paths[idx], self.hw, self.payload_bits, mode="1ch"
        )
        bits_np = (Wtrue_np[..., 0] >= 0.5).astype(np.int32).reshape(-1)  # (payload_bits,)

        C_t = torch.from_numpy(C).permute(2,0,1).float()          # (3,H,W)
        P_t = torch.from_numpy(P_np).permute(2,0,1).float()       # (1,H,W)
        bits_t = torch.from_numpy(bits_np.astype(np.float32))     # (payload_bits,)
        return C_t, P_t, bits_t

train_ds = CoverMarkRLDataset(trC[:n_train], trW[:n_train], HW, PAYLOAD_BITS)
test_ds  = CoverMarkRLDataset(teC[:n_test],  teW[:n_test],  HW, PAYLOAD_BITS)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False, num_workers=2)

print(f"Using n_train = {len(train_ds)} | n_test = {len(test_ds)} | eval = {min(EVAL_N, len(test_ds))}")

# -----------------------
# Instantiate models (from Cell 4)
# -----------------------
Enc = FLEXEncoder(payload_ch=1, width=64).to(DEVICE)
Dec = FLEXDecoderNonBlindGrid(gh=gh, gw=gw, width=64).to(DEVICE)
Pol = FLEXRLController(gh=gh, gw=gw, width=32, strength_levels=(0.25, 0.50, 0.75, 1.00)).to(DEVICE)

optED  = torch.optim.Adam(list(Enc.parameters()) + list(Dec.parameters()), lr=LR_ED, betas=(0.5,0.999))
optPol = torch.optim.Adam(Pol.parameters(), lr=LR_POL, betas=(0.5,0.999))

bce_logits = nn.BCEWithLogitsLoss()
l1 = nn.L1Loss()

# -----------------------
# Differentiable-ish attacks for training (torch)
# -----------------------
def t_resize(x, scale=0.75):
    B,C,H,W = x.shape
    nh, nw = max(4,int(H*scale)), max(4,int(W*scale))
    y = F.interpolate(x, size=(nh,nw), mode="bilinear", align_corners=False)
    y = F.interpolate(y, size=(H,W), mode="bilinear", align_corners=False)
    return y

def t_crop_resize(x, crop_frac=0.10):
    B,C,H,W = x.shape
    ch, cw = max(4,int(H*(1-crop_frac))), max(4,int(W*(1-crop_frac)))
    y0 = torch.randint(0, H-ch+1, (1,), device=x.device).item()
    x0 = torch.randint(0, W-cw+1, (1,), device=x.device).item()
    crop = x[:, :, y0:y0+ch, x0:x0+cw]
    y = F.interpolate(crop, size=(H,W), mode="bilinear", align_corners=False)
    return y

def t_blur(x):
    k=3; pad=1
    w = torch.ones((x.shape[1],1,k,k), device=x.device) / (k*k)
    return F.conv2d(x, w, padding=pad, groups=x.shape[1])

def t_sharpen(x):
    blur = t_blur(x)
    alpha = 0.7
    return torch.clamp(x + alpha*(x-blur), 0, 1)

def t_gamma(x, gamma=1.2):
    return torch.clamp(x,0,1) ** gamma

def t_noise(x, sigma=0.01):
    return torch.clamp(x + sigma*torch.randn_like(x), 0, 1)

def sample_train_channel(stage, crop_prob=0.10):
    if stage == "warmup":
        return ("clean", lambda z: z, None)

    if random.random() < JPEG_TRAIN_P:
        return ("jpeg", None, random.choice(JPEG_QUALS))

    if random.random() < crop_prob:
        return ("crop10", lambda z: t_crop_resize(z, 0.10), None)

    choices = [
        ("clean",     lambda z: z),
        ("resize075", lambda z: t_resize(z, 0.75)),
        ("blur",      lambda z: t_blur(z)),
        ("sharpen",   lambda z: t_sharpen(z)),
        ("gamma12",   lambda z: t_gamma(z, 1.2)),
        ("noise",     lambda z: t_noise(z, 0.01)),
    ]
    name, fn = random.choice(choices)
    return (name, fn, None)

# -----------------------
# Reward utilities
# -----------------------
def psnr_torch(a, b):
    mse = torch.mean((a-b)**2, dim=(1,2,3)).clamp_min(1e-12)
    return 10.0 * torch.log10(1.0 / mse)

rl_baseline = 0.0
BASELINE_MOM = 0.95

def train_one_epoch(stage, crop_prob):
    global rl_baseline
    Enc.train(); Dec.train(); Pol.train()

    last = None
    for (C, P, bits) in train_loader:
        C = C.to(DEVICE, non_blocking=True)
        P = P.to(DEVICE, non_blocking=True)
        bits = bits.to(DEVICE, non_blocking=True)

        # Controller: warmup uses full mask + full strength
        if stage == "warmup":
            mask = torch.ones((C.size(0), gh, gw), device=DEVICE)
            strength = torch.ones((C.size(0),), device=DEVICE)
            logprob = torch.zeros((C.size(0),), device=DEVICE)
        else:
            mask, strength, logprob, _ = Pol.sample_actions(C, deterministic=False)

        # Supervised embedder
        r_raw = Enc(C, P)
        Y, r_ctrl = apply_controller_to_residual(C, r_raw, mask, strength, EPS, HW)

        # Train channel (attacks)
        ch_name, ch_fn, ch_q = sample_train_channel(stage, crop_prob=crop_prob)

        if ch_name == "jpeg":
            Y_noised = jpeg_batch_torch(Y, quality=ch_q, device=DEVICE)
            C_noised = jpeg_batch_torch(C, quality=ch_q, device=DEVICE)
        else:
            Y_noised = ch_fn(Y) if ch_fn is not None else Y
            if ch_name in ("resize075","crop10"):
                C_noised = ch_fn(C)
            else:
                C_noised = C

        bits_grid = bits.view(-1, gh, gw)
        logits_grid = Dec(Y_noised, C_noised)

        loss_msg = bce_logits(logits_grid, bits_grid)
        loss_imp = l1(Y, C)
        loss_res = torch.mean(torch.abs(r_ctrl)) / (EPS + 1e-8)

        loss_sup = (LAMBDA_MSG * loss_msg +
                    LAMBDA_IMP * loss_imp +
                    LAMBDA_RES * loss_res)

        # RL loss (REINFORCE) with SMOOTH reward
        rl_loss = torch.tensor(0.0, device=DEVICE)
        if stage != "warmup":
            with torch.no_grad():
                # smooth correctness in [0,1]
                p = torch.sigmoid(logits_grid).view(C.size(0), -1)
                t = bits.view(C.size(0), -1)
                soft_acc = torch.mean(t * p + (1 - t) * (1 - p), dim=1)  # (B,)

                psnrC = psnr_torch(Y, C)
                psnr_pen = torch.clamp(PSNR_FLOOR - psnrC, min=0.0) / 10.0
                budget_pen = torch.mean(torch.abs(r_ctrl), dim=(1,2,3)) / (EPS + 1e-8)

                reward = (1.0 * soft_acc) - (0.20 * psnr_pen) - (0.10 * budget_pen)

                r_mean = reward.mean().item()
                rl_baseline = BASELINE_MOM * rl_baseline + (1-BASELINE_MOM) * r_mean

            adv = (reward - rl_baseline).detach()
            rl_loss = -torch.mean(adv * logprob)

        loss_total = loss_sup + (LAMBDA_RL * rl_loss)

        optED.zero_grad(set_to_none=True)
        optPol.zero_grad(set_to_none=True)
        loss_total.backward()
        torch.nn.utils.clip_grad_norm_(list(Enc.parameters()) + list(Dec.parameters()) + list(Pol.parameters()), 5.0)
        optED.step()
        optPol.step()

        last = (loss_total.item(), loss_msg.item(), loss_imp.item(), loss_res.item(), float(rl_loss.item()), ch_name)

    return last

# -----------------------
# TRAIN
# -----------------------
ensure_dir(OUT_DIR)
train_tracker = cc_start("train_flexmark_rl", OUT_DIR)
t0 = time.time()

for ep in range(EPOCHS_WARMUP):
    last = train_one_epoch("warmup", crop_prob=0.0)
    print(f"[Warmup] {ep+1}/{EPOCHS_WARMUP} | loss={last[0]:.4f} "
          f"(msg={last[1]:.4f}, imp={last[2]:.4f}, res={last[3]:.4f}, rl={last[4]:.4f}, ch={last[5]})")

for ep in range(EPOCHS_ROBUST):
    crop_p = CROP_P_START + (CROP_P_END - CROP_P_START) * (ep / max(1, EPOCHS_ROBUST-1))
    last = train_one_epoch("robust", crop_prob=float(crop_p))
    print(f"[Robust] {ep+1}/{EPOCHS_ROBUST} | loss={last[0]:.4f} "
          f"(msg={last[1]:.4f}, imp={last[2]:.4f}, res={last[3]:.4f}, rl={last[4]:.4f}, ch={last[5]}) | crop_p={crop_p:.3f}")

train_seconds = time.time() - t0
train_co2 = cc_stop(train_tracker)

Enc_path = os.path.join(OUT_DIR, "flexmark_rl_Enc.pt")
Dec_path = os.path.join(OUT_DIR, "flexmark_rl_DecGrid.pt")
Pol_path = os.path.join(OUT_DIR, "flexmark_rl_Policy.pt")
torch.save(Enc.state_dict(), Enc_path)
torch.save(Dec.state_dict(), Dec_path)
torch.save(Pol.state_dict(), Pol_path)

model_file_mb = (os.path.getsize(Enc_path)+os.path.getsize(Dec_path)+os.path.getsize(Pol_path)) / (1024**2)
print("Saved:", Enc_path, Dec_path, Pol_path, f"| total {model_file_mb:.2f} MB")

# ============================================================
# EVAL: ATTACKS + Decision reliability (ROC/AUC + confusion@1%FPR) + plots
# Paired geometry for non-blind decode (resize/crop/screenshot)
# ============================================================
infer_tracker = cc_start("infer_flexmark_rl", OUT_DIR)
t0 = time.time()
Enc.eval(); Dec.eval(); Pol.eval()

cached = []
with torch.no_grad():
    for j, (C, P, bits) in enumerate(test_loader):
        if j >= min(EVAL_N, len(test_ds)):
            break
        C = C.to(DEVICE); P = P.to(DEVICE)
        mask, strength, _, _ = Pol.sample_actions(C, deterministic=True)
        r_raw = Enc(C, P)
        Y, _ = apply_controller_to_residual(C, r_raw, mask, strength, EPS, HW)

        cached.append((
            C[0].detach().cpu().permute(1,2,0).numpy(),
            Y[0].detach().cpu().permute(1,2,0).numpy(),
            bits[0].numpy().astype(np.int32)
        ))

attacks = list(ATTACKS.keys())

# Paired geometry helpers (match HiDDeN paired_geom idea)
def paired_resize_np(Y, C, scale=0.75):
    H, W, _ = Y.shape
    nh, nw = max(1, int(H*scale)), max(1, int(W*scale))
    Yp = np_to_pil(Y).resize((nw, nh), Image.BICUBIC).resize((W, H), Image.BICUBIC)
    Cp = np_to_pil(C).resize((nw, nh), Image.BICUBIC).resize((W, H), Image.BICUBIC)
    return pil_to_np(Yp), pil_to_np(Cp)

def paired_crop10_np(Y, C, crop_frac=0.10):
    H, W, _ = Y.shape
    ch, cw = max(1, int(H*(1-crop_frac))), max(1, int(W*(1-crop_frac)))
    y0 = np.random.randint(0, H - ch + 1)
    x0 = np.random.randint(0, W - cw + 1)
    Yc = Y[y0:y0+ch, x0:x0+cw, :]
    Cc = C[y0:y0+ch, x0:x0+cw, :]
    Yp = np_to_pil(Yc).resize((W, H), Image.BICUBIC)
    Cp = np_to_pil(Cc).resize((W, H), Image.BICUBIC)
    return pil_to_np(Yp), pil_to_np(Cp)

def paired_screenshot_np(Y, C, q1=35, q2=80):
    Y1 = attack_jpeg(Y, q1); C1 = attack_jpeg(C, q1)
    Y2, C2 = paired_resize_np(Y1, C1, 0.92)
    Y3 = attack_jpeg(Y2, q2); C3 = attack_jpeg(C2, q2)
    return Y3, C3

GEOM_KEYS = {"resize075", "crop10", "screenshot"}

# Robustness (STANDARD)
rows = []
for (C_np, Y_np, bits_np) in cached:
    psnr_c = psnr(Y_np, C_np)
    ssim_c = ssim_np(Y_np, C_np)
    lpips_c = lpips_score_np(Y_np, C_np, device=DEVICE) if (TORCH_OK and LPIPS_OK) else float("nan")

    rob = {}
    for atk_name, atk_fn in ATTACKS.items():
        if atk_name == "resize075":
            Ya, Ca = paired_resize_np(Y_np, C_np, 0.75)
        elif atk_name == "crop10":
            Ya, Ca = paired_crop10_np(Y_np, C_np, 0.10)
        elif atk_name == "screenshot":
            Ya, Ca = paired_screenshot_np(Y_np, C_np, 35, 80)
        else:
            Ya = atk_fn(Y_np)
            Ca = atk_fn(C_np) if atk_name.startswith("jpeg") else C_np

        Ya_t = torch.from_numpy(Ya).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
        Ca_t = torch.from_numpy(Ca).permute(2,0,1).unsqueeze(0).float().to(DEVICE)

        with torch.no_grad():
            probs = torch.sigmoid(Dec(Ya_t, Ca_t))[0].detach().cpu().numpy().reshape(-1)

        pred_bits = (probs >= 0.5).astype(np.int32)
        ber_bits  = float(np.mean(pred_bits != bits_np) * 100.0)
        bit_acc   = float(100.0 - ber_bits)

        tb = (bits_np * 2 - 1).astype(np.float32)
        pb = (pred_bits * 2 - 1).astype(np.float32)
        nc_bits = float(np.mean(tb * pb))

        rob[atk_name] = {"NC_bits": nc_bits, "BER": ber_bits, "bit_acc": bit_acc}

    rows.append({"psnr_cover": psnr_c, "ssim_cover": ssim_c, "lpips_cover": lpips_c, "robustness": rob})

# Decision reliability: correct vs wrong payload (STANDARD)
bits_pool = [b for (_,_,b) in cached]
bits_pool_wrong = bits_pool[1:] + bits_pool[:1]

det_scores_std = {a: {"pos": [], "neg": []} for a in attacks}
det_auc = {}
det_conf = {}

for atk_name, atk_fn in ATTACKS.items():
    pos_scores, neg_scores = [], []
    for idx, (C_np, Y_np, bits_np) in enumerate(cached):
        if atk_name == "resize075":
            Ya, Ca = paired_resize_np(Y_np, C_np, 0.75)
        elif atk_name == "crop10":
            Ya, Ca = paired_crop10_np(Y_np, C_np, 0.10)
        elif atk_name == "screenshot":
            Ya, Ca = paired_screenshot_np(Y_np, C_np, 35, 80)
        else:
            Ya = atk_fn(Y_np)
            Ca = atk_fn(C_np) if atk_name.startswith("jpeg") else C_np

        Ya_t = torch.from_numpy(Ya).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
        Ca_t = torch.from_numpy(Ca).permute(2,0,1).unsqueeze(0).float().to(DEVICE)

        with torch.no_grad():
            probs = torch.sigmoid(Dec(Ya_t, Ca_t))[0].detach().cpu().numpy().reshape(-1)

        pred_bits = (probs >= 0.5).astype(np.int32)

        ber_pos = float(np.mean(pred_bits != bits_np) * 100.0)
        s_pos = 1.0 - ber_pos/100.0

        wrong_bits = bits_pool_wrong[idx]
        ber_neg = float(np.mean(pred_bits != wrong_bits) * 100.0)
        s_neg = 1.0 - ber_neg/100.0

        pos_scores.append(s_pos)
        neg_scores.append(s_neg)

    det_scores_std[atk_name]["pos"] = pos_scores
    det_scores_std[atk_name]["neg"] = neg_scores

    y_true = np.array([1]*len(pos_scores) + [0]*len(neg_scores))
    y_score = np.array(pos_scores + neg_scores)

    A = float(roc_auc_score(y_true, y_score))
    cm = confusion_at_fpr(pos_scores, neg_scores, target_fpr=0.01)

    det_auc[atk_name] = A
    det_conf[atk_name] = cm

    # ROC plot per attack (STANDARD)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={A:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title(f"FLEXMark-RL Decision ROC (STANDARD) — {atk_name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"flexrl_decision_roc_standard_{atk_name}.png"), dpi=220)
    plt.close()

    plot_confusion_heatmap(
        cm,
        f"Decision Confusion @1%FPR (STANDARD) — {atk_name}",
        os.path.join(OUT_DIR, f"flexrl_decision_confusion_standard_{atk_name}.png")
    )

# Clean ROC “main”
if "clean" in attacks:
    pos = det_scores_std["clean"]["pos"]
    neg = det_scores_std["clean"]["neg"]
    y_true = np.array([1]*len(pos) + [0]*len(neg))
    y_score = np.array(pos + neg)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    A = float(roc_auc_score(y_true, y_score))

    plt.figure()
    plt.plot(fpr, tpr, label=f"Clean AUC={A:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title("FLEXMark-RL Decision ROC (clean): correct vs wrong payload")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "flexrl_decision_roc_clean.png"), dpi=260)
    plt.close()

infer_seconds = time.time() - t0
infer_co2 = cc_stop(infer_tracker)

# ============================================================
# AI laundering (separate) + robustness plots + decision after laundering
# ============================================================
pairs = []
for (C_np, Y_np, _) in cached[:150]:
    pairs.append((attack_jpeg(Y_np, 10), C_np))

launder_fn, launder_meta = train_ai_launderer_patchwise(
    pairs=pairs, out_dir=OUT_DIR, patch=64, patches_per_img=2, epochs=1, lr=1e-3, max_pairs=150, device=DEVICE
)

rows_launder = []
det_auc_L, det_conf_L = {}, {}
det_scores_lau = {a: {"pos": [], "neg": []} for a in attacks}

if launder_fn is not None:
    # robustness after laundering
    for (C_np, Y_np, bits_np) in cached:
        robL = {}
        for atk_name, atk_fn in ATTACKS.items():
            if atk_name == "resize075":
                Ya, Ca = paired_resize_np(Y_np, C_np, 0.75)
            elif atk_name == "crop10":
                Ya, Ca = paired_crop10_np(Y_np, C_np, 0.10)
            elif atk_name == "screenshot":
                Ya, Ca = paired_screenshot_np(Y_np, C_np, 35, 80)
            else:
                Ya = atk_fn(Y_np)
                Ca = atk_fn(C_np) if atk_name.startswith("jpeg") else C_np

            if atk_name != "clean":
                Ya = np.clip(launder_fn(Ya), 0, 1)

            Ya_t = torch.from_numpy(Ya).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
            Ca_t = torch.from_numpy(Ca).permute(2,0,1).unsqueeze(0).float().to(DEVICE)

            with torch.no_grad():
                probs = torch.sigmoid(Dec(Ya_t, Ca_t))[0].detach().cpu().numpy().reshape(-1)

            pred_bits = (probs >= 0.5).astype(np.int32)
            ber_bits  = float(np.mean(pred_bits != bits_np) * 100.0)
            bit_acc   = float(100.0 - ber_bits)

            tb = (bits_np * 2 - 1).astype(np.float32)
            pb = (pred_bits * 2 - 1).astype(np.float32)
            nc_bits = float(np.mean(tb * pb))

            robL[atk_name] = {"NC_bits": nc_bits, "BER": ber_bits, "bit_acc": bit_acc}

        rows_launder.append({"robustness": robL})

    # decision after laundering
    for atk_name, atk_fn in ATTACKS.items():
        pos_scores, neg_scores = [], []
        for idx, (C_np, Y_np, bits_np) in enumerate(cached):
            if atk_name == "resize075":
                Ya, Ca = paired_resize_np(Y_np, C_np, 0.75)
            elif atk_name == "crop10":
                Ya, Ca = paired_crop10_np(Y_np, C_np, 0.10)
            elif atk_name == "screenshot":
                Ya, Ca = paired_screenshot_np(Y_np, C_np, 35, 80)
            else:
                Ya = atk_fn(Y_np)
                Ca = atk_fn(C_np) if atk_name.startswith("jpeg") else C_np

            if atk_name != "clean":
                Ya = np.clip(launder_fn(Ya), 0, 1)

            Ya_t = torch.from_numpy(Ya).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
            Ca_t = torch.from_numpy(Ca).permute(2,0,1).unsqueeze(0).float().to(DEVICE)

            with torch.no_grad():
                probs = torch.sigmoid(Dec(Ya_t, Ca_t))[0].detach().cpu().numpy().reshape(-1)

            pred_bits = (probs >= 0.5).astype(np.int32)

            ber_pos = float(np.mean(pred_bits != bits_np) * 100.0)
            s_pos = 1.0 - ber_pos/100.0

            wrong_bits = bits_pool_wrong[idx]
            ber_neg = float(np.mean(pred_bits != wrong_bits) * 100.0)
            s_neg = 1.0 - ber_neg/100.0

            pos_scores.append(s_pos)
            neg_scores.append(s_neg)

        det_scores_lau[atk_name]["pos"] = pos_scores
        det_scores_lau[atk_name]["neg"] = neg_scores

        y_true = np.array([1]*len(pos_scores) + [0]*len(neg_scores))
        y_score = np.array(pos_scores + neg_scores)

        A = float(roc_auc_score(y_true, y_score))
        cm = confusion_at_fpr(pos_scores, neg_scores, target_fpr=0.01)

        det_auc_L[atk_name] = A
        det_conf_L[atk_name] = cm

        fpr, tpr, _ = roc_curve(y_true, y_score)
        plt.figure()
        plt.plot(fpr, tpr, label=f"AUC={A:.3f}")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("FPR"); plt.ylabel("TPR")
        plt.title(f"FLEXMark-RL Decision ROC (LAUNDERED) — {atk_name}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f"flexrl_decision_roc_laundered_{atk_name}.png"), dpi=220)
        plt.close()

        plot_confusion_heatmap(
            cm,
            f"Decision Confusion @1%FPR (LAUNDERED) — {atk_name}",
            os.path.join(OUT_DIR, f"flexrl_decision_confusion_laundered_{atk_name}.png")
        )

# ============================================================
# Summary
# ============================================================
summary = {
    "model": "FLEXMark-RL_controller_datasetpayload_pairedgeom",
    "payload_bits": PAYLOAD_BITS,
    "hw": HW,
    "eps": EPS,
    "grid": [gh, gw],
    "n_train": len(train_ds),
    "n_test_eval": len(rows),

    "mean_psnr_cover": float(np.mean([r["psnr_cover"] for r in rows])),
    "mean_ssim_cover": float(np.mean([r["ssim_cover"] for r in rows])),
    "mean_lpips_cover": float(np.nanmean([r["lpips_cover"] for r in rows])),

    "mean_nc_bits":  {a: float(np.mean([r["robustness"][a]["NC_bits"] for r in rows])) for a in attacks},
    "mean_ber":      {a: float(np.mean([r["robustness"][a]["BER"]     for r in rows])) for a in attacks},
    "mean_bit_acc":  {a: float(np.mean([r["robustness"][a]["bit_acc"] for r in rows])) for a in attacks},

    "decision_auc_by_attack": det_auc,
    "decision_confusion_1pctfpr_by_attack": det_conf,

    "train_seconds": float(train_seconds),
    "infer_seconds": float(infer_seconds),
    "train_co2_kg": float(train_co2) if train_co2 is not None else None,
    "infer_co2_kg": float(infer_co2) if infer_co2 is not None else None,
    "model_file_mb": float(model_file_mb),

    "ai_laundering_meta": launder_meta,

    "train_recipe": {
        "warmup_epochs": EPOCHS_WARMUP,
        "robust_epochs": EPOCHS_ROBUST,
        "lambda_msg": LAMBDA_MSG,
        "lambda_imp": LAMBDA_IMP,
        "lambda_res": LAMBDA_RES,
        "lambda_rl": LAMBDA_RL,
        "psnr_floor": PSNR_FLOOR,
        "jpeg_train_p": JPEG_TRAIN_P,
        "jpeg_quals": JPEG_QUALS,
        "crop_p_start": CROP_P_START,
        "crop_p_end": CROP_P_END,
        "decoder": "nonblind_grid(ghxgw)",
        "controller": "RL(mask_grid + strength_level)",
        "reward": "soft_correctness - 0.20*psnr_pen - 0.10*budget_pen",
        "note": "Main robustness/decision = attacks only; laundering is separate adaptive removal attempt; paired geom transforms."
    }
}

if len(rows_launder) > 0:
    summary["mean_nc_bits_laundered"] = {a: float(np.mean([r["robustness"][a]["NC_bits"] for r in rows_launder])) for a in attacks}
    summary["mean_ber_laundered"]     = {a: float(np.mean([r["robustness"][a]["BER"]     for r in rows_launder])) for a in attacks}
    summary["mean_bit_acc_laundered"] = {a: float(np.mean([r["robustness"][a]["bit_acc"] for r in rows_launder])) for a in attacks}
    summary["decision_auc_by_attack_laundered"] = det_auc_L
    summary["decision_confusion_1pctfpr_by_attack_laundered"] = det_conf_L
else:
    summary["mean_nc_bits_laundered"] = None
    summary["mean_ber_laundered"]     = None
    summary["mean_bit_acc_laundered"] = None
    summary["decision_auc_by_attack_laundered"] = None
    summary["decision_confusion_1pctfpr_by_attack_laundered"] = None

# ============================================================
# Plots: robustness curves (standard + laundered + delta)
# ============================================================
plot_attack_metric_line(attacks, summary["mean_nc_bits"],
                        "FLEXMark-RL mean NC_bits under attacks (standard)",
                        os.path.join(OUT_DIR, "flexrl_NC_attacks.png"),
                        ylabel="NC_bits", ylim=(-1, 1))

plot_attack_metric_line(attacks, summary["mean_ber"],
                        "FLEXMark-RL mean BER (%) under attacks (standard)",
                        os.path.join(OUT_DIR, "flexrl_BER_attacks.png"),
                        ylabel="BER (%)", ylim=(0, 100))

plot_attack_metric_line(attacks, summary["mean_bit_acc"],
                        "FLEXMark-RL mean Bit Accuracy (%) under attacks (standard)",
                        os.path.join(OUT_DIR, "flexrl_bitacc_attacks.png"),
                        ylabel="Bit Accuracy (%)", ylim=(0, 105))

if summary["mean_bit_acc_laundered"] is not None:
    plot_attack_metric_line(attacks, summary["mean_nc_bits_laundered"],
                            "FLEXMark-RL mean NC_bits under attacks (after laundering)",
                            os.path.join(OUT_DIR, "flexrl_NC_attacks_laundered.png"),
                            ylabel="NC_bits", ylim=(-1, 1))

    plot_attack_metric_line(attacks, summary["mean_ber_laundered"],
                            "FLEXMark-RL mean BER (%) under attacks (after laundering)",
                            os.path.join(OUT_DIR, "flexrl_BER_attacks_laundered.png"),
                            ylabel="BER (%)", ylim=(0, 100))

    plot_attack_metric_line(attacks, summary["mean_bit_acc_laundered"],
                            "FLEXMark-RL mean Bit Accuracy (%) under attacks (after laundering)",
                            os.path.join(OUT_DIR, "flexrl_bitacc_attacks_laundered.png"),
                            ylabel="Bit Accuracy (%)", ylim=(0, 105))

    bit_std = np.array([summary["mean_bit_acc"][a] for a in attacks], dtype=float)
    bit_lau = np.array([summary["mean_bit_acc_laundered"][a] for a in attacks], dtype=float)
    delta = bit_std - bit_lau

    plt.figure(figsize=(12,4))
    plt.bar(attacks, delta)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Bit-Accuracy Drop (pp)")
    plt.title("AI Laundering Impact: Δ(BitAcc) = Standard − Laundered (FLEXMark-RL)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "flexrl_delta_bitacc_standard_minus_laundered.png"), dpi=220)
    plt.close()

# Save summary
save_json(summary, os.path.join(OUT_DIR, "flexmark_rl_summary.json"))
print(json.dumps(summary, indent=2))
print("✅ Outputs in:", OUT_DIR)

FLEXMark-RL curriculum: | warmup: 2 | robust: 6 | grid: (8, 16) | EPS: 0.02 | JPEG_P: 0.5 | JPEG_QUALS: [10, 30, 50] | PSNR_FLOOR: 35.0 | DEVICE: cuda
Using n_train = 5000 | n_test = 1000 | eval = 1000


[codecarbon WARNING @ 01:48:40] Multiple instances of codecarbon are allowed to run at the same time.


[Warmup] 1/2 | loss=0.1963 (msg=0.0162, imp=0.0132, res=0.6678, rl=0.0000, ch=clean)
[Warmup] 2/2 | loss=0.0602 (msg=0.0042, imp=0.0070, res=0.3563, rl=0.0000, ch=clean)
[Robust] 1/6 | loss=2.2116 (msg=0.1728, imp=0.0082, res=0.4225, rl=1.8462, ch=jpeg) | crop_p=0.050
[Robust] 2/6 | loss=2.1179 (msg=0.0522, imp=0.0098, res=0.5015, rl=6.2795, ch=resize075) | crop_p=0.090
[Robust] 3/6 | loss=2.3757 (msg=0.0556, imp=0.0080, res=0.4255, rl=7.1901, ch=clean) | crop_p=0.130
[Robust] 4/6 | loss=1.7003 (msg=0.0376, imp=0.0108, res=0.5523, rl=5.1842, ch=noise) | crop_p=0.170
[Robust] 5/6 | loss=1.1409 (msg=0.2869, imp=0.0113, res=0.5771, rl=-7.0314, ch=jpeg) | crop_p=0.210
[Robust] 6/6 | loss=1.4041 (msg=0.0167, imp=0.0112, res=0.5742, rl=4.8285, ch=clean) | crop_p=0.250
Saved: /kaggle/working/flexmark_out_bits128/flexmark_rl_Enc.pt /kaggle/working/flexmark_out_bits128/flexmark_rl_DecGrid.pt /kaggle/working/flexmark_out_bits128/flexmark_rl_Policy.pt | total 0.84 MB


/tmp/ipykernel_55/3889498849.py:689: RuntimeWarning: Mean of empty slice
  "mean_lpips_cover": float(np.nanmean([r["lpips_cover"] for r in rows])),


{
  "model": "FLEXMark-RL_controller_datasetpayload_pairedgeom",
  "payload_bits": 128,
  "hw": 128,
  "eps": 0.02,
  "grid": [
    8,
    16
  ],
  "n_train": 5000,
  "n_test_eval": 1000,
  "mean_psnr_cover": 36.914763626626815,
  "mean_ssim_cover": 0.9728525645136833,
  "mean_lpips_cover": NaN,
  "mean_nc_bits": {
    "clean": 0.999984375,
    "jpeg10": 0.7811875,
    "jpeg30": 0.9941875,
    "jpeg50": 0.99928125,
    "jpeg70": 0.999859375,
    "jpeg90": 0.99996875,
    "resize075": 1.0,
    "crop10": 0.404171875,
    "blur": 0.97084375,
    "sharpen": 0.999796875,
    "gamma12": 0.22096875,
    "screenshot": 0.996546875,
    "launder_only": 0.999984375
  },
  "mean_ber": {
    "clean": 0.00078125,
    "jpeg10": 10.940625,
    "jpeg30": 0.290625,
    "jpeg50": 0.0359375,
    "jpeg70": 0.00703125,
    "jpeg90": 0.0015625,
    "resize075": 0.0,
    "crop10": 29.79140625,
    "blur": 1.4578125,
    "sharpen": 0.01015625,
    "gamma12": 38.9515625,
    "screenshot": 0.17265625,
    "laun

### 256 bits RCRA-RL

In [6]:
# ============================================================
# CELL 4 — RCRA-RL (Controller) Model Classes
# Reviewer-aligned:
# - Grid decoder (8×16 for 128 bits; 16×16 for 256 bits)
# - Non-blind decode: inputs [Y_noised, C_noised]
# - RL is a CONTROLLER: chooses (mask grid + strength level)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------------------
# Small conv block
# ---------------------------
class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, s, p),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

# ============================================================
# RCRA Encoder (supervised)
# Input: concat([C (3ch), P (1ch)]) -> residual r_raw (3ch)
# ============================================================
class FLEXEncoder(nn.Module):
    def __init__(self, payload_ch=1, width=64):
        super().__init__()
        in_ch = 3 + payload_ch
        self.net = nn.Sequential(
            ConvBNReLU(in_ch, width),
            ConvBNReLU(width, width),
            ConvBNReLU(width, width),
            nn.Conv2d(width, 3, 1, 1, 0),
        )

    def forward(self, C, P):
        x = torch.cat([C, P], dim=1)
        r_raw = self.net(x)                    # unconstrained residual
        return r_raw

# ============================================================
# RCRA Grid Decoder (supervised)
# Output logits_grid: (B, gh, gw)
# ============================================================
class FLEXDecoderNonBlindGrid(nn.Module):
    def __init__(self, gh=8, gw=16, width=64):
        super().__init__()
        self.gh, self.gw = int(gh), int(gw)
        self.f = nn.Sequential(
            ConvBNReLU(6, width, s=1),
            ConvBNReLU(width, width, s=2),
            ConvBNReLU(width, width, s=2),
            ConvBNReLU(width, width, s=2),
            nn.Conv2d(width, 1, 1, 1, 0),
        )

    def forward(self, Y_noised, C_noised):
        x = torch.cat([Y_noised, C_noised], dim=1)
        z = self.f(x)  # (B,1,h,w)
        z = F.interpolate(z, size=(self.gh, self.gw), mode="bilinear", align_corners=False)
        return z.squeeze(1)  # (B,gh,gw)

# ============================================================
# RL Controller
# - action 1: mask grid over payload cells (Bernoulli per cell)
# - action 2: strength level (Categorical)
#
# IMPORTANT: RL is a controller, not the embedder.
# ============================================================
class FLEXRLController(nn.Module):
    def __init__(self, gh=8, gw=16, width=32, strength_levels=(0.25, 0.50, 0.75, 1.00)):
        super().__init__()
        self.gh, self.gw = int(gh), int(gw)
        self.strength_levels = list(map(float, strength_levels))
        nS = len(self.strength_levels)

        # small feature extractor on cover (fast)
        self.feat = nn.Sequential(
            ConvBNReLU(3, width, s=2),
            ConvBNReLU(width, width, s=2),
            ConvBNReLU(width, width, s=2),
        )

        # mask head -> logits over (gh, gw)
        self.mask_head = nn.Conv2d(width, 1, 1, 1, 0)

        # strength head -> logits over strength levels
        self.str_head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Linear(width, nS)
        )

    def forward(self, C):
        """
        Returns:
          mask_logits: (B, gh, gw)
          str_logits:  (B, nS)
        """
        f = self.feat(C)  # (B,width,h,w)
        m = self.mask_head(f)  # (B,1,h,w)
        m = F.interpolate(m, size=(self.gh, self.gw), mode="bilinear", align_corners=False).squeeze(1)  # (B,gh,gw)
        s = self.str_head(f)  # (B,nS)
        return m, s

    def sample_actions(self, C, deterministic=False):
        """
        Returns:
          mask: (B,gh,gw) float in {0,1} (or {0,1} det)
          strength: (B,) float from strength_levels
          logprob: (B,) sum log-prob of mask+strength (for REINFORCE)
          info: dict with indices
        """
        mask_logits, str_logits = self.forward(C)

        # strength categorical
        if deterministic:
            s_idx = torch.argmax(str_logits, dim=1)
            s_logprob = torch.zeros((C.size(0),), device=C.device)
        else:
            dist_s = torch.distributions.Categorical(logits=str_logits)
            s_idx = dist_s.sample()
            s_logprob = dist_s.log_prob(s_idx)

        strength = torch.tensor(self.strength_levels, device=C.device)[s_idx]  # (B,)

        # mask Bernoulli per cell
        if deterministic:
            mask = (torch.sigmoid(mask_logits) >= 0.5).float()
            m_logprob = torch.zeros((C.size(0),), device=C.device)
        else:
            dist_m = torch.distributions.Bernoulli(logits=mask_logits)
            mask = dist_m.sample()
            # sum logprob across grid for each sample
            m_logprob = dist_m.log_prob(mask).view(C.size(0), -1).sum(dim=1)

        logprob = m_logprob + s_logprob
        return mask, strength, logprob, {"s_idx": s_idx}

# ============================================================
# Helper: apply controller to residual
# r = tanh(r_raw)*eps
# r_ctrl = r * upsample(mask) * strength
# ============================================================
def apply_controller_to_residual(C, r_raw, mask_grid, strength, eps, hw):
    """
    C: (B,3,H,W)
    r_raw: (B,3,H,W)
    mask_grid: (B,gh,gw) 0/1
    strength: (B,) scalar
    Returns: Y, r_ctrl
    """
    B, _, H, W = C.shape
    assert H == hw and W == hw

    r = torch.tanh(r_raw) * float(eps)  # bounded residual

    # upsample mask grid to image resolution
    m = mask_grid.unsqueeze(1)  # (B,1,gh,gw)
    m_up = F.interpolate(m, size=(hw, hw), mode="nearest")  # (B,1,H,W)

    s = strength.view(B,1,1,1)  # broadcast
    r_ctrl = torch.clamp(r * m_up * s, -float(eps), float(eps))
    Y = torch.clamp(C + r_ctrl, 0.0, 1.0)
    return Y, r_ctrl

print("CELL 4 ready: FLEXEncoder + FLEXDecoderNonBlindGrid + FLEXRLController.")


CELL 4 ready: FLEXEncoder + FLEXDecoderNonBlindGrid + FLEXRLController.


In [7]:
# ============================================================
# CELL 5 — RCRA-RL (Controller): Curriculum Train + Eval (FINAL + visuals)
# CLEAN corrected version:
# Dataset-only payload from mark PNG (matches cGAN/HiDDeN protocol)
# RL reward uses SMOOTH (soft) correctness (REINFORCE-friendly)
# Paired geometric transforms for non-blind decoding (resize/crop/screenshot)
# Same reporting: PSNR/SSIM + NC/BER/bit_acc + decision ROC/AUC + confusion@1%FPR
# Separate AI laundering evaluation (optional)
# ============================================================

import os, time, random, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from io import BytesIO

import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------
# Utility: save json
# -----------------------
def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

# -----------------------
# Utility: confusion @ target FPR (with actual rates)
# -----------------------
def confusion_at_fpr(scores_pos, scores_neg, target_fpr=0.01):
    scores_pos = np.asarray(scores_pos, dtype=np.float64)
    scores_neg = np.asarray(scores_neg, dtype=np.float64)

    thr = float(np.quantile(scores_neg, 1.0 - target_fpr))

    TP = int(np.sum(scores_pos >= thr))
    FN = int(np.sum(scores_pos <  thr))
    FP = int(np.sum(scores_neg >= thr))
    TN = int(np.sum(scores_neg <  thr))

    fpr_actual = FP / max(1, (FP + TN))
    tpr_actual = TP / max(1, (TP + FN))

    return {
        "thr": thr,
        "TP": TP, "FP": FP, "TN": TN, "FN": FN,
        "fpr_actual": float(fpr_actual),
        "tpr_actual": float(tpr_actual),
    }

def plot_confusion_heatmap(cm_dict, title, out_path):
    mat = np.array([[cm_dict["TN"], cm_dict["FP"]],
                    [cm_dict["FN"], cm_dict["TP"]]], dtype=np.int64)
    plt.figure(figsize=(4.6, 4.1))
    plt.imshow(mat)
    plt.xticks([0,1], ["Pred 0", "Pred 1"])
    plt.yticks([0,1], ["True 0", "True 1"])
    for (r,c), v in np.ndenumerate(mat):
        plt.text(c, r, str(v), ha="center", va="center", fontsize=11)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=240)
    plt.close()

def plot_attack_metric_line(attacks, series_dict, title, out_path, ylabel=None, ylim=None):
    x = np.arange(len(attacks))
    y = [series_dict[a] for a in attacks]
    plt.figure(figsize=(12,4))
    plt.plot(x, y, marker="o")
    plt.xticks(x, attacks, rotation=45, ha="right")
    if ylabel is not None:
        plt.ylabel(ylabel)
    if ylim is not None:
        plt.ylim(*ylim)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=240)
    plt.close()

# -----------------------
# Real JPEG forward helper (per-batch, per-image)
# -----------------------
def _torch_bchw_to_np_bhwc01(x):
    return x.detach().clamp(0,1).cpu().permute(0,2,3,1).numpy()

def _np_bhwc01_to_torch_bchw(x_np, device):
    return torch.from_numpy(x_np).float().permute(0,3,1,2).to(device).clamp(0,1)

def jpeg_batch_torch(Y, quality=50, device=None):
    if device is None:
        device = Y.device
    Y_np = _torch_bchw_to_np_bhwc01(Y)
    out = []
    for i in range(Y_np.shape[0]):
        pil = Image.fromarray((Y_np[i] * 255).astype(np.uint8))
        buf = BytesIO()
        pil.save(buf, format="JPEG", quality=int(quality))
        buf.seek(0)
        rec = Image.open(buf).convert("RGB")
        out.append(np.asarray(rec).astype(np.float32) / 255.0)
    out_np = np.stack(out, axis=0)
    return _np_bhwc01_to_torch_bchw(out_np, device)

# -----------------------
# Config (uses Common Cell 2)
# -----------------------
HW = int(HW)
EPS = float(EPS)
PAYLOAD_BITS = int(PAYLOAD_BITS)
assert PAYLOAD_BITS in (128,256)
gh, gw = (8,16) if PAYLOAD_BITS == 128 else (16,16)

BATCH = 16
LR_ED = 2e-4
LR_POL = 2e-4

LAMBDA_MSG = 10.0
LAMBDA_IMP = 0.10
LAMBDA_RES = 0.05

LAMBDA_RL = 0.25
PSNR_FLOOR = 35.0

EPOCHS_WARMUP = 2
EPOCHS_ROBUST = 6

JPEG_TRAIN_P = 0.50
JPEG_QUALS = [10, 30, 50]

CROP_P_START = 0.05
CROP_P_END   = 0.25

print("FLEXMark-RL curriculum:",
      "| warmup:", EPOCHS_WARMUP,
      "| robust:", EPOCHS_ROBUST,
      "| grid:", (gh,gw),
      "| EPS:", EPS,
      "| JPEG_P:", JPEG_TRAIN_P,
      "| JPEG_QUALS:", JPEG_QUALS,
      "| PSNR_FLOOR:", PSNR_FLOOR,
      "| DEVICE:", DEVICE)

# ============================================================
# Dataset: DATASET-ONLY payload from mark PNG (paired cover+mark)
# Common Cell 2 provides: make_payload_for_model(mark_path, hw, bitsB, mode)
# Returns: (P_np (H,W,1), Wtrue_np (gh,gw,1), meta)
# ============================================================
class CoverMarkRLDataset(Dataset):
    def __init__(self, cover_paths, mark_paths, hw, payload_bits):
        assert len(cover_paths) == len(mark_paths), "cover/mark list mismatch"
        self.cover_paths = cover_paths
        self.mark_paths  = mark_paths
        self.hw = int(hw)
        self.payload_bits = int(payload_bits)

    def __len__(self):
        return len(self.cover_paths)

    def __getitem__(self, idx):
        C = read_rgb(self.cover_paths[idx], self.hw)

        # dataset-only payload derived deterministically from mark PNG
        P_np, Wtrue_np, _ = make_payload_for_model(
            self.mark_paths[idx], self.hw, self.payload_bits, mode="1ch"
        )
        bits_np = (Wtrue_np[..., 0] >= 0.5).astype(np.int32).reshape(-1)  # (payload_bits,)

        C_t = torch.from_numpy(C).permute(2,0,1).float()          # (3,H,W)
        P_t = torch.from_numpy(P_np).permute(2,0,1).float()       # (1,H,W)
        bits_t = torch.from_numpy(bits_np.astype(np.float32))     # (payload_bits,)
        return C_t, P_t, bits_t

train_ds = CoverMarkRLDataset(trC[:n_train], trW[:n_train], HW, PAYLOAD_BITS)
test_ds  = CoverMarkRLDataset(teC[:n_test],  teW[:n_test],  HW, PAYLOAD_BITS)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False, num_workers=2)

print(f"Using n_train = {len(train_ds)} | n_test = {len(test_ds)} | eval = {min(EVAL_N, len(test_ds))}")

# -----------------------
# Instantiate models (from Cell 4)
# -----------------------
Enc = FLEXEncoder(payload_ch=1, width=64).to(DEVICE)
Dec = FLEXDecoderNonBlindGrid(gh=gh, gw=gw, width=64).to(DEVICE)
Pol = FLEXRLController(gh=gh, gw=gw, width=32, strength_levels=(0.25, 0.50, 0.75, 1.00)).to(DEVICE)

optED  = torch.optim.Adam(list(Enc.parameters()) + list(Dec.parameters()), lr=LR_ED, betas=(0.5,0.999))
optPol = torch.optim.Adam(Pol.parameters(), lr=LR_POL, betas=(0.5,0.999))

bce_logits = nn.BCEWithLogitsLoss()
l1 = nn.L1Loss()

# -----------------------
# Differentiable-ish attacks for training (torch)
# -----------------------
def t_resize(x, scale=0.75):
    B,C,H,W = x.shape
    nh, nw = max(4,int(H*scale)), max(4,int(W*scale))
    y = F.interpolate(x, size=(nh,nw), mode="bilinear", align_corners=False)
    y = F.interpolate(y, size=(H,W), mode="bilinear", align_corners=False)
    return y

def t_crop_resize(x, crop_frac=0.10):
    B,C,H,W = x.shape
    ch, cw = max(4,int(H*(1-crop_frac))), max(4,int(W*(1-crop_frac)))
    y0 = torch.randint(0, H-ch+1, (1,), device=x.device).item()
    x0 = torch.randint(0, W-cw+1, (1,), device=x.device).item()
    crop = x[:, :, y0:y0+ch, x0:x0+cw]
    y = F.interpolate(crop, size=(H,W), mode="bilinear", align_corners=False)
    return y

def t_blur(x):
    k=3; pad=1
    w = torch.ones((x.shape[1],1,k,k), device=x.device) / (k*k)
    return F.conv2d(x, w, padding=pad, groups=x.shape[1])

def t_sharpen(x):
    blur = t_blur(x)
    alpha = 0.7
    return torch.clamp(x + alpha*(x-blur), 0, 1)

def t_gamma(x, gamma=1.2):
    return torch.clamp(x,0,1) ** gamma

def t_noise(x, sigma=0.01):
    return torch.clamp(x + sigma*torch.randn_like(x), 0, 1)

def sample_train_channel(stage, crop_prob=0.10):
    if stage == "warmup":
        return ("clean", lambda z: z, None)

    if random.random() < JPEG_TRAIN_P:
        return ("jpeg", None, random.choice(JPEG_QUALS))

    if random.random() < crop_prob:
        return ("crop10", lambda z: t_crop_resize(z, 0.10), None)

    choices = [
        ("clean",     lambda z: z),
        ("resize075", lambda z: t_resize(z, 0.75)),
        ("blur",      lambda z: t_blur(z)),
        ("sharpen",   lambda z: t_sharpen(z)),
        ("gamma12",   lambda z: t_gamma(z, 1.2)),
        ("noise",     lambda z: t_noise(z, 0.01)),
    ]
    name, fn = random.choice(choices)
    return (name, fn, None)

# -----------------------
# Reward utilities
# -----------------------
def psnr_torch(a, b):
    mse = torch.mean((a-b)**2, dim=(1,2,3)).clamp_min(1e-12)
    return 10.0 * torch.log10(1.0 / mse)

rl_baseline = 0.0
BASELINE_MOM = 0.95


def train_one_epoch(stage, crop_prob):
    global rl_baseline
    Enc.train(); Dec.train(); Pol.train()

    last = None
    for (C, P, bits) in train_loader:
        C = C.to(DEVICE, non_blocking=True)
        P = P.to(DEVICE, non_blocking=True)
        bits = bits.to(DEVICE, non_blocking=True)

        # Controller: warmup uses full mask + full strength
        if stage == "warmup":
            mask = torch.ones((C.size(0), gh, gw), device=DEVICE)
            strength = torch.ones((C.size(0),), device=DEVICE)
            logprob = torch.zeros((C.size(0),), device=DEVICE)
        else:
            mask, strength, logprob, _ = Pol.sample_actions(C, deterministic=False)

        # Supervised embedder
        r_raw = Enc(C, P)
        Y, r_ctrl = apply_controller_to_residual(C, r_raw, mask, strength, EPS, HW)

        # Train channel (attacks)
        ch_name, ch_fn, ch_q = sample_train_channel(stage, crop_prob=crop_prob)

        if ch_name == "jpeg":
            Y_noised = jpeg_batch_torch(Y, quality=ch_q, device=DEVICE)
            C_noised = jpeg_batch_torch(C, quality=ch_q, device=DEVICE)
        else:
            Y_noised = ch_fn(Y) if ch_fn is not None else Y
            if ch_name in ("resize075","crop10"):
                C_noised = ch_fn(C)
            else:
                C_noised = C

        bits_grid = bits.view(-1, gh, gw)
        logits_grid = Dec(Y_noised, C_noised)

        loss_msg = bce_logits(logits_grid, bits_grid)
        loss_imp = l1(Y, C)
        loss_res = torch.mean(torch.abs(r_ctrl)) / (EPS + 1e-8)

        loss_sup = (LAMBDA_MSG * loss_msg +
                    LAMBDA_IMP * loss_imp +
                    LAMBDA_RES * loss_res)

        # RL loss (REINFORCE) with SMOOTH reward
        rl_loss = torch.tensor(0.0, device=DEVICE)
        if stage != "warmup":
            with torch.no_grad():
                # smooth correctness in [0,1]
                p = torch.sigmoid(logits_grid).view(C.size(0), -1)
                t = bits.view(C.size(0), -1)
                soft_acc = torch.mean(t * p + (1 - t) * (1 - p), dim=1)  # (B,)

                psnrC = psnr_torch(Y, C)
                psnr_pen = torch.clamp(PSNR_FLOOR - psnrC, min=0.0) / 10.0
                budget_pen = torch.mean(torch.abs(r_ctrl), dim=(1,2,3)) / (EPS + 1e-8)

                reward = (1.0 * soft_acc) - (0.20 * psnr_pen) - (0.10 * budget_pen)

                r_mean = reward.mean().item()
                rl_baseline = BASELINE_MOM * rl_baseline + (1-BASELINE_MOM) * r_mean

            adv = (reward - rl_baseline).detach()
            rl_loss = -torch.mean(adv * logprob)

        loss_total = loss_sup + (LAMBDA_RL * rl_loss)

        optED.zero_grad(set_to_none=True)
        optPol.zero_grad(set_to_none=True)
        loss_total.backward()
        torch.nn.utils.clip_grad_norm_(list(Enc.parameters()) + list(Dec.parameters()) + list(Pol.parameters()), 5.0)
        optED.step()
        optPol.step()

        last = (loss_total.item(), loss_msg.item(), loss_imp.item(), loss_res.item(), float(rl_loss.item()), ch_name)

    return last

# -----------------------
# TRAIN (curriculum)
# -----------------------
ensure_dir(OUT_DIR)
train_tracker = cc_start("train_flexmark_rl", OUT_DIR)
t0 = time.time()

for ep in range(EPOCHS_WARMUP):
    last = train_one_epoch("warmup", crop_prob=0.0)
    print(f"[Warmup] {ep+1}/{EPOCHS_WARMUP} | loss={last[0]:.4f} "
          f"(msg={last[1]:.4f}, imp={last[2]:.4f}, res={last[3]:.4f}, rl={last[4]:.4f}, ch={last[5]})")

for ep in range(EPOCHS_ROBUST):
    crop_p = CROP_P_START + (CROP_P_END - CROP_P_START) * (ep / max(1, EPOCHS_ROBUST-1))
    last = train_one_epoch("robust", crop_prob=float(crop_p))
    print(f"[Robust] {ep+1}/{EPOCHS_ROBUST} | loss={last[0]:.4f} "
          f"(msg={last[1]:.4f}, imp={last[2]:.4f}, res={last[3]:.4f}, rl={last[4]:.4f}, ch={last[5]}) | crop_p={crop_p:.3f}")

train_seconds = time.time() - t0
train_co2 = cc_stop(train_tracker)

Enc_path = os.path.join(OUT_DIR, "flexmark_rl_Enc.pt")
Dec_path = os.path.join(OUT_DIR, "flexmark_rl_DecGrid.pt")
Pol_path = os.path.join(OUT_DIR, "flexmark_rl_Policy.pt")
torch.save(Enc.state_dict(), Enc_path)
torch.save(Dec.state_dict(), Dec_path)
torch.save(Pol.state_dict(), Pol_path)

model_file_mb = (os.path.getsize(Enc_path)+os.path.getsize(Dec_path)+os.path.getsize(Pol_path)) / (1024**2)
print("Saved:", Enc_path, Dec_path, Pol_path, f"| total {model_file_mb:.2f} MB")

# ============================================================
# EVAL: ATTACKS + Decision reliability (ROC/AUC + confusion@1%FPR) + plots
# Paired geometry for non-blind decode (resize/crop/screenshot)
# ============================================================
infer_tracker = cc_start("infer_flexmark_rl", OUT_DIR)
t0 = time.time()
Enc.eval(); Dec.eval(); Pol.eval()

cached = []
with torch.no_grad():
    for j, (C, P, bits) in enumerate(test_loader):
        if j >= min(EVAL_N, len(test_ds)):
            break
        C = C.to(DEVICE); P = P.to(DEVICE)
        mask, strength, _, _ = Pol.sample_actions(C, deterministic=True)
        r_raw = Enc(C, P)
        Y, _ = apply_controller_to_residual(C, r_raw, mask, strength, EPS, HW)

        cached.append((
            C[0].detach().cpu().permute(1,2,0).numpy(),
            Y[0].detach().cpu().permute(1,2,0).numpy(),
            bits[0].numpy().astype(np.int32)
        ))

attacks = list(ATTACKS.keys())

# Paired geometry helpers (match HiDDeN paired_geom idea)
def paired_resize_np(Y, C, scale=0.75):
    H, W, _ = Y.shape
    nh, nw = max(1, int(H*scale)), max(1, int(W*scale))
    Yp = np_to_pil(Y).resize((nw, nh), Image.BICUBIC).resize((W, H), Image.BICUBIC)
    Cp = np_to_pil(C).resize((nw, nh), Image.BICUBIC).resize((W, H), Image.BICUBIC)
    return pil_to_np(Yp), pil_to_np(Cp)

def paired_crop10_np(Y, C, crop_frac=0.10):
    H, W, _ = Y.shape
    ch, cw = max(1, int(H*(1-crop_frac))), max(1, int(W*(1-crop_frac)))
    y0 = np.random.randint(0, H - ch + 1)
    x0 = np.random.randint(0, W - cw + 1)
    Yc = Y[y0:y0+ch, x0:x0+cw, :]
    Cc = C[y0:y0+ch, x0:x0+cw, :]
    Yp = np_to_pil(Yc).resize((W, H), Image.BICUBIC)
    Cp = np_to_pil(Cc).resize((W, H), Image.BICUBIC)
    return pil_to_np(Yp), pil_to_np(Cp)

def paired_screenshot_np(Y, C, q1=35, q2=80):
    Y1 = attack_jpeg(Y, q1); C1 = attack_jpeg(C, q1)
    Y2, C2 = paired_resize_np(Y1, C1, 0.92)
    Y3 = attack_jpeg(Y2, q2); C3 = attack_jpeg(C2, q2)
    return Y3, C3

GEOM_KEYS = {"resize075", "crop10", "screenshot"}

# Robustness (STANDARD)
rows = []
for (C_np, Y_np, bits_np) in cached:
    psnr_c = psnr(Y_np, C_np)
    ssim_c = ssim_np(Y_np, C_np)
    lpips_c = lpips_score_np(Y_np, C_np, device=DEVICE) if (TORCH_OK and LPIPS_OK) else float("nan")

    rob = {}
    for atk_name, atk_fn in ATTACKS.items():
        if atk_name == "resize075":
            Ya, Ca = paired_resize_np(Y_np, C_np, 0.75)
        elif atk_name == "crop10":
            Ya, Ca = paired_crop10_np(Y_np, C_np, 0.10)
        elif atk_name == "screenshot":
            Ya, Ca = paired_screenshot_np(Y_np, C_np, 35, 80)
        else:
            Ya = atk_fn(Y_np)
            Ca = atk_fn(C_np) if atk_name.startswith("jpeg") else C_np

        Ya_t = torch.from_numpy(Ya).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
        Ca_t = torch.from_numpy(Ca).permute(2,0,1).unsqueeze(0).float().to(DEVICE)

        with torch.no_grad():
            probs = torch.sigmoid(Dec(Ya_t, Ca_t))[0].detach().cpu().numpy().reshape(-1)

        pred_bits = (probs >= 0.5).astype(np.int32)
        ber_bits  = float(np.mean(pred_bits != bits_np) * 100.0)
        bit_acc   = float(100.0 - ber_bits)

        tb = (bits_np * 2 - 1).astype(np.float32)
        pb = (pred_bits * 2 - 1).astype(np.float32)
        nc_bits = float(np.mean(tb * pb))

        rob[atk_name] = {"NC_bits": nc_bits, "BER": ber_bits, "bit_acc": bit_acc}

    rows.append({"psnr_cover": psnr_c, "ssim_cover": ssim_c, "lpips_cover": lpips_c, "robustness": rob})

# Decision reliability: correct vs wrong payload (STANDARD)
bits_pool = [b for (_,_,b) in cached]
bits_pool_wrong = bits_pool[1:] + bits_pool[:1]

det_scores_std = {a: {"pos": [], "neg": []} for a in attacks}
det_auc = {}
det_conf = {}

for atk_name, atk_fn in ATTACKS.items():
    pos_scores, neg_scores = [], []
    for idx, (C_np, Y_np, bits_np) in enumerate(cached):
        if atk_name == "resize075":
            Ya, Ca = paired_resize_np(Y_np, C_np, 0.75)
        elif atk_name == "crop10":
            Ya, Ca = paired_crop10_np(Y_np, C_np, 0.10)
        elif atk_name == "screenshot":
            Ya, Ca = paired_screenshot_np(Y_np, C_np, 35, 80)
        else:
            Ya = atk_fn(Y_np)
            Ca = atk_fn(C_np) if atk_name.startswith("jpeg") else C_np

        Ya_t = torch.from_numpy(Ya).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
        Ca_t = torch.from_numpy(Ca).permute(2,0,1).unsqueeze(0).float().to(DEVICE)

        with torch.no_grad():
            probs = torch.sigmoid(Dec(Ya_t, Ca_t))[0].detach().cpu().numpy().reshape(-1)

        pred_bits = (probs >= 0.5).astype(np.int32)

        ber_pos = float(np.mean(pred_bits != bits_np) * 100.0)
        s_pos = 1.0 - ber_pos/100.0

        wrong_bits = bits_pool_wrong[idx]
        ber_neg = float(np.mean(pred_bits != wrong_bits) * 100.0)
        s_neg = 1.0 - ber_neg/100.0

        pos_scores.append(s_pos)
        neg_scores.append(s_neg)

    det_scores_std[atk_name]["pos"] = pos_scores
    det_scores_std[atk_name]["neg"] = neg_scores

    y_true = np.array([1]*len(pos_scores) + [0]*len(neg_scores))
    y_score = np.array(pos_scores + neg_scores)

    A = float(roc_auc_score(y_true, y_score))
    cm = confusion_at_fpr(pos_scores, neg_scores, target_fpr=0.01)

    det_auc[atk_name] = A
    det_conf[atk_name] = cm

    # ROC plot per attack (STANDARD)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={A:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title(f"FLEXMark-RL Decision ROC (STANDARD) — {atk_name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"flexrl_decision_roc_standard_{atk_name}.png"), dpi=220)
    plt.close()

    plot_confusion_heatmap(
        cm,
        f"Decision Confusion @1%FPR (STANDARD) — {atk_name}",
        os.path.join(OUT_DIR, f"flexrl_decision_confusion_standard_{atk_name}.png")
    )

# Clean ROC “main”
if "clean" in attacks:
    pos = det_scores_std["clean"]["pos"]
    neg = det_scores_std["clean"]["neg"]
    y_true = np.array([1]*len(pos) + [0]*len(neg))
    y_score = np.array(pos + neg)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    A = float(roc_auc_score(y_true, y_score))

    plt.figure()
    plt.plot(fpr, tpr, label=f"Clean AUC={A:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title("FLEXMark-RL Decision ROC (clean): correct vs wrong payload")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "flexrl_decision_roc_clean.png"), dpi=260)
    plt.close()

infer_seconds = time.time() - t0
infer_co2 = cc_stop(infer_tracker)

# ============================================================
# AI laundering (separate) + robustness plots + decision after laundering
# ============================================================
pairs = []
for (C_np, Y_np, _) in cached[:150]:
    pairs.append((attack_jpeg(Y_np, 10), C_np))

launder_fn, launder_meta = train_ai_launderer_patchwise(
    pairs=pairs, out_dir=OUT_DIR, patch=64, patches_per_img=2, epochs=1, lr=1e-3, max_pairs=150, device=DEVICE
)

rows_launder = []
det_auc_L, det_conf_L = {}, {}
det_scores_lau = {a: {"pos": [], "neg": []} for a in attacks}

if launder_fn is not None:
    # robustness after laundering
    for (C_np, Y_np, bits_np) in cached:
        robL = {}
        for atk_name, atk_fn in ATTACKS.items():
            if atk_name == "resize075":
                Ya, Ca = paired_resize_np(Y_np, C_np, 0.75)
            elif atk_name == "crop10":
                Ya, Ca = paired_crop10_np(Y_np, C_np, 0.10)
            elif atk_name == "screenshot":
                Ya, Ca = paired_screenshot_np(Y_np, C_np, 35, 80)
            else:
                Ya = atk_fn(Y_np)
                Ca = atk_fn(C_np) if atk_name.startswith("jpeg") else C_np

            if atk_name != "clean":
                Ya = np.clip(launder_fn(Ya), 0, 1)

            Ya_t = torch.from_numpy(Ya).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
            Ca_t = torch.from_numpy(Ca).permute(2,0,1).unsqueeze(0).float().to(DEVICE)

            with torch.no_grad():
                probs = torch.sigmoid(Dec(Ya_t, Ca_t))[0].detach().cpu().numpy().reshape(-1)

            pred_bits = (probs >= 0.5).astype(np.int32)
            ber_bits  = float(np.mean(pred_bits != bits_np) * 100.0)
            bit_acc   = float(100.0 - ber_bits)

            tb = (bits_np * 2 - 1).astype(np.float32)
            pb = (pred_bits * 2 - 1).astype(np.float32)
            nc_bits = float(np.mean(tb * pb))

            robL[atk_name] = {"NC_bits": nc_bits, "BER": ber_bits, "bit_acc": bit_acc}

        rows_launder.append({"robustness": robL})

    # decision after laundering
    for atk_name, atk_fn in ATTACKS.items():
        pos_scores, neg_scores = [], []
        for idx, (C_np, Y_np, bits_np) in enumerate(cached):
            if atk_name == "resize075":
                Ya, Ca = paired_resize_np(Y_np, C_np, 0.75)
            elif atk_name == "crop10":
                Ya, Ca = paired_crop10_np(Y_np, C_np, 0.10)
            elif atk_name == "screenshot":
                Ya, Ca = paired_screenshot_np(Y_np, C_np, 35, 80)
            else:
                Ya = atk_fn(Y_np)
                Ca = atk_fn(C_np) if atk_name.startswith("jpeg") else C_np

            if atk_name != "clean":
                Ya = np.clip(launder_fn(Ya), 0, 1)

            Ya_t = torch.from_numpy(Ya).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
            Ca_t = torch.from_numpy(Ca).permute(2,0,1).unsqueeze(0).float().to(DEVICE)

            with torch.no_grad():
                probs = torch.sigmoid(Dec(Ya_t, Ca_t))[0].detach().cpu().numpy().reshape(-1)

            pred_bits = (probs >= 0.5).astype(np.int32)

            ber_pos = float(np.mean(pred_bits != bits_np) * 100.0)
            s_pos = 1.0 - ber_pos/100.0

            wrong_bits = bits_pool_wrong[idx]
            ber_neg = float(np.mean(pred_bits != wrong_bits) * 100.0)
            s_neg = 1.0 - ber_neg/100.0

            pos_scores.append(s_pos)
            neg_scores.append(s_neg)

        det_scores_lau[atk_name]["pos"] = pos_scores
        det_scores_lau[atk_name]["neg"] = neg_scores

        y_true = np.array([1]*len(pos_scores) + [0]*len(neg_scores))
        y_score = np.array(pos_scores + neg_scores)

        A = float(roc_auc_score(y_true, y_score))
        cm = confusion_at_fpr(pos_scores, neg_scores, target_fpr=0.01)

        det_auc_L[atk_name] = A
        det_conf_L[atk_name] = cm

        fpr, tpr, _ = roc_curve(y_true, y_score)
        plt.figure()
        plt.plot(fpr, tpr, label=f"AUC={A:.3f}")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("FPR"); plt.ylabel("TPR")
        plt.title(f"FLEXMark-RL Decision ROC (LAUNDERED) — {atk_name}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f"flexrl_decision_roc_laundered_{atk_name}.png"), dpi=220)
        plt.close()

        plot_confusion_heatmap(
            cm,
            f"Decision Confusion @1%FPR (LAUNDERED) — {atk_name}",
            os.path.join(OUT_DIR, f"flexrl_decision_confusion_laundered_{atk_name}.png")
        )

# ============================================================
# Summary
# ============================================================
summary = {
    "model": "FLEXMark-RL_controller_datasetpayload_pairedgeom",
    "payload_bits": PAYLOAD_BITS,
    "hw": HW,
    "eps": EPS,
    "grid": [gh, gw],
    "n_train": len(train_ds),
    "n_test_eval": len(rows),

    "mean_psnr_cover": float(np.mean([r["psnr_cover"] for r in rows])),
    "mean_ssim_cover": float(np.mean([r["ssim_cover"] for r in rows])),
    "mean_lpips_cover": float(np.nanmean([r["lpips_cover"] for r in rows])),

    "mean_nc_bits":  {a: float(np.mean([r["robustness"][a]["NC_bits"] for r in rows])) for a in attacks},
    "mean_ber":      {a: float(np.mean([r["robustness"][a]["BER"]     for r in rows])) for a in attacks},
    "mean_bit_acc":  {a: float(np.mean([r["robustness"][a]["bit_acc"] for r in rows])) for a in attacks},

    "decision_auc_by_attack": det_auc,
    "decision_confusion_1pctfpr_by_attack": det_conf,

    "train_seconds": float(train_seconds),
    "infer_seconds": float(infer_seconds),
    "train_co2_kg": float(train_co2) if train_co2 is not None else None,
    "infer_co2_kg": float(infer_co2) if infer_co2 is not None else None,
    "model_file_mb": float(model_file_mb),

    "ai_laundering_meta": launder_meta,

    "train_recipe": {
        "warmup_epochs": EPOCHS_WARMUP,
        "robust_epochs": EPOCHS_ROBUST,
        "lambda_msg": LAMBDA_MSG,
        "lambda_imp": LAMBDA_IMP,
        "lambda_res": LAMBDA_RES,
        "lambda_rl": LAMBDA_RL,
        "psnr_floor": PSNR_FLOOR,
        "jpeg_train_p": JPEG_TRAIN_P,
        "jpeg_quals": JPEG_QUALS,
        "crop_p_start": CROP_P_START,
        "crop_p_end": CROP_P_END,
        "decoder": "nonblind_grid(ghxgw)",
        "controller": "RL(mask_grid + strength_level)",
        "reward": "soft_correctness - 0.20*psnr_pen - 0.10*budget_pen",
        "note": "Main robustness/decision = attacks only; laundering is separate adaptive removal attempt; paired geom transforms."
    }
}

if len(rows_launder) > 0:
    summary["mean_nc_bits_laundered"] = {a: float(np.mean([r["robustness"][a]["NC_bits"] for r in rows_launder])) for a in attacks}
    summary["mean_ber_laundered"]     = {a: float(np.mean([r["robustness"][a]["BER"]     for r in rows_launder])) for a in attacks}
    summary["mean_bit_acc_laundered"] = {a: float(np.mean([r["robustness"][a]["bit_acc"] for r in rows_launder])) for a in attacks}
    summary["decision_auc_by_attack_laundered"] = det_auc_L
    summary["decision_confusion_1pctfpr_by_attack_laundered"] = det_conf_L
else:
    summary["mean_nc_bits_laundered"] = None
    summary["mean_ber_laundered"]     = None
    summary["mean_bit_acc_laundered"] = None
    summary["decision_auc_by_attack_laundered"] = None
    summary["decision_confusion_1pctfpr_by_attack_laundered"] = None

# ============================================================
# Plots: robustness curves (standard + laundered + delta)
# ============================================================
plot_attack_metric_line(attacks, summary["mean_nc_bits"],
                        "FLEXMark-RL mean NC_bits under attacks (standard)",
                        os.path.join(OUT_DIR, "flexrl_NC_attacks.png"),
                        ylabel="NC_bits", ylim=(-1, 1))

plot_attack_metric_line(attacks, summary["mean_ber"],
                        "FLEXMark-RL mean BER (%) under attacks (standard)",
                        os.path.join(OUT_DIR, "flexrl_BER_attacks.png"),
                        ylabel="BER (%)", ylim=(0, 100))

plot_attack_metric_line(attacks, summary["mean_bit_acc"],
                        "FLEXMark-RL mean Bit Accuracy (%) under attacks (standard)",
                        os.path.join(OUT_DIR, "flexrl_bitacc_attacks.png"),
                        ylabel="Bit Accuracy (%)", ylim=(0, 105))

if summary["mean_bit_acc_laundered"] is not None:
    plot_attack_metric_line(attacks, summary["mean_nc_bits_laundered"],
                            "FLEXMark-RL mean NC_bits under attacks (after laundering)",
                            os.path.join(OUT_DIR, "flexrl_NC_attacks_laundered.png"),
                            ylabel="NC_bits", ylim=(-1, 1))

    plot_attack_metric_line(attacks, summary["mean_ber_laundered"],
                            "FLEXMark-RL mean BER (%) under attacks (after laundering)",
                            os.path.join(OUT_DIR, "flexrl_BER_attacks_laundered.png"),
                            ylabel="BER (%)", ylim=(0, 100))

    plot_attack_metric_line(attacks, summary["mean_bit_acc_laundered"],
                            "FLEXMark-RL mean Bit Accuracy (%) under attacks (after laundering)",
                            os.path.join(OUT_DIR, "flexrl_bitacc_attacks_laundered.png"),
                            ylabel="Bit Accuracy (%)", ylim=(0, 105))

    bit_std = np.array([summary["mean_bit_acc"][a] for a in attacks], dtype=float)
    bit_lau = np.array([summary["mean_bit_acc_laundered"][a] for a in attacks], dtype=float)
    delta = bit_std - bit_lau

    plt.figure(figsize=(12,4))
    plt.bar(attacks, delta)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Bit-Accuracy Drop (pp)")
    plt.title("AI Laundering Impact: Δ(BitAcc) = Standard − Laundered (FLEXMark-RL)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "flexrl_delta_bitacc_standard_minus_laundered.png"), dpi=220)
    plt.close()

# Save summary
save_json(summary, os.path.join(OUT_DIR, "flexmark_rl_summary.json"))
print(json.dumps(summary, indent=2))
print("✅ Outputs in:", OUT_DIR)

FLEXMark-RL curriculum: | warmup: 2 | robust: 6 | grid: (16, 16) | EPS: 0.02 | JPEG_P: 0.5 | JPEG_QUALS: [10, 30, 50] | PSNR_FLOOR: 35.0 | DEVICE: cuda
Using n_train = 5000 | n_test = 1000 | eval = 1000


[codecarbon WARNING @ 01:59:50] Multiple instances of codecarbon are allowed to run at the same time.


[Warmup] 1/2 | loss=0.2196 (msg=0.0187, imp=0.0124, res=0.6223, rl=0.0000, ch=clean)
[Warmup] 2/2 | loss=0.0618 (msg=0.0046, imp=0.0062, res=0.3105, rl=0.0000, ch=clean)
[Robust] 1/6 | loss=3.7650 (msg=0.2307, imp=0.0077, res=0.3988, rl=5.7498, ch=jpeg) | crop_p=0.050
[Robust] 2/6 | loss=4.5167 (msg=0.0873, imp=0.0088, res=0.4568, rl=14.4793, ch=resize075) | crop_p=0.090
[Robust] 3/6 | loss=5.1086 (msg=0.0716, imp=0.0080, res=0.4275, rl=17.4833, ch=clean) | crop_p=0.130
[Robust] 4/6 | loss=3.7419 (msg=0.0530, imp=0.0104, res=0.5288, rl=12.7377, ch=noise) | crop_p=0.170
[Robust] 5/6 | loss=-1.3014 (msg=0.3968, imp=0.0105, res=0.5328, rl=-21.1866, ch=jpeg) | crop_p=0.210
[Robust] 6/6 | loss=3.4042 (msg=0.0125, imp=0.0112, res=0.5808, rl=12.9947, ch=clean) | crop_p=0.250
Saved: /kaggle/working/flexmark_out_bits256/flexmark_rl_Enc.pt /kaggle/working/flexmark_out_bits256/flexmark_rl_DecGrid.pt /kaggle/working/flexmark_out_bits256/flexmark_rl_Policy.pt | total 0.84 MB


/tmp/ipykernel_55/3889498849.py:689: RuntimeWarning: Mean of empty slice
  "mean_lpips_cover": float(np.nanmean([r["lpips_cover"] for r in rows])),


{
  "model": "FLEXMark-RL_controller_datasetpayload_pairedgeom",
  "payload_bits": 256,
  "hw": 128,
  "eps": 0.02,
  "grid": [
    16,
    16
  ],
  "n_train": 5000,
  "n_test_eval": 1000,
  "mean_psnr_cover": 34.92583075752885,
  "mean_ssim_cover": 0.9529190131425858,
  "mean_lpips_cover": NaN,
  "mean_nc_bits": {
    "clean": 0.9805703125,
    "jpeg10": 0.7833125,
    "jpeg30": 0.9679296875,
    "jpeg50": 0.9759921875,
    "jpeg70": 0.977171875,
    "jpeg90": 0.97828125,
    "resize075": 0.9798515625,
    "crop10": 0.2455234375,
    "blur": 0.9050390625,
    "sharpen": 0.9791328125,
    "gamma12": 0.0764609375,
    "screenshot": 0.9698359375,
    "launder_only": 0.9805703125
  },
  "mean_ber": {
    "clean": 0.971484375,
    "jpeg10": 10.834375,
    "jpeg30": 1.603515625,
    "jpeg50": 1.200390625,
    "jpeg70": 1.14140625,
    "jpeg90": 1.0859375,
    "resize075": 1.007421875,
    "crop10": 37.723828125,
    "blur": 4.748046875,
    "sharpen": 1.043359375,
    "gamma12": 46.1769531

### AUC, BitAcc and BER Graphs and Confusion metrics

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# Output directory
# -------------------------
os.makedirs("figures", exist_ok=True)

ATTACKS = ["clean","jpeg10","jpeg30","jpeg50","jpeg70","jpeg90","resize075","crop10","blur","sharpen","gamma12","screenshot"]

# -------------------------
# Helper: ordering + safe access
# -------------------------
def series_from_dict(d, attacks=ATTACKS):
    return [float(d.get(a, np.nan)) for a in attacks]

def fmt_pct(x):
    return f"{100.0*x:.1f}%"  # for TPR/FPR, x in [0,1]

# -------------------------
# BASE, non-laundered
# -------------------------
RESULTS = [
    {
        "name":"SRE-128", "bits":128, "signal":"Detector",
        "psnr":47.974823397031656, "ssim":0.9960154550671577,
        "size_mb":8.79301643371582, "train_s":204.0207953453064, "infer_s":69.2060489654541,
        "bit_acc": {
            "clean":100.0,"jpeg10":52.2234375,"jpeg30":58.14296875,"jpeg50":62.97734375,"jpeg70":70.5375,"jpeg90":95.590625,
            "resize075":73.421875,"crop10":50.49140625,"blur":60.9,"sharpen":99.42578125,"gamma12":67.640625,"screenshot":58.16796875
        },
        "ber": {
            "clean":0.0,"jpeg10":47.7765625,"jpeg30":41.85703125,"jpeg50":37.02265625,"jpeg70":29.4625,"jpeg90":4.409375,
            "resize075":26.578125,"crop10":49.50859375,"blur":39.1,"sharpen":0.57421875,"gamma12":32.359375,"screenshot":41.83203125
        },
        "auc": {
            "clean":1.0,"jpeg10":0.6257365,"jpeg30":0.8503609999999999,"jpeg50":0.9564405,"jpeg70":0.9958385,"jpeg90":1.0,
            "resize075":0.9860815,"crop10":0.536954,"blur":0.837781,"sharpen":1.0,"gamma12":0.835994,"screenshot":0.85644
        },
        "conf_1pct": {
            "clean":{"thr":0.6161445170640946,"TP":1000,"FP":10,"TN":990,"FN":0,"fpr_actual":0.01,"tpr_actual":1.0},
            "jpeg90":{"thr":0.6241433763504028,"TP":1000,"FP":10,"TN":990,"FN":0,"fpr_actual":0.01,"tpr_actual":1.0},
            "resize075":{"thr":0.6103182816505432,"TP":878,"FP":10,"TN":990,"FN":122,"fpr_actual":0.01,"tpr_actual":0.878},
            "crop10":{"thr":0.5961285954713821,"TP":13,"FP":10,"TN":990,"FN":987,"fpr_actual":0.01,"tpr_actual":0.013},
        }
    },
    {
        "name":"SRE-256", "bits":256, "signal":"Detector",
        "psnr":46.93630845161286, "ssim":0.9948052486181259,
        "size_mb":8.79301643371582, "train_s":181.41535663604736, "infer_s":67.9333987236023,
        "bit_acc": {
            "clean":100.0,"jpeg10":52.23671875,"jpeg30":56.996875,"jpeg50":60.74296875,"jpeg70":67.609765625,"jpeg90":90.978515625,
            "resize075":68.761328125,"crop10":50.292578125,"blur":57.79296875,"sharpen":98.670703125,"gamma12":67.591796875,"screenshot":56.79453125
        },
        "ber": {
            "clean":0.0,"jpeg10":47.76328125,"jpeg30":43.003125,"jpeg50":39.25703125,"jpeg70":32.390234375,"jpeg90":9.021484375,
            "resize075":31.238671875,"crop10":49.707421875,"blur":42.20703125,"sharpen":1.329296875,"gamma12":32.408203125,"screenshot":43.20546875
        },
        "auc": {
            "clean":1.0,"jpeg10":0.6532775,"jpeg30":0.867725,"jpeg50":0.945735,"jpeg70":0.991603,"jpeg90":1.0,
            "resize075":0.9875085,"crop10":0.519386,"blur":0.8692255,"sharpen":1.0,"gamma12":0.8586475,"screenshot":0.863484
        },
        "conf_1pct": {
            "clean":{"thr":0.5881248211860657,"TP":1000,"FP":10,"TN":990,"FN":0,"fpr_actual":0.01,"tpr_actual":1.0},
            "jpeg90":{"thr":0.5610750210285187,"TP":1000,"FP":10,"TN":990,"FN":0,"fpr_actual":0.01,"tpr_actual":1.0},
            "resize075":{"thr":0.5394311040639878,"TP":902,"FP":10,"TN":990,"FN":98,"fpr_actual":0.01,"tpr_actual":0.902},
            "crop10":{"thr":0.502740626335144,"TP":15,"FP":10,"TN":990,"FN":985,"fpr_actual":0.01,"tpr_actual":0.015},
        }
    },
    {
        "name":"HiDDeN-128", "bits":128, "signal":"Presence",
        "psnr":37.39373970794593, "ssim":0.9793275873064995,
        "size_mb":1.0527715682983398, "train_s":236.44342041015625, "infer_s":67.13384866714478,
        "bit_acc": {
            "clean":100.0,"jpeg10":88.625,"jpeg30":99.84921875,"jpeg50":99.98515625,"jpeg70":99.99765625,"jpeg90":100.0,
            "resize075":100.0,"crop10":70.9453125,"blur":96.31640625,"sharpen":99.99453125,"gamma12":55.31015625,"screenshot":99.8203125
        },
        "ber": {
            "clean":0.0,"jpeg10":11.375,"jpeg30":0.15078125,"jpeg50":0.01484375,"jpeg70":0.00234375,"jpeg90":0.0,
            "resize075":0.0,"crop10":29.0546875,"blur":3.68359375,"sharpen":0.00546875,"gamma12":44.68984375,"screenshot":0.1796875
        },
        "auc": {  # presence_auc
            "clean":0.90862,"jpeg10":0.549324,"jpeg30":0.654176,"jpeg50":0.688495,"jpeg70":0.716278,"jpeg90":0.819753,
            "resize075":0.874786,"crop10":0.625782,"blur":0.837681,"sharpen":0.904964,"gamma12":0.882174,"screenshot":0.658679
        },
        "conf_1pct": {
            "clean":{"thr":0.6799125617742539,"TP":534,"FP":10,"TN":990,"FN":466,"fpr_actual":0.01,"tpr_actual":0.534},
            "jpeg90":{"thr":0.6633475315570831,"TP":153,"FP":10,"TN":990,"FN":847,"fpr_actual":0.01,"tpr_actual":0.153},
            "resize075":{"thr":0.6852093404531479,"TP":336,"FP":10,"TN":990,"FN":664,"fpr_actual":0.01,"tpr_actual":0.336},
            "crop10":{"thr":0.6619956833124161,"TP":17,"FP":10,"TN":990,"FN":983,"fpr_actual":0.01,"tpr_actual":0.017},
        }
    },
    {
        "name":"HiDDeN-256", "bits":256, "signal":"Presence",
        "psnr":37.094904775992475, "ssim":0.9696250320672989,
        "size_mb":1.0527715682983398, "train_s":250.69281697273254, "infer_s":70.82372164726257,
        "bit_acc": {
            "clean":99.998828125,"jpeg10":81.11015625,"jpeg30":97.738671875,"jpeg50":99.7234375,"jpeg70":99.962109375,"jpeg90":99.99921875,
            "resize075":100.0,"crop10":61.962109375,"blur":91.64296875,"sharpen":99.9734375,"gamma12":55.057421875,"screenshot":98.344921875
        },
        "ber": {
            "clean":0.001171875,"jpeg10":18.88984375,"jpeg30":2.261328125,"jpeg50":0.2765625,"jpeg70":0.037890625,"jpeg90":0.00078125,
            "resize075":0.0,"crop10":38.037890625,"blur":8.35703125,"sharpen":0.0265625,"gamma12":44.942578125,"screenshot":1.655078125
        },
        "auc": {  # presence_auc
            "clean":0.973894,"jpeg10":0.5592955,"jpeg30":0.7810115,"jpeg50":0.8915645,"jpeg70":0.917486,"jpeg90":0.929916,
            "resize075":0.990521,"crop10":0.747611,"blur":0.96695,"sharpen":0.972451,"gamma12":0.977284,"screenshot":0.82402
        },
        "conf_1pct": {
            "clean":{"thr":0.5942571341991425,"TP":832,"FP":10,"TN":990,"FN":168,"fpr_actual":0.01,"tpr_actual":0.832},
            "jpeg90":{"thr":0.5831363087892532,"TP":363,"FP":10,"TN":990,"FN":637,"fpr_actual":0.01,"tpr_actual":0.363},
            "resize075":{"thr":0.5711280745267868,"TP":880,"FP":10,"TN":990,"FN":120,"fpr_actual":0.01,"tpr_actual":0.88},
            "crop10":{"thr":0.5791082400083541,"TP":37,"FP":10,"TN":990,"FN":963,"fpr_actual":0.01,"tpr_actual":0.037},
        }
    },
    {
        "name":"RCRA-128", "bits":128, "signal":"Decision",
        "psnr":36.914763626626815, "ssim":0.9728525645136833,
        "size_mb":0.8379936218261719, "train_s":235.9663074016571, "infer_s":89.74547028541565,
        "bit_acc": {
            "clean":99.99921875,"jpeg10":89.059375,"jpeg30":99.709375,"jpeg50":99.9640625,"jpeg70":99.99296875,"jpeg90":99.9984375,
            "resize075":100.0,"crop10":70.20859375,"blur":98.5421875,"sharpen":99.98984375,"gamma12":61.0484375,"screenshot":99.82734375
        },
        "ber": {
            "clean":0.00078125,"jpeg10":10.940625,"jpeg30":0.290625,"jpeg50":0.0359375,"jpeg70":0.00703125,"jpeg90":0.0015625,
            "resize075":0.0,"crop10":29.79140625,"blur":1.4578125,"sharpen":0.01015625,"gamma12":38.9515625,"screenshot":0.17265625
        },
        "auc": {  # decision_auc
            "clean":1.0,"jpeg10":1.0,"jpeg30":1.0,"jpeg50":1.0,"jpeg70":1.0,"jpeg90":1.0,
            "resize075":1.0,"crop10":0.9954415,"blur":1.0,"sharpen":1.0,"gamma12":0.840001,"screenshot":1.0
        },
        "conf_1pct": {
            "clean":{"thr":0.6015625,"TP":1000,"FP":11,"TN":989,"FN":0,"fpr_actual":0.011,"tpr_actual":1.0},
            "jpeg90":{"thr":0.6015625,"TP":1000,"FP":11,"TN":989,"FN":0,"fpr_actual":0.011,"tpr_actual":1.0},
            "resize075":{"thr":0.6015625,"TP":1000,"FP":11,"TN":989,"FN":0,"fpr_actual":0.011,"tpr_actual":1.0},
            "crop10":{"thr":0.593828125,"TP":950,"FP":10,"TN":990,"FN":50,"fpr_actual":0.01,"tpr_actual":0.95},
        }
    },
    {
        "name":"RCRA-256", "bits":256, "signal":"Decision",
        "psnr":34.92583075752885, "ssim":0.9529190131425858,
        "size_mb":0.8379936218261719, "train_s":253.03425121307373, "infer_s":91.54237365722656,
        "bit_acc": {
            "clean":99.028515625,"jpeg10":89.165625,"jpeg30":98.396484375,"jpeg50":98.799609375,"jpeg70":98.85859375,"jpeg90":98.9140625,
            "resize075":98.992578125,"crop10":62.276171875,"blur":95.251953125,"sharpen":98.956640625,"gamma12":53.823046875,"screenshot":98.491796875
        },
        "ber": {
            "clean":0.971484375,"jpeg10":10.834375,"jpeg30":1.603515625,"jpeg50":1.200390625,"jpeg70":1.14140625,"jpeg90":1.0859375,
            "resize075":1.007421875,"crop10":37.723828125,"blur":4.748046875,"sharpen":1.043359375,"gamma12":46.176953125,"screenshot":1.508203125
        },
        "auc": {  # decision_auc
            "clean":1.0,"jpeg10":0.9999565,"jpeg30":1.0,"jpeg50":1.0,"jpeg70":1.0,"jpeg90":1.0,
            "resize075":1.0,"crop10":0.9834705,"blur":1.0,"sharpen":1.0,"gamma12":0.7028425,"screenshot":1.0
        },
        "conf_1pct": {
            "clean":{"thr":0.5742578125,"TP":1000,"FP":10,"TN":990,"FN":0,"fpr_actual":0.01,"tpr_actual":1.0},
            "jpeg90":{"thr":0.5742578125,"TP":1000,"FP":10,"TN":990,"FN":0,"fpr_actual":0.01,"tpr_actual":1.0},
            # NOTE: resize075 has fpr_actual=0.02 in your output; we still plot and label it.
            "resize075":{"thr":0.57421875,"TP":1000,"FP":20,"TN":980,"FN":0,"fpr_actual":0.02,"tpr_actual":1.0},
            "crop10":{"thr":0.57421875,"TP":846,"FP":11,"TN":989,"FN":154,"fpr_actual":0.011,"tpr_actual":0.846},
        }
    },
    {
        "name":"SVR-128", "bits":128, "signal":"Detector",
        "psnr":39.67181896154149, "ssim":0.9925884660482407,
        "size_mb":1.0668525695800781, "train_s":3454.774219751358, "infer_s":8674.89072227478,
        "bit_acc": {
            "clean":68.11015625,"jpeg10":54.56953125,"jpeg30":57.1421875,"jpeg50":58.38046875,"jpeg70":59.45078125,"jpeg90":62.78828125,
            "resize075":61.63203125,"crop10":53.26015625,"blur":57.93203125,"sharpen":56.05703125,"gamma12":50.7953125,"screenshot":57.56328125
        },
        "ber": {
            "clean":31.88984375,"jpeg10":45.43046875,"jpeg30":42.8578125,"jpeg50":41.61953125,"jpeg70":40.54921875,"jpeg90":37.21171875,
            "resize075":38.36796875,"crop10":46.73984375,"blur":42.06796875,"sharpen":43.94296875,"gamma12":49.2046875,"screenshot":42.43671875
        },
        "auc": {
            "clean":0.984209,"jpeg10":0.7015335,"jpeg30":0.817228,"jpeg50":0.8508535,"jpeg70":0.885838,"jpeg90":0.94761,
            "resize075":0.9320265,"crop10":0.635574,"blur":0.8252465,"sharpen":0.7694165,"gamma12":0.56573,"screenshot":0.8330615
        },
        "conf_1pct": {
            "clean":{"thr":0.6216221887782933,"TP":886,"FP":10,"TN":990,"FN":114,"fpr_actual":0.01,"tpr_actual":0.886},
            "jpeg90":{"thr":0.5930778417691956,"TP":576,"FP":10,"TN":990,"FN":424,"fpr_actual":0.01,"tpr_actual":0.576},
            "resize075":{"thr":0.5830084575428619,"TP":550,"FP":10,"TN":990,"FN":450,"fpr_actual":0.01,"tpr_actual":0.55},
            "crop10":{"thr":0.6301493374726773,"TP":42,"FP":10,"TN":990,"FN":958,"fpr_actual":0.01,"tpr_actual":0.042},
        }
    },
    {
        "name":"SVR-256", "bits":256, "signal":"Detector",
        "psnr":41.51801903618308, "ssim":0.9948320853710174,
        "size_mb":1.0668525695800781, "train_s":3353.840486764908, "infer_s":8928.373472690582,
        "bit_acc": {
            "clean":62.75,"jpeg10":52.994921875,"jpeg30":54.422265625,"jpeg50":55.129296875,"jpeg70":55.785546875,"jpeg90":57.48359375,
            "resize075":57.31171875,"crop10":51.85546875,"blur":54.8671875,"sharpen":53.744140625,"gamma12":50.61953125,"screenshot":54.5265625
        },
        "ber": {
            "clean":37.25,"jpeg10":47.005078125,"jpeg30":45.577734375,"jpeg50":44.870703125,"jpeg70":44.214453125,"jpeg90":42.51640625,
            "resize075":42.68828125,"crop10":48.14453125,"blur":45.1328125,"sharpen":46.255859375,"gamma12":49.38046875,"screenshot":45.4734375
        },
        "auc": {
            "clean":0.978617,"jpeg10":0.6897155,"jpeg30":0.7803485,"jpeg50":0.810609,"jpeg70":0.835959,"jpeg90":0.900222,
            "resize075":0.8896915,"crop10":0.6024635,"blur":0.7856025,"sharpen":0.727878,"gamma12":0.555281,"screenshot":0.7802645
        },
        "conf_1pct": {
            "clean":{"thr":0.5822404619753109,"TP":862,"FP":10,"TN":990,"FN":138,"fpr_actual":0.01,"tpr_actual":0.862},
            "jpeg90":{"thr":0.5595719827865281,"TP":325,"FP":10,"TN":990,"FN":675,"fpr_actual":0.01,"tpr_actual":0.325},
            "resize075":{"thr":0.5518004743952067,"TP":367,"FP":10,"TN":990,"FN":633,"fpr_actual":0.01,"tpr_actual":0.367},
            "crop10":{"thr":0.6109803119439995,"TP":32,"FP":10,"TN":990,"FN":968,"fpr_actual":0.01,"tpr_actual":0.032},
        }
    },
]

# -------------------------
# Plot 1: AUC across attacks
# -------------------------
plt.figure(figsize=(12, 5))
x = np.arange(len(ATTACKS))
for r in RESULTS:
    y = series_from_dict(r["auc"])
    plt.plot(x, y, marker="o", linewidth=2, label=f'{r["name"]} ({r["signal"]})')
plt.xticks(x, ATTACKS, rotation=35, ha="right")
plt.ylim(0.45, 1.01)
plt.xlabel("Attack")
plt.ylabel("AUC")
plt.title("Security-aligned reliability (AUC) across attacks")
plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
plt.legend(ncol=2, fontsize=8, frameon=True)
plt.tight_layout()
plt.savefig("figures/auc_attacks_combined.png", dpi=200)
plt.close()

# -------------------------
# Plot 2: Bit Accuracy across attacks
# -------------------------
plt.figure(figsize=(12, 5))
for r in RESULTS:
    y = series_from_dict(r["bit_acc"])
    plt.plot(x, y, marker="o", linewidth=2, label=r["name"])
plt.xticks(x, ATTACKS, rotation=35, ha="right")
plt.ylim(0, 101)
plt.xlabel("Attack")
plt.ylabel("Bit Accuracy (%)")
plt.title("Payload recovery (Bit Accuracy %) across attacks")
plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
plt.legend(ncol=4, fontsize=8, frameon=True)
plt.tight_layout()
plt.savefig("figures/bitacc_attacks_combined.png", dpi=200)
plt.close()

# -------------------------
# Plot 3: BER across attacks
# -------------------------
plt.figure(figsize=(12, 5))
for r in RESULTS:
    y = series_from_dict(r["ber"])
    plt.plot(x, y, marker="o", linewidth=2, label=r["name"])
plt.xticks(x, ATTACKS, rotation=35, ha="right")
plt.ylim(0, 55)
plt.xlabel("Attack")
plt.ylabel("BER (%)")
plt.title("Payload recovery (BER %) across attacks")
plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
plt.legend(ncol=4, fontsize=8, frameon=True)
plt.tight_layout()
plt.savefig("figures/ber_attacks_combined.png", dpi=200)
plt.close()

# -------------------------
# Plot 4: Representative confusion summaries @ ~1% FPR (base)
# Make a 4xN grid image (attacks as rows, models as columns).
# Each cell shows: TPR, FPR, thr
# -------------------------
REP_ATKS = ["clean","jpeg90","resize075","crop10"]
N = len(RESULTS)
R = len(REP_ATKS)

fig = plt.figure(figsize=(2.6*N, 1.9*R))
for i, atk in enumerate(REP_ATKS):
    for j, r in enumerate(RESULTS):
        ax = plt.subplot(R, N, i*N + j + 1)
        c = r["conf_1pct"][atk]
        tpr = c["tpr_actual"]
        fpr = c["fpr_actual"]
        thr = c["thr"]

        # 2x2 confusion heat (TP/FN on top row; FP/TN on bottom row)
        mat = np.array([[c["TP"], c["FN"]],[c["FP"], c["TN"]]], dtype=float)
        ax.imshow(mat, aspect="auto")

        ax.set_xticks([])
        ax.set_yticks([])

        if i == 0:
            ax.set_title(r["name"], fontsize=9)
        if j == 0:
            ax.set_ylabel(atk, fontsize=9)

        ax.text(0.02, 0.98,
                f"TPR={tpr:.3f}\nFPR={fpr:.3f}\nthr={thr:.3f}",
                transform=ax.transAxes, va="top", ha="left",
                fontsize=8, bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"))

plt.suptitle("Operating-point confusion summaries at ~1% FPR (base)", y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig("figures/confusion_grid_rep_base.png", dpi=200, bbox_inches="tight")
plt.close()

print("Saved figures:")
for p in [
    "figures/auc_attacks_combined.png",
    "figures/bitacc_attacks_combined.png",
    "figures/ber_attacks_combined.png",
    "figures/confusion_grid_rep_base.png",
]:
    print(" -", p)

Saved figures:
 - figures/auc_attacks_combined.png
 - figures/bitacc_attacks_combined.png
 - figures/ber_attacks_combined.png
 - figures/confusion_grid_rep_base.png
